# RMT analysis functions

- Per-movie: set globals (`traj`, `com`, `spat`, ...) via `set_current_movie()`.
- Each analysis block is a function that reads globals, stores key outputs into `mv_smry_s`, and saves figures.
- A driver loop runs the full suite for all movies.


In [ ]:
from pathlib import Path
import pickle
import tifffile

import numpy as np
import scipy as sp
import pandas as pd

import hdbscan
import sklearn as skl
import sklearn.preprocessing as skl_pre
import pingouin as pg

import matplotlib.pyplot as plt
import matplotlib.patches as plt_ptc

from collections import defaultdict








## helper functions

In [ ]:
def stdz(arr, axis):
  mn = np.nanmean(arr, axis, keepdims=True)
  std = np.nanstd(arr, axis, keepdims=True)
  std = np.where(np.isclose(std, 0), np.nan, std)
  return (arr - mn) / std


def delay_embed(traj, stk_n, frm_sep, stk_mtd):
  if stk_mtd == "interior":
    stk_frm_n = frm_n - (stk_n - 1) * frm_sep
    stk_traj = np.concatenate(
      [traj[:, (advance:=(stk_n - stk_idx - 1) * frm_sep): advance + stk_frm_n]
       for stk_idx in range(stk_n)], axis=0)
    stk_traj = stdz(stk_traj, 1)
    
  elif stk_mtd == "padding":
    std_traj = stdz(traj, 1)
    stk_frm_n = frm_n + (stk_n - 1) * frm_sep
    stk_traj = np.zeros((neur_n * stk_n, stk_frm_n))
    for stk_idx in range(stk_n):
      stk_traj[
        stk_idx * neur_n : (stk_idx+1) * neur_n,
        (delay:=stk_idx * frm_sep) : delay + frm_n
        ] = std_traj
      
  return stk_traj, stk_frm_n


def undo_delay_embed(stk_traj, stk_n, frm_sep, stk_mtd):
  # Dynamical principal components (dynamical PCs) are PCA components of the delay-embedded activity.
  # first find shape due to possible dynamical PC dimensions
  stk_dim, *mid_shape, stk_frm_n = stk_traj.shape
  neur_n = stk_dim // stk_n
  # reshape
  stk_traj = np.reshape(
    stk_traj,(stk_n, neur_n, *mid_shape, stk_frm_n))

  
  if stk_mtd == "interior":
    frm_n = stk_frm_n + (stk_n - 1) * frm_sep
    traj = np.full((stk_n, neur_n, *mid_shape, frm_n), np.nan)
    for stk_idx in range(stk_n):
      traj[
        stk_idx, ...,
        (advance:=(stk_n-stk_idx-1)*frm_sep) : advance + stk_frm_n
        ] = stk_traj[stk_idx]
    traj = np.nanmean(traj, 0)
    
  elif stk_mtd == "padding":
    frm_n = stk_frm_n - (stk_n - 1) * frm_sep
    traj = np.full((stk_n, neur_n, *mid_shape, frm_n), np.nan)
    for stk_idx in range(stk_n):
      traj[stk_idx] = stk_traj[
        stk_idx, ..., 
        (delay:=stk_idx * frm_sep) : delay + frm_n]
    traj = np.mean(traj, 0)
    
  return traj


def rolled_covariation_fct(traj, output_holder,
                             shift_s=np.array([0]), roll_axis=-2,
                             no_mean=True):
    shift_n = shift_s.shape[0]
    output_len = output_holder.shape[-1]
    output_holder = np.full((shift_n,) + traj.shape[:-1] + (output_len,), 0.)
    fluct = traj - no_mean * np.mean(traj, -1, keepdims=True)
    frame_n = traj.shape[-1]
    convolution_len = 2 * frame_n - 1
    ft = np.fft.rfft(fluct, convolution_len)
    for shift_idx in range(0, shift_n):
      convolution_for = np.fft.irfft(
        ft * np.roll(np.conj(ft), shift_s[shift_idx], roll_axis),
        convolution_len
      )[..., :frame_n]
      convolution_for = convolution_for / (np.flip(np.arange(frame_n), -1) + 1)
      output_holder[shift_idx] = convolution_for[..., :output_len]
    return output_holder
  

def bwr_plt(ax, mtx, vmax=None, cmap='bwr', **kwargs):
  if vmax is None:
    vmax = np.max(np.abs(mtx))
  ax.matshow(mtx, cmap=cmap, vmin=-vmax, vmax=vmax, **kwargs)


## Result saving

In [ ]:
mv_smry_s = defaultdict(dict)

def _store(stat_name, value):
  mv_smry_s[stat_name][mv_name] = value
  globals()[stat_name] = value
  return value

# def _require(stat_name):
#   if mv_idx not in mv_smry_s.get(stat_name, {}):
#     raise KeyError(
#       f"Missing required '{stat_name}' for mv_name={mv_name}. "
#       f"Call the producing analysis function first."
#     )
#   return mv_smry_s[stat_name][mv_name]


## Variables global across functions

In [ ]:
def preprocess_current_movie(frame_dt=0.264):
  global raw_traj, raw_neur_lbl, raw_com, raw_spat, raw_bg_x, raw_bg_t
  global cnmf_traj, traj, com, spat, bg_x, bg_t
  global neur_n, frm_n, mv_time

  row = movie_df.iloc[mv_idx]
  raw_path = Path(row["cnmf_path"])
  with np.load(raw_path) as npz:
    raw_traj, raw_neur_lbl, raw_com, raw_spat, raw_bg_x, raw_bg_t = [
      npz[f"arr_{arr_idx}"] for arr_idx in range(6)
    ]

  # keep raw spatial data for analysis; downsample only inside plotting code
  com = raw_com
  spat = raw_spat
  bg_x = raw_bg_x
  bg_t = raw_bg_t

  # remove non-neurons
  neur_mask = raw_neur_lbl >= 0
  traj = raw_traj[neur_mask]
  com = com[neur_mask]
  spat = spat[neur_mask]

  # remove dead neurons before standardization
  # dead := all-NaN OR zero-variance trajectory
  finite_row = np.any(np.isfinite(traj), 1)
  row_std = np.nanstd(traj, 1)
  var_row = ~np.isclose(row_std, 0, equal_nan=True)
  alive = finite_row & var_row

  traj = traj[alive]
  com = com[alive]
  spat = spat[alive]

  # save cleaned trajectory before standardization
  cnmf_traj = traj.copy()

  # standardize for downstream analysis
  traj = stdz(traj, 1)
  bg_t = stdz(bg_t, 1)

  _store("cnmf_traj", cnmf_traj)
  _store("traj", traj)
  _store("com", com)

  # _store also binds the module-level global, so downstream cells can read
  # frame_dt instead of repeating the literal
  _store("frame_dt", frame_dt)

  neur_n, frm_n = traj.shape
  mv_time = np.arange(frm_n) * frame_dt
  return traj


def set_current_movie(idx, frame_dt=None):
  # which one
  global mv_idx, mv_name, brst_type, tsu_type
  mv_idx = int(idx)
  row = movie_df.iloc[mv_idx]

  mv_name = row["mv_name"]
  tsu_type = row["tsu_type"]
  if frame_dt is None:
    frame_dt = 0.261 if tsu_type == "cab-" else 0.264

  brst_type = "none"
  local_idx = int(row["tsu_local_idx"])
  for _brst_type in ['multi', 'single', 'none']:
    if local_idx in mv_brst_type_dict[tsu_type][_brst_type]:
      brst_type = _brst_type
      break

  _store("mv_name", mv_name)
  _store("brst_type", brst_type)
  _store("tsu_type", tsu_type)
  _store("tsu", row["tsu"])
  _store("cond", row["cond"])

  preprocess_current_movie(frame_dt=frame_dt)

  return mv_name


## Analysis functions

In [ ]:
def find_bursts():
  fltr_frac = 0.6
  
  pop_mean = np.mean(traj,0)
  fltr_len= int(fltr_frac * frm_n)
  # find baseline and fluct
  bsln = sp.ndimage.median_filter(
    pop_mean, size=fltr_len, mode = 'nearest')
  fluct = pop_mean - bsln

  global all_brst_win, brst_len
  if brst_type == 'multi':
    # find peak
    peak = sp.signal.find_peaks(fluct, np.max(fluct)/3)[0]
    width = np.max(
      sp.signal.peak_widths(fluct, peak, rel_height=0.6)[0])
    peak = sp.signal.find_peaks(
      fluct, np.max(fluct)/3, distance = width)[0]
  
    # find window
    global rel_window
    width_gtr = int(np.minimum(
      1 * np.max(sp.signal.peak_widths(fluct, peak, rel_height=0.8)[0]), 
      np.min(np.diff(peak)) * 0.75))
    rel_window = (np.array([-0.6,1.2]) * width_gtr).astype(int)
    all_brst_win = peak[:,None] + rel_window
    brst_len = rel_window[1] - rel_window[0]

  else:
    all_brst_win = np.zeros((0,2))
    brst_len = 60
    
  _store("all_brst_win", all_brst_win)
  _store("brst_len", brst_len)


  # plot
  fig, ax_s = plt.subplots(3,1, figsize = (6,7))
  fig.suptitle(mv_name)
  ax_s[0].set(xlabel='frame idx', ylabel='activity')
  ax_s[1].set(xlabel='frame idx', ylabel='activity')

  ax = ax_s[0]
  ax.plot(pop_mean, label = 'pop mean')
  ax.plot(bsln, label = 'med filt')
  ax.legend()

  ax = ax_s[1]
  pop_fluct = traj - bsln
  # fluctuation
  bwr_plt(ax, pop_fluct, aspect = 3.5)  
  if brst_type == 'multi':
    for win in all_brst_win:
      rect = plt_ptc.Rectangle(
        (win[0], 0), brst_len, neur_n, 
        color = 'gray', alpha = 0.2)
      ax.add_patch(rect)
  
  ax = ax_s[2]
  # fluctuation
  ax.plot(fluct, label = 'residual')
  # mark edges
  ax.vlines([0, frm_n-1], 0,np.max(fluct)*1.2, color = 'black')
  if brst_type == 'multi':
    # mark peaks
    ax.vlines(peak, 0,np.max(fluct)*1.2, color = 'red', lw =0.5)
    # mark windows
    for win in all_brst_win:
      ax.fill_between(win, -0.1, 0.9*np.max(fluct), color='red', alpha=0.1)
    ax.fill_between([0, 0], 0, 0, color='red', alpha=0.1, label='win')
  ax.legend()

  fig.tight_layout()
  fig.savefig(main_path / f'analysis/result/{brst_type}/01-wins-{mv_name}.pdf')


In [ ]:
def fit_acov():
  fltr_frac = 0.3
  poly_deg = 3

  global acov, a_fit, frm_sep
  
  # avoid acov dominated by slow timescales
  fltr_len = int(fltr_frac * frm_n)
  drift = sp.ndimage.median_filter(
    traj, size=fltr_len, axes = 1)
  # smooth it to avoid short timescale artifacts 
  frm_vld = np.arange(fltr_len//2, frm_n - fltr_len//2)
  for neur_idx in range(neur_n):
    coef = np.polyfit(
      frm_vld, 
      drift[neur_idx, frm_vld], deg=poly_deg)
    drift[neur_idx] = np.polyval(coef, np.arange(frm_n))
  traj_no_drift = traj - drift
  traj_no_drift = stdz(traj_no_drift, 1) # so acov starts at 1

  acov_frm_n = 5
  acov = rolled_covariation_fct(traj_no_drift, np.zeros((neur_n, acov_frm_n)))[0]
  
  fit_len = [0, acov_frm_n]
  mean_acov = np.mean(acov, 0)
  a_fit = np.exp(
    np.polyfit(
      np.arange(*fit_len),
      np.log(mean_acov[fit_len[0] : fit_len[1]]), 1
    )[0]
  )

  target_acov = 0.85 # 1/e**(1/6)
  half_life = np.log(target_acov) / np.log(a_fit)
  candidate_s = np.unique(np.maximum(1, np.array([
    np.floor(half_life), np.ceil(half_life)
  ], dtype=int)))
  frm_sep = int(candidate_s[np.argmin(np.abs(a_fit ** candidate_s - target_acov))])
  frm_sep = 3 * frm_sep

  # _store("acov", acov)
  _store("a_fit", a_fit)
  _store("frm_sep", frm_sep)


In [ ]:
def stack_trajs():
  global frm_sep, stk_n, stk_mtd
  global stk_traj, stk_frm_n
  if "frm_sep" not in globals():
    raise RuntimeError("Run fit_acov() before stack_trajs() to set frm_sep")
  stk_n = 5 # brst_len // frm_sep
  stk_mtd = "padding" # "interior" # 
  stk_traj, stk_frm_n = delay_embed(
    traj, stk_n, frm_sep, stk_mtd)
  _store("frm_sep", frm_sep)
  _store("stk_n", stk_n)
  _store("stk_mtd", stk_mtd)
  _store("stk_traj", stk_traj)
  _store("stk_frm_n", stk_frm_n)

  
  global brst_n, all_brst_win
  if stk_mtd == "interior":
    intact_win = (
      (((stk_n - 1) * frm_sep) < all_brst_win[:, 0])
      & (all_brst_win[:, 1] < (frm_n - 1 - (stk_n - 1) * frm_sep))
    )
    all_brst_win = all_brst_win[intact_win]
  elif stk_mtd == "padding":
    intact_win = (
      (0 <= all_brst_win[:, 0]) & (all_brst_win[:, 1] < frm_n)
    )
    all_brst_win = all_brst_win[intact_win]
  brst_n = len(all_brst_win)
  _store("all_brst_win", all_brst_win)
  _store("brst_n", brst_n)



In [ ]:
def numeric_rmt(): 
  
  global num_rmt_spct, stk_num_rmt_spct
  rmt_trial_n = 20
  num_rmt_acov = a_fit ** np.arange(frm_n)
  num_rmt_mtd = 'dyn'
  # initialize
  num_rmt_spct = np.zeros((rmt_trial_n, np.min(traj.shape)))
  stk_num_rmt_spct = np.zeros((rmt_trial_n, np.min(stk_traj.shape)))

  
  rng = np.random.default_rng(0)
  for rmt_trial_idx in range(rmt_trial_n):
    
    # build traj
    if num_rmt_mtd == 'cov':
      num_rmt_traj = (
        rng.standard_normal((neur_n, frm_n))
        @ np.linalg.cholesky(sp.linalg.toeplitz(num_rmt_acov)).T
      )
    elif num_rmt_mtd == 'dyn':
      num_rmt_traj = rng.normal(0, 1, (neur_n, frm_n))
      for frm_idx in range(frm_n):
        num_rmt_traj[:, frm_idx] = (
          a_fit * num_rmt_traj[:, frm_idx - 1]
          + np.sqrt(1 - a_fit ** 2) * num_rmt_traj[:, frm_idx]
        )
        
    # compute spectrum
    num_rmt_traj = (
      (num_rmt_traj - np.mean(num_rmt_traj, 1)[:, None])
      / np.std(num_rmt_traj, 1)[:, None]
    )
    diag = np.linalg.svd(
      num_rmt_traj / np.sqrt(frm_n), full_matrices=False, compute_uv=False)
    num_rmt_spct[rmt_trial_idx] = diag ** 2

    # stacked
    stk_num_rmt_traj, _ = delay_embed(
      num_rmt_traj, stk_n, frm_sep, stk_mtd)
    diag = np.linalg.svd(
      stk_num_rmt_traj / np.sqrt(
        # different normalization depending on stack/edge treatment
        stk_frm_n if stk_mtd == "interior" else frm_n), 
      full_matrices=False, compute_uv=False)
    stk_num_rmt_spct[rmt_trial_idx] = diag ** 2

  num_rmt_spct = np.median(np.sort(num_rmt_spct, axis = 1), axis = 0)
  stk_num_rmt_spct = np.median(np.sort(stk_num_rmt_spct, axis = 1), axis = 0)
  _store("num_rmt_spct", num_rmt_spct)
  _store("stk_num_rmt_spct", stk_num_rmt_spct)


  acov_frm_n = acov.shape[1]
  num_rmt_acov = rolled_covariation_fct(num_rmt_traj, np.zeros((neur_n, acov_frm_n)))[0]
  
  fig, ax_s = plt.subplots(1, 2)
  fig.suptitle(f"frm_sep = {frm_sep}")
  for ax in ax_s:
    ax.plot(np.mean(acov, 0), label='real')
    ax.plot(a_fit ** np.arange(acov_frm_n), ls="--", label='null (ana)')
    ax.plot(np.mean(num_rmt_acov, 0), ls="--", label='null (num)')
  ax_s[0].set(yscale="log", ylabel='acov', xlabel='frame idx')
  ax_s[0].legend()
  fig.tight_layout()
  fig.savefig(main_path / f'analysis/result/{brst_type}/02-acov-{mv_name}.pdf')



In [ ]:
def analytic_rmt_n_comparison():
  def exp_tcor_rmt(a, q, rmt_val_n=200, iter_n=3, ges_fctr=3):
    b = (1 + a ** 2) / (1 - a ** 2)
    q_eff = 1 + b * q - np.sqrt(1 + q ** 2 * (b ** 2 - 1))
    z_est_range = (1 + np.sqrt(q_eff)) ** 2
    z = np.linspace(0, ges_fctr * z_est_range, rmt_val_n + 1)[1:]
    rho = np.zeros(rmt_val_n)
    for iter_idx in range(iter_n):
      for val_idx, z_val in enumerate(z):
        t_brc_s = np.roots([
          -q ** 2, -2 * q ** 2 + 2 * z_val * b * q,
          1 - q ** 2 + 2 * z_val * b * q - z_val ** 2, 2, 1
        ])
        g_brc_s = (1 + t_brc_s) / z_val
        rho[val_idx] = np.max(-np.imag(g_brc_s) / np.pi)
      lft_idx = np.maximum(0, np.where(rho > 0)[0][0] - 1)
      rgt_idx = np.minimum(rmt_val_n - 1, np.where(rho > 0)[0][-1] + 1)
      if iter_idx != iter_n - 1:
        z = np.linspace(0, 1.1 * z[rgt_idx], rmt_val_n + 1)[1:]
    return ([z, rho], z[[lft_idx, rgt_idx]])
  
  global rmt_edge, stk_rmt_edge
  q_rmt = neur_n / frm_n
  stk_q_rmt = stk_n * neur_n / stk_frm_n
  rmt_edge = exp_tcor_rmt(a_fit, q_rmt)[1]
  stk_rmt_edge = exp_tcor_rmt(a_fit, stk_q_rmt)[1]
  _store("rmt_edge", rmt_edge)
  _store("stk_rmt_edge", stk_rmt_edge)


  global pc, stk_pc
  lft, diag, rgt = np.linalg.svd(
    traj / np.sqrt(frm_n), full_matrices=False)
  pc = [diag[::-1] ** 2, lft[:, ::-1]] # small to large as eigh
  lft, diag, rgt = np.linalg.svd(
    stk_traj / np.sqrt(
      # different normalization depending on stack/edge treatment
      stk_frm_n if stk_mtd == "interior" else frm_n),
    full_matrices=False)
  stk_pc = [diag[::-1] ** 2, lft[:, ::-1]]
  _store("pc", pc)
  _store("stk_pc", stk_pc)


  fig, ax_s = plt.subplots(2, 1)
  ax_s[0].set(xlabel = 'eigenvalue', ylabel = 'postural pca vs null')
  ax_s[1].set(xlabel = 'eigenvalue', ylabel = 'delayed pca vs null')
  for ax_idx, prfx in enumerate(['', 'stk_']):
    ax = ax_s[ax_idx]
    for plt_idx, (spctm, plt_lbl) in enumerate(
      [(globals()[prfx + 'pc'][0], 'dat'),
       (globals()[prfx + 'num_rmt_spct'], 'num')]
    ):
      ax.scatter(np.real(spctm), plt_idx + np.imag(spctm), label=plt_lbl)
    ax.vlines(globals()[prfx + 'rmt_edge'], -0.3, 1.3, color='red', label='ana')
    ax.vlines(
      [globals()[prfx + 'num_rmt_spct'][0], 
       globals()[prfx + 'num_rmt_spct'][-1]],
      -0.3, 1.3, color='orange'
    )
    ax.legend()

  fig.tight_layout()
  fig.savefig(main_path / f'analysis/result/{brst_type}/03-spctm-{mv_name}.pdf')


In [ ]:
def get_significant_modes():
  global mode_n, mode_traj, mode_sign
  # tolerate some fluctuations
  mode_n = int(np.sum(stk_pc[0] > np.sort(stk_num_rmt_spct)[-1]))
  stk_mode = stk_pc[1][:, -mode_n:].reshape(stk_n, neur_n, mode_n)
  # mode_spat = np.einsum(
  #   "sik,ixy->ksxy", stk_mode, spat)
  mode_traj = stk_pc[1][:, -mode_n:].T @ stk_traj
  mode_sign = np.sign(np.mean(stk_mode, axis=(0, 1)))
  _store("mode_n", mode_n)
  # _store("mode_spat", mode_spat)
  _store("mode_traj", mode_traj)
  _store("mode_sign", mode_sign)

  global mode_pr
  mode_pr = (
    np.mean(stk_pc[0][-mode_n:])**2 
    / np.mean(stk_pc[0][-mode_n:]**2))
  _store("mode_pr", mode_pr)
  
  global avg_pop_pr
  pop_pr = (
    np.mean(neur_pwr:= np.mean(stk_mode**2, 0), 0)**2
    / np.mean(neur_pwr**2, 0))
  avg_pop_pr = (
    np.sum(pop_pr * stk_pc[0][-mode_n:]) 
    / np.sum(stk_pc[0][-mode_n:]))
  _store("avg_pop_pr", avg_pop_pr)
  
  global dyn_var, avg_dyn_var
  global itrm_avg_dyn_var, stb_avg_dyn_var, oth_avg_dyn_var
  # under normalization, 
  # 1 = T * total dynamic variance + T * total static power 
  dyn_var = stk_n * np.sum(np.var(stk_mode, 0), 0)
  stc_pwr = stk_n * np.sum(np.mean(stk_mode, 0)**2, 0)
  avg_dyn_var = (
    np.sum(dyn_var * stk_pc[0][-mode_n:]) 
    / np.sum(stk_pc[0][-mode_n:]))
  itrm_avg_dyn_var = np.nan
  stb_avg_dyn_var = np.nan
  oth_avg_dyn_var = np.nan
  _store("dyn_var", dyn_var)
  _store("avg_dyn_var", avg_dyn_var)
  _store("itrm_avg_dyn_var", itrm_avg_dyn_var)
  _store("stb_avg_dyn_var", stb_avg_dyn_var)
  _store("oth_avg_dyn_var", oth_avg_dyn_var)
  
  global syn_pwr, avg_syn_pwr
  # factor out the effect of PR
  same_pr_stk_mode = stk_mode / np.linalg.norm(stk_mode, axis = 0)
  same_pr_stk_mode = same_pr_stk_mode / np.linalg.norm(
    same_pr_stk_mode, axis=(0,1))
  spt_var = neur_n * np.sum(np.var(same_pr_stk_mode, 1), 0)
  syn_pwr = neur_n * np.sum(np.mean(same_pr_stk_mode, 1)**2, 0)
  avg_syn_pwr = (
    np.sum(syn_pwr * stk_pc[0][-mode_n:]) 
    / np.sum(stk_pc[0][-mode_n:]))
  _store("syn_pwr", syn_pwr)
  _store("avg_syn_pwr", avg_syn_pwr)

  
  fig, ax_s = plt.subplots(2,mode_n, figsize = (2*mode_n, 4+1))
  ax_s[0,0].set(ylabel = 'neuron in dynamical PC')
  ax_s[1,0].set(ylabel = 'dynamical PC trajectory')
  ax_s[-1,0].set(xlabel = 'frame')

  for mode_idx in range(mode_n):
    ax_s[0,mode_idx].set(
      title = "dynamical PC {}\ndynamical PC pr {:.2%}\ndyn var {:.2%}\nsyn pwr {:.2%}".format(
        mode_idx, pop_pr[-mode_idx-1],
        dyn_var[-mode_idx-1], syn_pwr[-mode_idx-1]))
    ax_s[0,mode_idx].plot(
      mode_sign[-mode_idx-1] * np.sqrt(stk_n * neur_n)
      * stk_pc[1][:,-mode_idx-1].reshape((stk_n, neur_n)))
    ax_s[1,mode_idx].plot(mode_sign[-mode_idx-1] * mode_traj[-mode_idx-1])
    
  fig.savefig(main_path / f'analysis/result/{brst_type}/04-significant modes-{mv_name}.pdf')



In [ ]:
def fit_mode_propagator():
  """Fit a one-step propagator on the significant dynamical PC trajectories.

  Eigenvalues only. Each conjugate pair carries one oscillation frequency, and
  the period of the fastest resolved pair is the interpretable timescale
  reported in place of the dynamical variance ratio.

  Fitted on the whole valid recording. A fit restricted to burst windows
  returns roughly the reciprocal of the window, which is set by find_bursts()
  and would make the frequency a readout of the segmentation.
  """
  global prop_freq, prop_period, prop_mu_abs, prop_freq_lag_cv
  global prop_fund_freq, prop_real_n, prop_freq_floor

  lag_s = (1, 2, 3)

  # guard: get_significant_modes() slices with [-mode_n:], and [-0:] would
  # select every dynamical PC rather than none, so refuse to fit in that case
  if mode_n < 2:
    raise RuntimeError(
      f"{mv_name}: mode_n = {mode_n}; need at least 2 dynamical PCs to fit a propagator")

  # mode_traj stores amplitude-carrying dynamical PC trajectories; rows follow ascending stk_pc[0], with the leading component first
  md_traj = mode_traj[::-1]

  # under "padding" only these columns have every delay tap filled; mask by
  # column rather than by burst, since bursts can start inside the padding
  col_lo = (stk_n - 1) * frm_sep if stk_mtd == "padding" else 0
  col_hi = frm_n if stk_mtd == "padding" else stk_frm_n
  vld_n = col_hi - col_lo

  # two cycles inside the usable window is the floor for calling a frequency
  # resolved
  prop_freq_floor = 2.0 / (vld_n * frame_dt)

  def _ols(x_1, x_2):
    return x_2 @ np.linalg.pinv(x_1)

  def _fit(lag):
    """Forward-backward propagator.

    Ordinary least squares is biased toward the stable region under
    measurement noise; forward-backward removes that bias.
    """
    col_s = np.arange(col_lo, col_hi - lag)
    x_1, x_2 = md_traj[:, col_s], md_traj[:, col_s + lag]
    a_fwd = _ols(x_1, x_2)
    a_bwd = _ols(x_2, x_1)
    val_s, vec_s = np.linalg.eig(a_fwd @ np.linalg.inv(a_bwd))
    a_prop = np.real(
      vec_s @ np.diag(np.sqrt(val_s.astype(complex))) @ np.linalg.inv(vec_s))
    return np.linalg.eig(a_prop)

  def _to_freq(val, lag):
    return np.abs(np.angle(val)) / (2 * np.pi * lag * frame_dt)

  # the lag-1 fit defines the dynamical modes; conjugate pairs are deduped by Im > 0
  val_1, vec_1 = _fit(lag_s[0])
  pair_s = np.where(np.imag(val_1) > 0)[0]
  prop_real_n = int(np.sum(np.abs(np.imag(val_1)) <= 0))

  prop_freq = _to_freq(val_1[pair_s], lag_s[0])
  prop_mu_abs = np.abs(val_1[pair_s])
  prop_period = np.where(
    prop_freq > 0, 1.0 / np.maximum(prop_freq, 1e-12), np.inf)

  # Lag consistency, the validation that the linear description holds across
  # timescales. Eigenvectors are the lag-invariant object, since a propagator
  # fitted at lag L is the lag-1 propagator raised to the Lth power, so match
  # dynamical modes are defined by eigenvectors. |mu| sits near 1 for every mode and cannot
  # be used to track them.
  freq_by_lag = np.full((len(lag_s), pair_s.size), np.nan)
  freq_by_lag[0] = prop_freq
  for lag_idx, lag in enumerate(lag_s[1:], start=1):
    val_l, vec_l = _fit(lag)
    ovlp = np.abs(np.conj(vec_1[:, pair_s]).T @ vec_l)
    ovlp = ovlp / np.maximum(
      np.linalg.norm(vec_1[:, pair_s], axis=0)[:, None]
      * np.linalg.norm(vec_l, axis=0)[None, :], 1e-12)
    row_s, col_s = sp.optimize.linear_sum_assignment(-ovlp)
    for row, col in zip(row_s, col_s):
      # a dynamical mode aliases once the lag pushes it past Nyquist
      if prop_freq[row] < 1.0 / (2 * lag * frame_dt):
        freq_by_lag[lag_idx, row] = _to_freq(val_l[col], lag)

  with np.errstate(invalid="ignore"):
    prop_freq_lag_cv = (
      np.nanstd(freq_by_lag, 0) / np.nanmean(freq_by_lag, 0))

  # The fundamental is the lowest resolved frequency, the rate of the collective
  # rhythm. The fastest dynamical mode is deliberately not reported: it tracks frm_sep
  # at Spearman rho = +0.82, and fit_acov() sets frm_sep from each recording's own
  # autocorrelation, so it reads out the analysis timescale. The fundamental does
  # not (rho = -0.38, p = 0.17).
  rslv = prop_freq >= prop_freq_floor
  prop_fund_freq = float(np.min(prop_freq[rslv])) if np.any(rslv) else np.nan

  for _nm in ("prop_freq", "prop_period", "prop_mu_abs", "prop_freq_lag_cv",
              "prop_fund_freq", "prop_real_n",
              "prop_freq_floor"):
    _store(_nm, globals()[_nm])

  fig, ax_s = plt.subplots(1, 2, figsize=(6, 3))
  fig.suptitle(mv_name)
  ax = ax_s[0]
  ax.scatter(np.real(val_1), np.imag(val_1), s=12)
  ang = np.linspace(0, 2 * np.pi, 200)
  ax.plot(np.cos(ang), np.sin(ang), lw=0.5, color="gray")
  ax.set(xlabel="Re mu", ylabel="Im mu", aspect="equal")
  ax = ax_s[1]
  ax.scatter(prop_freq, prop_freq_lag_cv, s=12)
  ax.axvline(prop_freq_floor, ls="--", lw=0.5, color="red")
  ax.set(xlabel="frequency (Hz)", ylabel="lag CV", xscale="log")
  fig.tight_layout()
  fig.savefig(
    main_path / f'analysis/result/{brst_type}/06-propagator-{mv_name}.pdf')


In [ ]:
def evaluate_partition_mode_significance():
  global part_edge_hi
  global itrm_adj_spct, stb_adj_spct, oth_adj_spct
  global itrm_sig_mode_n, stb_sig_mode_n, oth_sig_mode_n
  global itrm_avg_dyn_var, stb_avg_dyn_var, oth_avg_dyn_var

  eig_val = np.asarray(stk_pc[0][-mode_n:], dtype=float)
  stk_mode = stk_pc[1][:, -mode_n:].reshape(stk_n, neur_n, mode_n)
  mode_pwr_all = np.mean(stk_mode ** 2, axis=(0, 1))
  # PCA loading vectors are normalized, so total component power is 1 for each component.
  part_edge_hi = float(np.max(stk_num_rmt_spct))

  def _safe_ratio(numer, denom):
    numer = np.asarray(numer, dtype=float)
    denom = np.asarray(denom, dtype=float)
    return np.divide(numer, denom, out=np.zeros_like(numer), where=(denom > 0))

  def _part_adj_spct(mask):
    mask = np.asarray(mask, dtype=bool)
    if np.sum(mask) == 0:
      return np.full(mode_n, np.nan)
    part_mode_pwr = np.mean(stk_mode[:, mask, :] ** 2, axis=(0, 1))
    return eig_val * _safe_ratio(part_mode_pwr, mode_pwr_all)

  def _part_avg_dyn_var(mask):
    mask = np.asarray(mask, dtype=bool)
    if np.sum(mask) == 0:
      return np.nan

    part_dyn_var = stk_n * np.sum(np.var(stk_mode[:, mask, :], axis=0), axis=0)
    part_stc_pwr = stk_n * np.sum(np.mean(stk_mode[:, mask, :], axis=0) ** 2, axis=0)
    part_dyn_var_frac = _safe_ratio(part_dyn_var, part_dyn_var + part_stc_pwr)

    part_mode_pwr = np.sum(stk_mode[:, mask, :] ** 2, axis=(0, 1))
    adj_eig_val = eig_val * part_mode_pwr
    if np.sum(adj_eig_val) <= 0:
      return np.nan
    return np.sum(part_dyn_var_frac * adj_eig_val) / np.sum(adj_eig_val)

  itrm_adj_spct = _part_adj_spct(itrm_neur)
  stb_adj_spct = _part_adj_spct(stb_neur)
  oth_adj_spct = _part_adj_spct(oth_neur)

  itrm_sig_mode_n = int(np.sum(itrm_adj_spct > part_edge_hi)) if np.any(np.isfinite(itrm_adj_spct)) else 0
  stb_sig_mode_n = int(np.sum(stb_adj_spct > part_edge_hi)) if np.any(np.isfinite(stb_adj_spct)) else 0
  oth_sig_mode_n = int(np.sum(oth_adj_spct > part_edge_hi)) if np.any(np.isfinite(oth_adj_spct)) else 0

  itrm_avg_dyn_var = _part_avg_dyn_var(itrm_neur)
  stb_avg_dyn_var = _part_avg_dyn_var(stb_neur)
  oth_avg_dyn_var = _part_avg_dyn_var(oth_neur)

  _store("part_edge_hi", part_edge_hi)
  _store("itrm_adj_spct", itrm_adj_spct)
  _store("stb_adj_spct", stb_adj_spct)
  _store("oth_adj_spct", oth_adj_spct)
  _store("itrm_sig_mode_n", itrm_sig_mode_n)
  _store("stb_sig_mode_n", stb_sig_mode_n)
  _store("oth_sig_mode_n", oth_sig_mode_n)
  _store("itrm_avg_dyn_var", itrm_avg_dyn_var)
  _store("stb_avg_dyn_var", stb_avg_dyn_var)
  _store("oth_avg_dyn_var", oth_avg_dyn_var)


In [ ]:
def get_denoised_traj():
  global flt_traj, rlb_traj
  # denoised trajectory
  rlb_stk_md_traj = stk_pc[1][:,-mode_n:,None] * mode_traj
  # rlb_stk_md_traj = (
  #   dyn_mode.astype(float)[:,None] # dyn_mode = dyn_var > 0.5
  #   * stk_pc[1][:,-mode_n:,None] * mode_traj)
  rlb_md_traj = undo_delay_embed(
    rlb_stk_md_traj, stk_n, frm_sep, stk_mtd)
  # sum over significant dynamical PCs
  rlb_traj = np.sum(rlb_md_traj, 1)

  rlb_traj = sp.ndimage.uniform_filter1d(
    rlb_traj, size=frm_sep, axis=1, mode="reflect")

  # mean filtering on both trajectories (time axis)
  flt_traj = sp.ndimage.uniform_filter1d(
    traj, size=stk_n * frm_sep, axis=1, mode="reflect")
  _store("flt_traj", flt_traj)
  _store("rlb_traj", rlb_traj)

  global flt_all_brst_traj, all_brst_traj
  # bursts from filtered trajectories
  flt_all_brst_traj = np.array([
    flt_traj[:,brst_win[0]:brst_win[1]] for brst_win in all_brst_win])
  all_brst_traj = np.array([
    rlb_traj[:,brst_win[0]:brst_win[1]] for brst_win in all_brst_win])
  _store("flt_all_brst_traj", flt_all_brst_traj)
  _store("all_brst_traj", all_brst_traj)

  fig, ax = plt.subplots(1,1)
  ax.plot(rlb_traj.T, lw = 1)
  ax.set(xlabel = 'frame', ylabel = 'activity')
  fig.savefig(main_path / f'analysis/result/{brst_type}/05-denoised traj-{mv_name}.pdf')


In [ ]:
def extract_timing():
  global flt_all_spk_time, all_spk_time
  softmax_beta = 3

  def _extract_one_spk_time(one_all_brst_traj, one_res_std):
    one_all_spk_time = np.zeros(one_all_brst_traj.shape[:2])
    for brst_idx in range(brst_n):
      for neur_idx in range(neur_n):
        brst_traj = one_all_brst_traj[brst_idx, neur_idx]

        baseline = np.linspace(brst_traj[0], brst_traj[-1], brst_traj.shape[0])  # traj too short to use scipy detrend
        dev = brst_traj - baseline

        if np.max(dev) <= one_res_std[neur_idx]:
          # if peak greater than typical fluctuation in bulk
          one_all_spk_time[brst_idx, neur_idx] = np.nan
          continue

        # find peaks on detrended trajectory
        peaks = sp.signal.find_peaks(dev)[0]
        # reject monotonic increase
        if len(peaks) == 0:
          one_all_spk_time[brst_idx, neur_idx] = np.nan
          continue

        # decide which: largest peak
        peak = peaks[np.argmax(dev[peaks])]

        # find edges
        _, *edge = sp.signal.peak_prominences(dev, [peak])
        edge = np.array(edge).astype(int)[:,0]
        one_all_spk_time[brst_idx, neur_idx] = (
          np.sum(
            np.arange(*edge) * np.exp(softmax_beta * dev[slice(*edge)]),
            -1)
          / np.sum(np.exp(softmax_beta * dev[slice(*edge)]), -1))

    one_all_spk_time = one_all_spk_time - np.nanmedian(one_all_spk_time, 1)[:,None]
    return one_all_spk_time

  # filtered trajectory timing (rms-gated)
  # flt_res_std = np.std(flt_traj - traj, axis=1)
  flt_res_std = np.zeros(neur_n)  # no standard for bursting
  flt_all_spk_time = _extract_one_spk_time(
    flt_all_brst_traj, one_res_std=flt_res_std)

  # denoised timing (rms-gated)
  rlb_res_std = np.std(rlb_traj - traj, axis=1)
  all_spk_time = _extract_one_spk_time(
    all_brst_traj, one_res_std=rlb_res_std)

  _store("flt_all_spk_time", flt_all_spk_time)
  _store("all_spk_time", all_spk_time)


  global brst_part, brst_p, brst_neur, brst_neur_n, brst_neur_frac, ustb_rcrtmnt
  global stb_neur, stb_neur_n, itrm_neur, itrm_neur_n, oth_neur, oth_neur_n
  brst_part = (~np.isnan(all_spk_time))
  brst_p = np.mean(brst_part, 0)
  brst_neur = (~np.isclose(brst_p, 0))
  brst_neur_n = np.sum(brst_neur)
  stb_neur = np.isclose(brst_p, 1)
  stb_neur_n = np.sum(stb_neur)
  itrm_neur = brst_neur & ~stb_neur
  itrm_neur_n = np.sum(itrm_neur)
  oth_neur = np.isclose(brst_p, 0)
  oth_neur_n = np.sum(oth_neur)
  brst_neur_frac = np.mean(brst_p)**2 / np.mean(brst_p**2)
  ustb_rcrtmnt = (
    0 if neur_n == brst_neur_n else
    (neur_n - brst_neur_n)
    * np.nanmean(1-brst_p[~brst_neur])**2 / np.nanmean((1-brst_p[~brst_neur])**2))
  _store("brst_part", brst_part)
  _store("brst_p", brst_p)
  _store("brst_neur", brst_neur)
  _store("brst_neur_n", brst_neur_n)
  _store("stb_neur", stb_neur)
  _store("stb_neur_n", stb_neur_n)
  _store("itrm_neur", itrm_neur)
  _store("itrm_neur_n", itrm_neur_n)
  _store("oth_neur", oth_neur)
  _store("oth_neur_n", oth_neur_n)
  _store("brst_neur_frac", brst_neur_frac)
  _store("ustb_rcrtmnt", ustb_rcrtmnt)

  global mean_spk_time, flt_mean_spk_time, spk_time_std, spk_time_ord, prop_rate, tail_dev

  # average latency (primary path: denoised)
  mean_spk_time = np.full(neur_n, np.nan)
  spk_time_std = np.full(neur_n, np.nan)
  mean_spk_time[brst_neur] = np.nanmean(all_spk_time[:,brst_neur], 0)
  spk_time_std[brst_neur] = np.nanstd(all_spk_time[:,brst_neur], 0)

  # ordering is based on filtered trajectory timings
  flt_mean_spk_time = np.full(neur_n, np.nan)
  flt_mean_spk_time[brst_neur] = np.nanmean(flt_all_spk_time[:,brst_neur], 0)
  spk_time_ord = np.argsort(flt_mean_spk_time)

  # middle quantile fit
  stable_mean_spk_time = mean_spk_time[spk_time_ord[:brst_neur_n]]
  stable_mean_spk_time = stable_mean_spk_time - np.nanmedian(stable_mean_spk_time)
  q25, q75 = np.nanquantile(stable_mean_spk_time, [0.25, 0.75])
  mid_mask = (stable_mean_spk_time >= q25) & (stable_mean_spk_time <= q75)
  rnk_med = np.arange(brst_neur_n, dtype=float)
  rnk_med = rnk_med - np.nanmedian(rnk_med)
  rnk_med_mid = rnk_med[mid_mask]
  lat_mid = stable_mean_spk_time[mid_mask]
  prop_rate = np.sum(rnk_med_mid**2) / np.sum(rnk_med_mid * lat_mid)

  # tail deviation
  fit_spk_time = rnk_med / prop_rate
  tail_mask = ~mid_mask
  tail_dev = np.mean(np.abs(stable_mean_spk_time[tail_mask] - fit_spk_time[tail_mask]))

  _store("mean_spk_time", mean_spk_time)
  _store("flt_mean_spk_time", flt_mean_spk_time)
  _store("spk_time_std", spk_time_std)
  _store("spk_time_ord", spk_time_ord)
  _store("prop_rate", prop_rate)
  _store("tail_dev", tail_dev)

  fig, ax_s = plt.subplots(2,1)
  ax_s[0].matshow(brst_part, vmin = 0, vmax = 1)
  ax_s[0].set(xlabel = 'neuron', ylabel = 'burst')
  ax_s[1].hist(brst_p)
  ax_s[1].scatter(brst_p, np.full(neur_n,-3), alpha = 0.6, marker = 'x')
  ax_s[1].set(xlabel = '%participation in bursts', ylabel = 'neuron count')
  fig.savefig(main_path / f'analysis/result/{brst_type}/06-participation-{mv_name}.pdf')


In [ ]:
def analyze_order_consistency():
  global order_cor, kendall, flt_order_cor, flt_kendall

  def _compute_order_metrics(one_all_spk_time):
    eligible_neur = np.any(np.isfinite(one_all_spk_time), axis=0)
    one_order_cor_mat = np.full((brst_n, brst_n), np.nan)
    one_kendall_mat = np.full((brst_n, brst_n), np.nan)
    for idx_1 in range(brst_n):
      for idx_2 in range(idx_1 + 1, brst_n):
        pair_neur = (
          eligible_neur
          & np.isfinite(one_all_spk_time[idx_1])
          & np.isfinite(one_all_spk_time[idx_2]))
        if np.sum(pair_neur) < 2:
          continue
        time_1 = one_all_spk_time[idx_1, pair_neur]
        time_2 = one_all_spk_time[idx_2, pair_neur]
        one_order_cor_mat[idx_1, idx_2] = sp.stats.pearsonr(time_1, time_2)[0]
        one_kendall_mat[idx_1, idx_2] = sp.stats.kendalltau(time_1, time_2)[0]
    pair_idx = np.triu_indices(brst_n, 1)
    one_order_cor = np.nanmean(one_order_cor_mat[pair_idx])
    one_kendall = np.nanmean(one_kendall_mat[pair_idx])
    return one_order_cor, one_kendall

  # primary metrics from denoised timings
  order_cor, kendall = _compute_order_metrics(all_spk_time)
  # corresponding metrics from CNMF timings
  flt_order_cor, flt_kendall = _compute_order_metrics(flt_all_spk_time)

  _store('order_cor', order_cor)
  _store('kendall', kendall)
  _store('flt_order_cor', flt_order_cor)
  _store('flt_kendall', flt_kendall)

  fig, ax = plt.subplots(1,1)
  ax.set(title = f"kendall {kendall:.2f}, order cor = {order_cor:}",
         xlabel = 'rank of mean time', ylabel = 'latency in burst')

  ax.errorbar(
    np.arange(brst_neur_n),
    mean_spk_time[spk_time_ord[:brst_neur_n]],
    spk_time_std[spk_time_ord[:brst_neur_n]])

  # fitted line
  stable_mean_spk_time = mean_spk_time[spk_time_ord[:brst_neur_n]]
  rnk_med = np.arange(brst_neur_n, dtype=float)
  rnk_med = rnk_med - np.nanmedian(rnk_med)
  fit_spk_time = rnk_med / prop_rate + np.nanmedian(stable_mean_spk_time)
  ax.plot(np.arange(brst_neur_n), fit_spk_time, c='k', lw=2, ls='--')
  finite_lat = mean_spk_time[np.isfinite(mean_spk_time)]
  lat_abs = float(np.nanmax(np.abs(finite_lat))) if finite_lat.size > 0 else np.nan
  if (not np.isfinite(lat_abs)) or (lat_abs <= 0):
    lat_abs = 1.0
  vmin, vmax = -lat_abs, lat_abs

  for brst_neur_idx in range(brst_neur_n):
    one_order = spk_time_ord[brst_neur_idx]
    ax.scatter(
      np.full(brst_n, brst_neur_idx),
      all_spk_time[:, one_order],
      c=np.full(brst_n, mean_spk_time[one_order]),
      vmin = vmin, vmax = vmax,
      cmap='summer', alpha = 0.7)

  # colorbar
  sc = ax.scatter(
    [], [], c=[],
    vmin = vmin, vmax = vmax,
    cmap='summer')
  fig.colorbar(sc, ax=ax)

  fig.savefig(main_path / f'analysis/result/{brst_type}/07-order-{mv_name}.pdf')


In [ ]:
def mbd_n_clstr():
  global tot_mds, mds_ord, tot_mds_dist
  stk_mode = (
    np.sqrt(stk_n * neur_n) 
    * stk_pc[1][:, -mode_n:].reshape(stk_n, neur_n, mode_n))
  stk_mode_pwr = np.sqrt(np.mean(stk_mode ** 2, axis=0, keepdims=True))
  stk_mode_pwr = np.where(np.isclose(stk_mode_pwr, 0), 1.0, stk_mode_pwr)
  stk_mode = stk_mode / stk_mode_pwr

  # mds
  all_mode_corr = np.einsum(
    'tnk,tmk->nmk', stk_mode, stk_mode) / stk_n
  mode_wgt = stk_pc[0][-mode_n:]
  mode_wgt = mode_wgt / np.sum(mode_wgt)
  agg_mode_corr = np.einsum('nmk,k->nm', all_mode_corr, mode_wgt)
  plt.figure()
  plt.imshow(agg_mode_corr)
  plt.colorbar()
  tot_mds_dist = np.sqrt(2 * np.maximum(0, 1 - agg_mode_corr))

  # single mds embedding from the aggregate distance
  tot_mds = skl.manifold.MDS(
    dissimilarity="precomputed",
    n_components=2,
    random_state=0,
    normalized_stress="auto"
    ).fit_transform(tot_mds_dist)
  tot_mds = tot_mds - np.mean(tot_mds, axis=0, keepdims=True)
  mds_ord = np.argsort(tot_mds[:,0])
  _store("tot_mds", tot_mds)
  _store("mds_ord", mds_ord)


  # clusterability
  global clstr_scr, clstr_scr_s, no_clstr_frac, clstr_lbl, clstr_n
  global part_lbl, part_clstr_scr

  # dist
  tot_mds_dist[np.diag_indices(neur_n)] = 0
  _store("tot_mds_dist", tot_mds_dist.copy())
  clstr_alg = hdbscan.HDBSCAN(
    metric = "precomputed",
    min_cluster_size = max(2, int(np.ceil(neur_n / 10))),
    gen_min_span_tree = True)
  clstr_alg.fit(tot_mds_dist)
  clstr_lbl = clstr_alg.labels_
  clstr_scr = float(hdbscan.validity.validity_index(
    tot_mds_dist, clstr_lbl,
    metric="precomputed", d=stk_n))
  non_noise_lbl_scr_s = np.sort(np.unique(clstr_lbl[clstr_lbl != -1]))
  if non_noise_lbl_scr_s.size >= 2:
    try:
      _, per_clstr_scr = hdbscan.validity.validity_index(
        tot_mds_dist, clstr_lbl,
        metric="precomputed", d=stk_n, per_cluster_scores=True)
      per_clstr_scr = np.asarray(per_clstr_scr, dtype=float)
      if per_clstr_scr.size == non_noise_lbl_scr_s.size:
        clstr_scr_s = {int(lbl): float(scr) for lbl, scr in zip(non_noise_lbl_scr_s, per_clstr_scr)}
      else:
        clstr_scr_s = {
          int(lbl): float(per_clstr_scr[int(lbl)]) if int(lbl) < per_clstr_scr.size else np.nan
          for lbl in non_noise_lbl_scr_s
        }
    except (TypeError, ValueError):
      clstr_scr_s = {int(lbl): np.nan for lbl in non_noise_lbl_scr_s}
  else:
    clstr_scr_s = {}
  no_clstr_frac = float(np.mean(clstr_lbl == -1))
  clstr_n = len(set(clstr_lbl)) - (1 if -1 in clstr_lbl else 0)
  part_lbl = np.full(neur_n, 0, dtype=int)
  part_lbl[itrm_neur] = 1
  part_lbl[stb_neur] = 2
  for lbl in np.unique(part_lbl):
    if np.sum(part_lbl == lbl) < 2:
      part_lbl[part_lbl == lbl] = -1
  if np.sum(np.unique(part_lbl) != -1) <= 1:
    part_lbl[:] = -1
    part_clstr_scr = np.nan
  else:
    try:
      part_clstr_scr = float(hdbscan.validity.validity_index(
        tot_mds_dist, part_lbl,
        metric="precomputed", d=stk_n))
    except ValueError:
      part_clstr_scr = np.nan

  _store("clstr_scr", clstr_scr)
  _store("clstr_scr_s", clstr_scr_s)
  _store("no_clstr_frac", no_clstr_frac)
  _store("clstr_lbl", clstr_lbl)
  _store("clstr_n", clstr_n)
  _store("part_lbl", part_lbl)
  _store("part_clstr_scr", part_clstr_scr)


  # cluster in space and time
  global x_clstr_scr, t_clstr_scr, t_self_clstr_scr
  x_dist = np.linalg.norm(com[:,None,:] - com, axis = 2)
  non_noise_mask = (clstr_lbl != -1)
  non_noise_lbl_s = np.unique(clstr_lbl[non_noise_mask])
  if non_noise_lbl_s.size >= 2:
    try:
      x_clstr_scr = float(hdbscan.validity.validity_index(
        x_dist[np.ix_(non_noise_mask, non_noise_mask)],
        clstr_lbl[non_noise_mask],
        metric="precomputed", d=com.shape[1]))
    except ValueError:
      x_clstr_scr = np.nan
  else:
    x_clstr_scr = np.nan
  _store("x_clstr_scr", x_clstr_scr)

  t_dist = np.abs(mean_spk_time[:,None] - mean_spk_time[None,:])
  finite_time_mask = non_noise_mask & ~np.isnan(mean_spk_time)
  finite_time_lbl_s = np.unique(clstr_lbl[finite_time_mask])
  if finite_time_lbl_s.size >= 2:
    try:
      t_clstr_scr = float(hdbscan.validity.validity_index(
        t_dist[np.ix_(finite_time_mask, finite_time_mask)],
        clstr_lbl[finite_time_mask],
        metric="precomputed", d=1))
    except ValueError:
      t_clstr_scr = np.nan
  else:
    t_clstr_scr = np.nan
  _store("t_clstr_scr", t_clstr_scr)

  # Self-cluster latency profiles using all available burst timings.
  # Rows that are all NaN carry no timing dimension; partially observed neurons remain.
  t_profile = np.asarray(all_spk_time, dtype=float)
  nonempty_burst_mask = np.any(np.isfinite(t_profile), axis=1)
  t_profile = t_profile[nonempty_burst_mask]
  retained_burst_n = int(t_profile.shape[0])
  finite_self_time_mask = np.any(np.isfinite(t_profile), axis=0) if retained_burst_n > 0 else np.zeros(neur_n, dtype=bool)
  finite_self_time_idx = np.where(finite_self_time_mask)[0]
  n_finite_time = int(finite_self_time_idx.size)
  if n_finite_time >= 2:
    t_profile_valid = t_profile[:, finite_self_time_mask]
    valid_pair = np.isfinite(t_profile_valid[:, :, None]) & np.isfinite(t_profile_valid[:, None, :])
    sq_diff = (t_profile_valid[:, :, None] - t_profile_valid[:, None, :]) ** 2
    shared_n = np.sum(valid_pair, axis=0)
    sq_sum = np.nansum(np.where(valid_pair, sq_diff, np.nan), axis=0)
    t_self_dist = np.full((n_finite_time, n_finite_time), np.nan, dtype=float)
    shared_pair_mask = shared_n > 0
    t_self_dist[shared_pair_mask] = np.sqrt(sq_sum[shared_pair_mask] / shared_n[shared_pair_mask])

    no_shared_pair_mask = ~shared_pair_mask
    if np.any(no_shared_pair_mask):
      mean_time_valid = np.asarray(mean_spk_time, dtype=float)[finite_self_time_mask]
      mean_time_dist = np.abs(mean_time_valid[:, None] - mean_time_valid[None, :])
      t_self_dist[no_shared_pair_mask] = mean_time_dist[no_shared_pair_mask]

    np.fill_diagonal(t_self_dist, 0.0)
    finite_dist_mask = np.all(np.isfinite(t_self_dist), axis=0)
    t_self_dist = t_self_dist[np.ix_(finite_dist_mask, finite_dist_mask)]
    n_finite_time = int(t_self_dist.shape[0])

    if n_finite_time >= 2:
      t_self_clstr_alg = hdbscan.HDBSCAN(
        metric="precomputed",
        min_cluster_size=max(2, int(np.ceil(n_finite_time / 10))),
        gen_min_span_tree=True)
      t_self_clstr_alg.fit(t_self_dist)
      t_self_lbl = t_self_clstr_alg.labels_
      t_self_non_noise_lbl_s = np.unique(t_self_lbl[t_self_lbl != -1])
      if t_self_non_noise_lbl_s.size >= 2:
        try:
          t_self_clstr_scr = float(hdbscan.validity.validity_index(
            t_self_dist,
            t_self_lbl,
            metric="precomputed", d=max(1, retained_burst_n)))
        except ValueError:
          t_self_clstr_scr = np.nan
      else:
        # Adequate latency data but no multi-cluster solution: no detected
        # latency-profile clustering, represented as zero in Figure 3E.
        t_self_clstr_scr = 0.0
    else:
      t_self_clstr_scr = np.nan
  else:
    t_self_clstr_scr = np.nan
  _store("t_self_clstr_scr", t_self_clstr_scr)


  # visualizaiton, so no errorbars
  fig, ax_s = plt.subplots(1,2, figsize = (10,4))
  ax_l, ax_r = ax_s

  # shared color mapping for clusters (-1 is gray)
  uniq_lbl_s = np.sort(np.unique(clstr_lbl))
  non_noise_lbl_s = [lbl for lbl in uniq_lbl_s if lbl != -1]
  lbl_color = {-1: 'gray'}
  tab10_colors = plt.get_cmap('tab10').colors
  for idx, lbl in enumerate(non_noise_lbl_s):
    lbl_color[lbl] = tab10_colors[idx % len(tab10_colors)]

  # left plot: mds embedding
  ax_l.set(
    title = f"clstr_scr: {clstr_scr:.2f}, noise_frac: {no_clstr_frac:.2f}, n: {clstr_n:.2f}",
    xlabel = 'mds-1', ylabel = 'mds-2')
  finite_lat = mean_spk_time[np.isfinite(mean_spk_time)]
  lat_abs = float(np.nanmax(np.abs(finite_lat))) if finite_lat.size > 0 else np.nan
  if (not np.isfinite(lat_abs)) or (lat_abs <= 0):
    lat_abs = 1.0
  vmin, vmax = -lat_abs, lat_abs
  ax_l.scatter(
    *tot_mds[brst_neur].T,
    c=mean_spk_time[brst_neur],
    vmin=vmin, vmax=vmax,
    cmap='summer', alpha=0.7)
  ax_l.scatter(
    *tot_mds[~brst_neur].T,
    color='gray', marker='x', alpha=0.7)
  for lbl in uniq_lbl_s:
    cl_mask = (clstr_lbl == lbl)
    ax_l.scatter(
      *tot_mds[cl_mask].T,
      facecolors='none',
      edgecolors=lbl_color.get(lbl, 'gray'),
      s=34,
      linewidths=0.8,
      alpha=0.9)
  for lbl in uniq_lbl_s:
    ax_l.scatter([], [], color=lbl_color.get(lbl, 'gray'), label=f'label {lbl}')
  leg = ax_l.legend(loc='best', fontsize=8, title='cluster', framealpha=0.95)
  leg.get_frame().set_facecolor('white')
  leg.get_frame().set_alpha(0.95)
  sc = ax_l.scatter([], [], c=[], vmin=vmin, vmax=vmax, cmap='summer')
  fig.colorbar(sc, ax=ax_l)

  # right plot: cluster positions
  ax_r.set(title = f"x_clstr_scr: {x_clstr_scr:.2f}, t_clstr_scr: {t_clstr_scr:.2f}")
  plot_res = 8
  plot_spat = spat[:, ::plot_res, ::plot_res]
  plot_com = com / plot_res
  ax_r.imshow( # for cell contour, makes aspect 1
    np.mean(plot_spat, 0) > 0, cmap = 'binary', alpha = 0.)
  for lbl in uniq_lbl_s:
    ax_r.scatter(
      *plot_com[clstr_lbl == lbl].T,
      color=lbl_color.get(lbl, 'gray'), s=10)


  fig.savefig(main_path / f'analysis/result/{brst_type}/08-cluster-{mv_name}.pdf')


In [ ]:
def cluster_centroid_validation():
  global clstr_centroid_axes, clstr_centroid_contribution, clstr_centroid_var_ratio

  non_noise_mask = np.asarray(clstr_lbl, dtype=int) != -1
  non_noise_label_s = np.sort(np.unique(np.asarray(clstr_lbl, dtype=int)[non_noise_mask]))
  axis_n = max(0, int(non_noise_label_s.size) - 1)
  clstr_centroid_axes = np.full((axis_n, mode_n, stk_n), np.nan, dtype=float)
  clstr_centroid_contribution = np.full((axis_n, mode_n), np.nan, dtype=float)
  clstr_centroid_var_ratio = np.full(axis_n, np.nan, dtype=float)

  fig_path = main_path / f'analysis/result/{brst_type}/08-centroid-{mv_name}.pdf'
  if (axis_n <= 0) or (mode_n <= 0) or (stk_n <= 0):
    _store('clstr_centroid_axes', clstr_centroid_axes)
    _store('clstr_centroid_contribution', clstr_centroid_contribution)
    _store('clstr_centroid_var_ratio', clstr_centroid_var_ratio)
    return

  # Match the normalized activity-form representation used by MDS/HDBSCAN.
  stk_mode = (
    np.sqrt(stk_n * neur_n)
    * stk_pc[1][:, -mode_n:].reshape(stk_n, neur_n, mode_n))
  stk_mode_pwr = np.sqrt(np.mean(stk_mode ** 2, axis=0, keepdims=True))
  stk_mode_pwr = np.where(np.isclose(stk_mode_pwr, 0), 1.0, stk_mode_pwr)
  stk_mode = stk_mode / stk_mode_pwr

  mode_var_raw = np.asarray(stk_pc[0][-mode_n:], dtype=float)
  mode_wgt = np.asarray(mode_var_raw, dtype=float)
  if np.any(np.isfinite(mode_wgt)) and (np.nansum(mode_wgt) > 0):
    mode_wgt = mode_wgt / np.nansum(mode_wgt)
  else:
    mode_wgt = np.ones(mode_n, dtype=float) / mode_n
  mode_scale = np.sqrt(np.maximum(mode_wgt, 0.0))

  neur_mode_profile = np.moveaxis(stk_mode, 1, 0)
  neur_mode_profile = np.moveaxis(neur_mode_profile, 1, 2)
  x_all = neur_mode_profile.reshape(neur_n, mode_n * stk_n)
  feature_scale = np.repeat(mode_scale, stk_n)
  y_all = np.asarray(clstr_lbl, dtype=int)
  x = x_all[non_noise_mask]
  y = y_all[non_noise_mask]

  valid_feature_mask = np.all(np.isfinite(x), axis=0)
  if (np.sum(valid_feature_mask) < 1) or (x.shape[0] <= non_noise_label_s.size):
    _store('clstr_centroid_axes', clstr_centroid_axes)
    _store('clstr_centroid_contribution', clstr_centroid_contribution)
    _store('clstr_centroid_var_ratio', clstr_centroid_var_ratio)
    return

  # Standardize in the non-noise population, then apply dynamical PC variance weights.
  # Centroid axes describe cluster mean profile separation without covariance whitening.
  scaler = skl_pre.StandardScaler()
  x_std = scaler.fit_transform(x[:, valid_feature_mask])
  x_std = x_std * feature_scale[valid_feature_mask][None, :]

  x_all_weighted = np.full((neur_n, mode_n * stk_n), np.nan, dtype=float)
  x_all_std = scaler.transform(x_all[:, valid_feature_mask])
  x_all_weighted[:, valid_feature_mask] = x_all_std * feature_scale[valid_feature_mask][None, :]
  x_all_weighted = x_all_weighted.reshape(neur_n, mode_n, stk_n)

  centroid_s = []
  weight_s = []
  for lbl in non_noise_label_s:
    mask = y == lbl
    if not np.any(mask):
      continue
    centroid_s.append(np.nanmean(x_std[mask], axis=0))
    weight_s.append(int(np.sum(mask)))
  centroid_s = np.asarray(centroid_s, dtype=float)
  weight_s = np.asarray(weight_s, dtype=float)
  if centroid_s.shape[0] < 2:
    _store('clstr_centroid_axes', clstr_centroid_axes)
    _store('clstr_centroid_contribution', clstr_centroid_contribution)
    _store('clstr_centroid_var_ratio', clstr_centroid_var_ratio)
    return

  centroid_mean = np.average(centroid_s, axis=0, weights=weight_s)
  centroid_ctr = centroid_s - centroid_mean[None, :]
  centroid_weighted = centroid_ctr * np.sqrt(weight_s[:, None])
  _, sing_val, vt = np.linalg.svd(centroid_weighted, full_matrices=False)
  comp_n = min(axis_n, vt.shape[0])
  var_val = sing_val[:comp_n] ** 2
  var_sum = float(np.sum(sing_val ** 2))
  if np.isfinite(var_sum) and (var_sum > 0):
    clstr_centroid_var_ratio[:comp_n] = var_val / var_sum

  valid_axes = np.full((comp_n, mode_n * stk_n), np.nan, dtype=float)
  valid_axes[:, valid_feature_mask] = vt[:comp_n]
  for axis_idx in range(comp_n):
    axis_vec = valid_axes[axis_idx]
    axis_norm = float(np.sqrt(np.nansum(axis_vec ** 2)))
    if np.isfinite(axis_norm) and (axis_norm > 0):
      valid_axes[axis_idx] = axis_vec / axis_norm
  clstr_centroid_axes[:comp_n] = valid_axes.reshape(comp_n, mode_n, stk_n)
  clstr_centroid_contribution[:comp_n] = np.nansum(clstr_centroid_axes[:comp_n] ** 2, axis=2)
  denom = np.nansum(clstr_centroid_contribution[:comp_n], axis=1, keepdims=True)
  clstr_centroid_contribution[:comp_n] = np.divide(
    clstr_centroid_contribution[:comp_n], denom,
    out=np.full_like(clstr_centroid_contribution[:comp_n], np.nan),
    where=(denom > 0))

  centroid_axis_score = np.full((comp_n, len(non_noise_label_s)), np.nan, dtype=float)
  for axis_idx in range(comp_n):
    axis_valid = valid_axes[axis_idx, valid_feature_mask]
    centroid_axis_score[axis_idx] = centroid_s @ axis_valid

  centroid_proj = np.full((axis_n, neur_n, mode_n, stk_n), np.nan, dtype=float)
  for axis_idx in range(comp_n):
    for mode_idx in range(mode_n):
      axis_mode = clstr_centroid_axes[axis_idx, mode_idx]
      axis_norm2 = float(np.nansum(axis_mode ** 2))
      if (not np.isfinite(axis_norm2)) or np.isclose(axis_norm2, 0):
        centroid_proj[axis_idx, :, mode_idx, :] = 0.0
        continue
      score_mode = np.nansum(x_all_weighted[:, mode_idx, :] * axis_mode[None, :], axis=1)
      centroid_proj[axis_idx, :, mode_idx, :] = score_mode[:, None] * axis_mode[None, :] / axis_norm2

  uniq_lbl_s = np.sort(np.unique(y_all))
  lbl_color = {-1: 'gray'}
  lbl_name = {-1: 'noise'}
  tab10_colors = plt.get_cmap('tab10').colors
  tab10_name_s = ['blue', 'orange', 'green', 'red', 'purple', 'brown', 'pink', 'gray', 'olive', 'cyan']
  non_noise_lbl_plot_s = [int(lbl) for lbl in uniq_lbl_s if int(lbl) != -1]
  for idx, lbl in enumerate(non_noise_lbl_plot_s):
    lbl_color[lbl] = tab10_colors[idx % len(tab10_colors)]
    lbl_name[lbl] = tab10_name_s[idx % len(tab10_name_s)]

  finite_profile = neur_mode_profile[np.isfinite(neur_mode_profile)]
  profile_abs = float(np.nanmax(np.abs(finite_profile))) if finite_profile.size > 0 else np.nan
  if (not np.isfinite(profile_abs)) or (profile_abs <= 0):
    profile_abs = 1.0

  finite_proj = centroid_proj[np.isfinite(centroid_proj)]
  proj_abs = float(np.nanmax(np.abs(finite_proj))) if finite_proj.size > 0 else np.nan
  if (not np.isfinite(proj_abs)) or (proj_abs <= 0):
    proj_abs = 1.0

  def _plot_cluster_traces(ax, trace_arr, mode_idx, alpha, lw, zorder=1):
    for lbl in uniq_lbl_s:
      lbl = int(lbl)
      mask = y_all == lbl
      if not np.any(mask):
        continue
      for one_trace in trace_arr[mask, mode_idx, :]:
        ax.plot(
          np.arange(stk_n), one_trace,
          color=lbl_color.get(lbl, 'gray'),
          alpha=alpha,
          lw=lw,
          zorder=zorder,
        )
    ax.axhline(0, color='0.5', lw=0.6, ls='--', zorder=0)

  row_n = axis_n + 1
  fig, ax_s = plt.subplots(
    row_n, mode_n, squeeze=False,
    figsize=(max(1.9 * mode_n, 4.2), max(1.8 * row_n, 3.0)))

  for mode_idx in range(mode_n):
    ax = ax_s[0, mode_idx]
    _plot_cluster_traces(ax, neur_mode_profile, mode_idx, alpha=0.28, lw=0.55)
    ax.set_ylim(-1.05 * profile_abs, 1.05 * profile_abs)
    mode_var_val = mode_var_raw[mode_idx] if mode_idx < mode_var_raw.size else np.nan
    ax.set_title(f'var: {mode_var_val:.2g}' if np.isfinite(mode_var_val) else 'var: nan')
    if mode_idx == 0:
      ax.set_ylabel('activity form')
    else:
      ax.tick_params(labelleft=False)
    ax.tick_params(labelbottom=False)

  for axis_idx in range(axis_n):
    for mode_idx in range(mode_n):
      ax = ax_s[axis_idx + 1, mode_idx]
      _plot_cluster_traces(ax, centroid_proj[axis_idx], mode_idx, alpha=0.22, lw=0.55, zorder=1)
      if axis_idx < comp_n and np.any(np.isfinite(centroid_axis_score[axis_idx])):
        hi_pos = int(np.nanargmax(centroid_axis_score[axis_idx]))
        lo_pos = int(np.nanargmin(centroid_axis_score[axis_idx]))
        hi_lbl = int(non_noise_label_s[hi_pos])
        lo_lbl = int(non_noise_label_s[lo_pos])
        hi_mean = np.nanmean(centroid_proj[axis_idx, y_all == hi_lbl, mode_idx, :], axis=0)
        lo_mean = np.nanmean(centroid_proj[axis_idx, y_all == lo_lbl, mode_idx, :], axis=0)
        contrast_trace = 0.5 * (hi_mean - lo_mean)
        ax.plot(np.arange(stk_n), contrast_trace, color='k', lw=1.1, zorder=4)
      else:
        hi_lbl = -1
        lo_lbl = -1
      ax.set_ylim(-1.05 * proj_abs, 1.05 * proj_abs)
      if mode_idx == 0:
        if axis_idx < len(clstr_centroid_var_ratio) and np.isfinite(clstr_centroid_var_ratio[axis_idx]):
          ax.set_ylabel(
            rf'$\mathrm{{{lbl_name.get(hi_lbl, str(hi_lbl))}\ vs\ {lbl_name.get(lo_lbl, str(lo_lbl))}}}$'
            + chr(10)
            + rf'$\mathrm{{var={clstr_centroid_var_ratio[axis_idx]:.2f}}}$'
          )
        else:
          ax.set_ylabel(rf'$\mathrm{{{lbl_name.get(hi_lbl, str(hi_lbl))}\ vs\ {lbl_name.get(lo_lbl, str(lo_lbl))}}}$')
      else:
        ax.tick_params(labelleft=False)
      if axis_idx == axis_n - 1:
        ax.set_xlabel('stack index')
      else:
        ax.tick_params(labelbottom=False)
      if np.isfinite(clstr_centroid_contribution[axis_idx, mode_idx]):
        ax.text(
          0.97, 0.95, rf'$\times{clstr_centroid_contribution[axis_idx, mode_idx]:.2f}$',
          transform=ax.transAxes, ha='right', va='top', fontsize=8,
          bbox={'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.55},
          zorder=5,
        )

  fig.suptitle(f'{mv_name} cluster centroid axes')
  fig.tight_layout()
  fig.savefig(fig_path)

  _store('clstr_centroid_axes', clstr_centroid_axes)
  _store('clstr_centroid_contribution', clstr_centroid_contribution)
  _store('clstr_centroid_var_ratio', clstr_centroid_var_ratio)





In [ ]:
def space_time_corr():
  global x_t_cor, x_t_cor_p, mds_t_cor, mds_t_cor_p
  global clstr_x_t_cor, clstr_x_t_cor_p, clstr_mds_t_cor, clstr_mds_t_cor_p
  global clstr_time_eta2, clstr_time_eta2_p, clstr_time_eta2_z

  early_n = 2
  hist_pmt_n = 1000

  def _empty_obs(space_xy):
    space_xy = np.asarray(space_xy, dtype=float)
    ref_dim = space_xy.shape[1] if space_xy.ndim == 2 else 1
    return dict(
      cor=np.nan,
      p=np.nan,
      dist_time=np.full((2, 0), np.nan, dtype=float),
      ref_idx=-1,
      ref_xy=np.full(ref_dim, np.nan, dtype=float),
      ref_time=np.nan,
    )

  def best_for_latency(space_xy, latency, neur_mask):
    space_xy = np.asarray(space_xy, dtype=float)
    latency = np.asarray(latency, dtype=float)
    neur_mask = np.asarray(neur_mask, dtype=bool)
    stable_idx = np.where(neur_mask & np.isfinite(latency))[0]
    out = _empty_obs(space_xy)
    if stable_idx.size < 3:
      return out

    early_cnt = min(int(early_n), stable_idx.size)
    early_idx = stable_idx[np.argsort(latency[stable_idx])[:early_cnt]]
    ref_xy = np.nanmean(space_xy[early_idx], axis=0)
    ref_time = float(np.nanmean(latency[early_idx]))

    dist = np.linalg.norm(space_xy - ref_xy, axis=-1)
    dt = latency - ref_time
    one_dist = dist[stable_idx]
    one_dt = dt[stable_idx]
    good = np.isfinite(one_dist) & np.isfinite(one_dt)
    if np.sum(good) < 2:
      return out
    if np.isclose(np.nanstd(one_dist[good]), 0) or np.isclose(np.nanstd(one_dt[good]), 0):
      return out

    dist_time = np.vstack([one_dist[good], one_dt[good]])
    cor, p_val = sp.stats.pearsonr(*dist_time)
    out.update(dict(
      cor=float(cor),
      p=float(p_val),
      dist_time=dist_time,
      ref_idx=int(early_idx[0]),
      ref_xy=np.asarray(ref_xy, dtype=float),
      ref_time=ref_time,
    ))
    return out

  def run_metric(space_xy, neur_mask):
    neur_mask = np.asarray(neur_mask, dtype=bool)
    obs = best_for_latency(space_xy, mean_spk_time, neur_mask)
    if not np.isfinite(obs['cor']):
      return obs, np.nan

    stable_idx = np.where(neur_mask & np.isfinite(mean_spk_time))[0]
    if stable_idx.size < 3:
      return obs, np.nan

    rng = np.random.default_rng(0)
    null_cor = np.full(hist_pmt_n, np.nan)
    for idx in range(hist_pmt_n):
      null_lat = np.copy(mean_spk_time)
      null_lat[stable_idx] = rng.permutation(null_lat[stable_idx])
      null_cor[idx] = best_for_latency(space_xy, null_lat, neur_mask)['cor']

    p_val = float(np.nanmean(null_cor >= obs['cor']))
    return obs, p_val

  def latency_eta2(label_s):
    label_s = np.asarray(label_s)
    valid_mask = (label_s != -1) & ~np.isnan(mean_spk_time)
    valid_lbl_s = np.sort(np.unique(label_s[valid_mask]))
    group_s = []
    for lbl in valid_lbl_s:
      one_time = mean_spk_time[valid_mask & (label_s == lbl)]
      if one_time.size > 0:
        group_s.append(one_time)
    if len(group_s) < 2:
      return np.nan

    all_time = np.concatenate(group_s)
    grand_mean = float(np.mean(all_time))
    ss_tot = float(np.sum((all_time - grand_mean) ** 2))
    if np.isclose(ss_tot, 0):
      return np.nan

    ss_between = float(sum(one.size * (float(np.mean(one)) - grand_mean) ** 2 for one in group_s))
    return float(ss_between / ss_tot)

  obs_x, x_t_cor_p = run_metric(com, brst_neur)
  obs_mds, mds_t_cor_p = run_metric(tot_mds, brst_neur)
  x_t_cor = float(obs_x['cor'])
  mds_t_cor = float(obs_mds['cor'])

  clstr_x_t_cor = {}
  clstr_x_t_cor_p = {}
  clstr_mds_t_cor = {}
  clstr_mds_t_cor_p = {}
  obs_clstr_s = {}
  if np.all(clstr_lbl == -1):
    clstr_x_t_cor[-1] = np.nan
    clstr_x_t_cor_p[-1] = np.nan
    clstr_mds_t_cor[-1] = np.nan
    clstr_mds_t_cor_p[-1] = np.nan
  else:
    for lbl in np.sort(np.unique(clstr_lbl)):
      lbl = int(lbl)
      if lbl == -1:
        continue
      lbl_mask = brst_neur & (clstr_lbl == lbl)

      obs, p_val = run_metric(com, lbl_mask)
      clstr_x_t_cor[lbl] = float(obs['cor'])
      clstr_x_t_cor_p[lbl] = float(p_val)
      obs_clstr_s[lbl] = obs

      obs_mds_lbl, p_val_mds = run_metric(tot_mds, lbl_mask)
      clstr_mds_t_cor[lbl] = float(obs_mds_lbl['cor'])
      clstr_mds_t_cor_p[lbl] = float(p_val_mds)

  clstr_time_eta2 = latency_eta2(clstr_lbl)
  if np.isfinite(clstr_time_eta2):
    finite_time_idx = np.where(~np.isnan(mean_spk_time))[0]
    rng = np.random.default_rng(0)
    null_eta2 = np.full(hist_pmt_n, np.nan, dtype=float)
    for idx in range(hist_pmt_n):
      null_lbl = np.array(clstr_lbl, copy=True)
      null_lbl[finite_time_idx] = rng.permutation(null_lbl[finite_time_idx])
      null_eta2[idx] = latency_eta2(null_lbl)
    null_mean = float(np.nanmean(null_eta2))
    null_std = float(np.nanstd(null_eta2))
    if np.isfinite(null_std) and (not np.isclose(null_std, 0)):
      clstr_time_eta2_z = float((clstr_time_eta2 - null_mean) / null_std)
    else:
      clstr_time_eta2_z = np.nan
    clstr_time_eta2_p = float(np.nanmean(null_eta2 >= clstr_time_eta2))
  else:
    clstr_time_eta2_z = np.nan
    clstr_time_eta2_p = np.nan

  _store("x_t_cor", x_t_cor)
  _store("x_t_cor_p", x_t_cor_p)
  _store("mds_t_cor", mds_t_cor)
  _store("mds_t_cor_p", mds_t_cor_p)
  _store('clstr_x_t_cor', clstr_x_t_cor)
  _store('clstr_x_t_cor_p', clstr_x_t_cor_p)
  _store('clstr_mds_t_cor', clstr_mds_t_cor)
  _store('clstr_mds_t_cor_p', clstr_mds_t_cor_p)
  _store('clstr_time_eta2', clstr_time_eta2)
  _store('clstr_time_eta2_p', clstr_time_eta2_p)
  _store('clstr_time_eta2_z', clstr_time_eta2_z)

  finite_lat = mean_spk_time[np.isfinite(mean_spk_time)]
  lat_abs = float(np.nanmax(np.abs(finite_lat))) if finite_lat.size > 0 else np.nan
  if (not np.isfinite(lat_abs)) or (lat_abs <= 0):
    lat_abs = 1.0
  vmin, vmax = -lat_abs, lat_abs

  fig, ax_s = plt.subplots(2, 2, figsize=(10, 8))

  ax_s[0, 0].set(title='latency over real space (com)')
  plot_res = 8
  plot_spat = spat[:, ::plot_res, ::plot_res]
  plot_com = com / plot_res
  sc = ax_s[0, 0].scatter(
    *plot_com[brst_neur].T,
    c=mean_spk_time[brst_neur], vmin=vmin, vmax=vmax,
    cmap='summer', s=10)
  ax_s[0, 0].imshow(
    np.mean(plot_spat, 0) > 0, cmap='binary', alpha=0.)
  if obs_x['ref_idx'] >= 0:
    ax_s[0, 0].scatter(
      *(obs_x['ref_xy'] / plot_res), c=[obs_x['ref_time']],
      vmin=vmin, vmax=vmax, cmap='summer', s=36, marker='*')
  ax_s[0, 0].scatter(*plot_com[~brst_neur].T, color='gray', s=10)
  fig.colorbar(sc, ax=ax_s[0, 0])

  ax_s[0, 1].set(
    title=f"real space cor={x_t_cor:.2f}, perm p={x_t_cor_p:.3f}",
    xlabel='distance to early centroid',
    ylabel='latency relative to early centroid')
  if obs_x['dist_time'].shape[1] > 0:
    ax_s[0, 1].scatter(*obs_x['dist_time'])

  ax_s[1, 0].set(title='latency over mds space')
  sc = ax_s[1, 0].scatter(
    *tot_mds[brst_neur].T,
    c=mean_spk_time[brst_neur], vmin=vmin, vmax=vmax,
    cmap='summer', s=10)
  if obs_mds['ref_idx'] >= 0:
    ax_s[1, 0].scatter(
      *obs_mds['ref_xy'], c=[obs_mds['ref_time']],
      vmin=vmin, vmax=vmax, cmap='summer', s=36, marker='*')
  ax_s[1, 0].scatter(*tot_mds[~brst_neur].T, color='gray', marker='x', s=10)
  ax_s[1, 0].set_aspect('equal', adjustable='box')
  fig.colorbar(sc, ax=ax_s[1, 0])

  ax_s[1, 1].set(
    title=f"mds space cor={mds_t_cor:.2f}, perm p={mds_t_cor_p:.3f}",
    xlabel='distance to early centroid',
    ylabel='latency relative to early centroid')
  if obs_mds['dist_time'].shape[1] > 0:
    ax_s[1, 1].scatter(*obs_mds['dist_time'])

  fig.tight_layout()
  fig.savefig(main_path / f'analysis/result/{brst_type}/09-space_time_cor-{mv_name}.pdf')


In [ ]:
def variation_over_bursts():
  fig, ax_s = plt.subplots(
    2, brst_n, squeeze = False,
    figsize = (np.maximum(6*brst_n*brst_len/(brst_neur_n+neur_n), 6), 6))

  ax_s[0,0].set(xlabel = 'frame', ylabel = 'neuron sorted by mean spike time')
  ax_s[1,0].set(ylabel = 'neuron sorted by mean spike time')

  if spk_time_ord.size == neur_n:
    sort_idx = spk_time_ord
  else:
    sort_idx = np.arange(neur_n, dtype=int)

  for brst_idx in range(brst_n):
    ax_s[0,brst_idx].set(title = f'burst {brst_idx}')
    bwr_plt(ax_s[0,brst_idx], flt_all_brst_traj[brst_idx][sort_idx])
    bwr_plt(ax_s[1,brst_idx], all_brst_traj[brst_idx][sort_idx])


  fig.tight_layout()
  fig.savefig(main_path / f'analysis/result/{brst_type}/10-bursts-{mv_name}.pdf')


## Driver

In [ ]:
def run_suite_for_current_movie(mv_idx):
  set_current_movie(mv_idx)

  find_bursts()
  fit_acov()
  stack_trajs()
  numeric_rmt()
  analytic_rmt_n_comparison()
  get_significant_modes()
  get_denoised_traj()

  fit_mode_propagator()

  if brst_type == 'multi':
    extract_timing()
    evaluate_partition_mode_significance()
    analyze_order_consistency()
    mbd_n_clstr()
    cluster_centroid_validation()
    space_time_corr()

  plt.close("all")






# Run per movie analysis 

## Preprocessing

In [ ]:
main_path = Path("/Users/zzhao89/Documents/Git_Code/organoid/")

result_root = main_path / "analysis" / "result"
for _brst_type in ("multi", "single", "none"):
  (result_root / _brst_type).mkdir(parents=True, exist_ok=True)


In [ ]:
def _parse_movie_name(name):
  name_part_s = name.split('-')
  return {
    "mv_name": name,
    "tsu_type": f"{name_part_s[0]}-",
    "tsu": "-".join([name_part_s[idx] for idx in (0, 1, 2)]),
    "cond": "-".join(name_part_s[3:]),
  }


# minimal preprocessing: index movie names/paths only
cnmf_root = main_path / "cnmf"
mcorr_root = main_path / "mcorr"

cnmf_path_map = {
  p.stem: main_path.joinpath(p.relative_to(main_path))
  for p in cnmf_root.iterdir()
  if p.name.startswith(("org-", "cab-"))
}
mcorr_path_map = {
  p.stem: main_path.joinpath(p.relative_to(main_path))
  for p in mcorr_root.iterdir()
  if p.name.startswith(("org-", "cab-"))
}

common_name_s = [name for name in cnmf_path_map if name in mcorr_path_map]
movie_row_s = []
for _tsu_type in ["org-", "cab-"]:
  tsu_name_s = [name for name in common_name_s if name.startswith(_tsu_type)]
  for tsu_local_idx, name in enumerate(tsu_name_s):
    row = _parse_movie_name(name)
    row["tsu_local_idx"] = tsu_local_idx
    row["cnmf_path"] = str(cnmf_path_map[name])
    row["mcorr_path"] = str(mcorr_path_map[name])
    movie_row_s.append(row)

movie_df = pd.DataFrame(movie_row_s)
traj_n = len(movie_df)
name_s = movie_df["mv_name"].tolist()
print(f"Indexed {traj_n} movies (org+cab)")


## Plot all

In [ ]:
tsu_type = "cab-" # "org-"

plot_df = movie_df[movie_df["tsu_type"] == tsu_type].reset_index(drop=True)
if plot_df.empty:
  raise ValueError(f"No movies found for tsu_type={tsu_type}")

cond_order_map = {
  "org-": ['spon', 'musc', 'spon-am', 'bic'],
  "cab-": ['spon', 'stim'],
}
cond_order = cond_order_map[tsu_type]

unq_tsu = list(np.sort(plot_df["tsu"].unique()))
cond_idx_s = [cond_order.index(cond) for cond in plot_df["cond"].tolist()]
tsu_idx_s = [unq_tsu.index(tsu) for tsu in plot_df["tsu"].tolist()]

traj_plot_s = []
for idx in plot_df.index:
  raw_path = Path(plot_df.loc[idx, "cnmf_path"])
  with np.load(raw_path) as npz:
    raw_traj = npz["arr_0"]
    raw_neur_lbl = npz["arr_1"]

  traj = raw_traj[raw_neur_lbl >= 0]
  traj = stdz(traj, 1)
  nonzero = ~np.all(np.isnan(traj), 1)
  traj = traj[nonzero]
  traj_plot_s.append(traj)


for plot_content in ["trajectory rasters"]:
  fig, ax_s = plt.subplots(
    len(unq_tsu), len(cond_order),
    figsize=(4 * len(cond_order), 0.7 * len(unq_tsu)))

  fig.suptitle(
    plot_content + ' (neuron x frame)', fontsize=20)
  for idx, tsu in enumerate(unq_tsu):
    ax_s[idx, 0].set_ylabel(tsu)
  for idx, cond in enumerate(cond_order):
    ax_s[0, idx].set_title(cond)

  for idx in range(len(traj_plot_s)):
    ax = ax_s[tsu_idx_s[idx], cond_idx_s[idx]]

    if plot_content == "trajectory rasters":
      bwr_plt(ax, traj_plot_s[idx])
      ax.set(xticks=[], yticks=[])

  fig.tight_layout()


## Run

In [ ]:
mv_brst_type_dict = {
  'org-': {
    "multi": np.array([0, 1, 2, 4, 6, 9, 10, 12, 15]),
    "single": np.array([14]),
    "none": np.array([3, 5, 7, 8, 11, 13])},
  'cab-': {
    "multi": np.array([0, 2, 4, 6, 8, 9, 10, 11]),
    "single": np.array([]),
    "none": np.array([1, 3, 5, 7])}} # 9


# mv_smry_s = pickle.load(open(main_path / "analysis/result/mv_smry_s.pkl", "rb"))


for mv_idx in np.arange(traj_n):
  run_suite_for_current_movie(mv_idx)

# # Example: only run cab movies
# for mv_idx in movie_df.index[movie_df["tsu_type"] == "cab-"]:
#   run_suite_for_current_movie(mv_idx)

with open(main_path / "analysis/result/mv_smry_s.pkl", "wb") as f:
  pickle.dump(dict(mv_smry_s), f)


In [ ]:
# Cache the image and contour inputs used by manuscript figures.
# Plotting cells reload this file and do not reopen TIFF or CNMF files.
figure_imshow_mv_s = [
  "org-11-1-bic",
  "cab-6-1-spon",
  "cab-6-1-stim",
]
figure_imshow = {}

for mv_name in figure_imshow_mv_s:
  movie = tifffile.memmap(main_path / "mcorr" / f"{mv_name}.tif")
  frame_activity = np.nanmean(movie, axis=(1, 2))
  best_frame = int(np.nanargmax(frame_activity))
  mean_img = np.array(movie[best_frame])
  del movie

  with np.load(main_path / "cnmf" / f"{mv_name}.npz") as npz:
    neur_lbl = np.asarray(npz["arr_1"], dtype=int)
    spat = np.asarray(npz["arr_3"])[neur_lbl >= 0]

  if spat.size and spat.shape[1:] != mean_img.shape:
    height = min(mean_img.shape[0], spat.shape[1])
    width = min(mean_img.shape[1], spat.shape[2])
    mean_img = mean_img[:height, :width]
    spat = spat[:, :height, :width]

  figure_imshow[mv_name] = {
    "mean_img": mean_img,
    "spat": spat,
  }

with open(main_path / "analysis/result/figure_imshow.pkl", "wb") as f:
  pickle.dump(figure_imshow, f, protocol=pickle.HIGHEST_PROTOCOL)


# paper figures

In [ ]:
from pathlib import Path
import pickle

import numpy as np
import scipy as sp
import pandas as pd
import pingouin as pg
import tifffile

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
from matplotlib import patches as plt_ptc
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from mpl_toolkits.axes_grid1 import make_axes_locatable
import seaborn as sns

import os
os.environ["PATH"] = "/Library/TeX/texbin:" + os.environ.get("PATH", "")


mpl.rcParams['text.usetex'] = True

FIG_FONT_PT = 8
FIG_PANEL_LABEL_FONT_PT = 12
FIG_XLABELPAD = 1.5
FIG_YLABELPAD = 2.5
mpl.rcParams.update({
    "font.size": FIG_FONT_PT,
    "axes.labelsize": FIG_FONT_PT,
    "axes.titlesize": FIG_FONT_PT,
    "xtick.labelsize": FIG_FONT_PT,
    "ytick.labelsize": FIG_FONT_PT,
    "legend.fontsize": FIG_FONT_PT,
    "legend.title_fontsize": FIG_FONT_PT,
    "figure.titlesize": FIG_FONT_PT,
    "mathtext.default": "regular",
})



def add_panel_labels(fig, panel_s, fontsize=FIG_PANEL_LABEL_FONT_PT, x_shift=-0.055, y_shift=0.0):
  fig.canvas.draw()
  letter_w = 0.60 * (fontsize / 72.0) / fig.get_figwidth()
  letter_h = (fontsize / 72.0) / fig.get_figheight()
  for panel in panel_s:
    label, ax_or_ax_s = panel[:2]
    dx_letters = panel[2] if len(panel) > 2 else 0.0
    dy_letters = panel[3] if len(panel) > 3 else 0.0
    if isinstance(ax_or_ax_s, (list, tuple, np.ndarray)):
      ax_s = list(ax_or_ax_s)
    else:
      ax_s = [ax_or_ax_s]
    bbox_s = [ax.get_position() for ax in ax_s]
    x0 = min(bbox.x0 for bbox in bbox_s)
    y1 = max(bbox.y1 for bbox in bbox_s)
    fig.text(
      x0 + x_shift + dx_letters * letter_w,
      y1 + y_shift + dy_letters * letter_h,
      rf"$\mathrm{{{label}}}$",
      ha="left",
      va="center",
      fontsize=fontsize,
    )


# Deposited plotting inputs. These cells do not reopen TIFF or CNMF files.
if "main_path" not in globals():
  _plot_cwd = Path.cwd().resolve()
  main_path = _plot_cwd.parent if _plot_cwd.name == "analysis" else _plot_cwd

with open(main_path / "analysis/result/mv_smry_s.pkl", "rb") as f:
  mv_smry_s = pickle.load(f)
with open(main_path / "analysis/result/figure_imshow.pkl", "rb") as f:
  figure_imshow = pickle.load(f)

def _load_figure_imshow(mv_name):
  if mv_name not in figure_imshow:
    raise KeyError(f"{mv_name} missing from figure_imshow.pkl")
  return figure_imshow[mv_name]


In [ ]:
# paper-figure shared movie table (mv_smry_s-first; independent of legacy df cell)

from statsmodels.stats.multitest import multipletests
import itertools

TSU_TYPE_MAP = {"org-": "org", "cab-": "cab"}
COND_MAP = {"spon-am": "spon", "bic": "stim"}
ORDER = ["org", "cab"]
HUE_ORDER = ["spon", "stim"]
ALPHA = 0.05

LW_MAIN = 1.0
LW_SECONDARY = 0.8
LW_REFERENCE = 1.0
LW_CONTOUR = 0.9

ALPHA_MAIN = 0.85
ALPHA_SECONDARY = 0.70
ALPHA_BACKGROUND = 0.35
ALPHA_BOX = 0.30

S_STRIP = 5
S_SCATTER = 20
S_SCATTER_SMALL = 12
S_SCATTER_HIGHLIGHT = 30

LEGEND_MARKER = 5
LEGEND_MARKER_SMALL = 4
FIG_FONT_PT = 8
FIG_PANEL_LABEL_FONT_PT = 12

COND_COLOR = {
  "spon": "tab:olive",
  "stim": "tab:purple",
  "musc": "tab:cyan",
}
PART_COLOR = {
  "unrecruited": "0.55",
  "partial": "tab:pink",
  "recruited": "tab:brown",
}
SOURCE_PALETTE = {
  "no-burst": "0.25",
  "unrecruited": PART_COLOR["unrecruited"],
  "partial": PART_COLOR["partial"],
  "recruited": PART_COLOR["recruited"],
}

def _blend_toward_white(color, frac=0.40):
  rgb = np.asarray(mcolors.to_rgb(color), dtype=float)
  return tuple((1.0 - frac) * rgb + frac * np.ones(3, dtype=float))

COND_BOX_COLOR = {k: _blend_toward_white(v, frac=0.40) for k, v in COND_COLOR.items()}
TSU_MARKER = {"org": "v", "cab": "^"}


def add_panel_labels(fig, panel_s, fontsize=FIG_PANEL_LABEL_FONT_PT, x_shift=-0.055, y_shift=0.0):
  fig.canvas.draw()
  letter_w = 0.60 * (fontsize / 72.0) / fig.get_figwidth()
  letter_h = (fontsize / 72.0) / fig.get_figheight()
  for panel in panel_s:
    label, ax_or_ax_s = panel[:2]
    dx_letters = panel[2] if len(panel) > 2 else 0.0
    dy_letters = panel[3] if len(panel) > 3 else 0.0
    if isinstance(ax_or_ax_s, (list, tuple, np.ndarray)):
      ax_s = list(ax_or_ax_s)
    else:
      ax_s = [ax_or_ax_s]
    bbox_s = [ax.get_position() for ax in ax_s]
    x0 = min(bbox.x0 for bbox in bbox_s)
    y1 = max(bbox.y1 for bbox in bbox_s)
    fig.text(
      x0 + x_shift + dx_letters * letter_w,
      y1 + y_shift + dy_letters * letter_h,
      rf"$\mathrm{{{label}}}$",
      ha="left",
      va="center",
      fontsize=fontsize,
    )

# Paper-figure demo movie choices. Keep these centralized so figure cells do not drift.
# Demo recordings, named for the figure label each one feeds. The old FIG<n>_
# names did not match figure order: FIG3 drove the appendix adjust-mode figure
# while FIG4/FIG5 drove main-text Figure 3, which caused a misdiagnosis.
#   MODE_N_*      -> fig:mode-n   (main Figure 1) and fig:mode-more (appendix)
#   SEQ_PART_*    -> fig:seq-part (main Figure 2)
#   CLUSTER_*     -> fig:cluster  (main Figure 3)
#   ADJUST_MODE_* -> fig:adjust-mode (appendix)
# fig:physical (main Figure 4) picks its demo from the data, see that cell.
MODE_N_DEMO_MV = "org-11-1-bic"
MODE_N_SNAPSHOT_MV_S = ["cab-6-1-spon", "org-11-1-bic"]
# CCN recording behind fig:mode-n panel A right and panel D; the same network as
# the supplement's CCN example, which shows its excited recording.
MODE_N_DEMO_MV_CAB = "cab-6-1-spon"
MODE_MORE_MV_CAB = "cab-6-1-stim"
MODE_N_CARTOON_DELAY_MV = "org-11-1-bic"
SEQ_PART_DEMO_MV = "cab-2-1-spon"
ADJUST_MODE_DEMO_MV = "cab-2-1-spon"
CLUSTER_DEMO_MV = "org-13-4-spon" # "cab-6-1-stim"
CLUSTER_LATENCY_DEMO_MV = "org-13-4-spon"


def _build_multi_plot_df(stat_name_s=None):
  base_stat_name_s = [
    "mode_n", "mode_pr",
    "itrm_sig_mode_n", "stb_sig_mode_n", "oth_sig_mode_n",
    "avg_pop_pr", "avg_dyn_var", "avg_syn_pwr",
    "prop_fund_freq", "prop_freq_floor", "frame_dt",
    "itrm_avg_dyn_var", "stb_avg_dyn_var", "oth_avg_dyn_var",
    "brst_n", "brst_len", "prop_rate", "tail_dev",
    "brst_neur_n", "stb_neur_n", "itrm_neur_n", "oth_neur_n", "ustb_rcrtmnt", "brst_neur_frac",
    "order_cor", "kendall",
    "clstr_scr", "clstr_scr_s", "part_clstr_scr", "no_clstr_frac", "clstr_n", "x_clstr_scr", "t_clstr_scr", "t_self_clstr_scr",
    "x_t_cor", "x_t_cor_p", "mds_t_cor", "mds_t_cor_p",
    "clstr_x_t_cor", "clstr_x_t_cor_p", "clstr_mds_t_cor", "clstr_mds_t_cor_p",
    "clstr_time_eta2", "clstr_time_eta2_p", "clstr_time_eta2_z",
  ]

  if stat_name_s is None:
    stat_name_s = base_stat_name_s
  else:
    stat_name_s = list(dict.fromkeys(base_stat_name_s + list(stat_name_s)))

  rows = []
  for mv_name in sorted(mv_smry_s["mv_name"].keys()):
    if mv_smry_s["brst_type"].get(mv_name) != "multi":
      continue

    row = {
      "mv_name": mv_name,
      "tsu": mv_smry_s["tsu"].get(mv_name),
      "tsu_type": TSU_TYPE_MAP.get(mv_smry_s["tsu_type"].get(mv_name), mv_smry_s["tsu_type"].get(mv_name)),
      "cond": COND_MAP.get(mv_smry_s["cond"].get(mv_name), mv_smry_s["cond"].get(mv_name)),
    }

    for stat_name in stat_name_s:
      row[stat_name] = mv_smry_s.get(stat_name, {}).get(mv_name, np.nan)

    rows.append(row)

  plot_df = pd.DataFrame(rows)
  if plot_df.empty:
    return plot_df

  plot_df = plot_df[
    plot_df["tsu_type"].isin(ORDER)
    & plot_df["cond"].isin(HUE_ORDER)
  ].copy()
  plot_df["tsu_type"] = pd.Categorical(plot_df["tsu_type"], categories=ORDER, ordered=True)
  plot_df["cond"] = pd.Categorical(plot_df["cond"], categories=HUE_ORDER, ordered=True)
  return plot_df.reset_index(drop=True)


def anova_with_holm(data, dv, between):
  anova_tbl = pg.anova(
    data=data,
    dv=dv,
    between=between,
    detailed=True,
    effsize="np2",
  ).copy()

  anova_tbl["p_holm"] = np.nan
  if isinstance(between, list):
    term_s = [str(x) for x in between]
    if len(term_s) == 2:
      term_s.append(f"{term_s[0]} * {term_s[1]}")
  else:
    term_s = [str(between)]

  mask = anova_tbl["Source"].isin(term_s)
  if mask.any():
    anova_tbl.loc[mask, "p_holm"] = multipletests(
      anova_tbl.loc[mask, "p_unc"].to_numpy(dtype=float),
      method="holm",
    )[1]
  return anova_tbl


def _anova_term_fp(anova_tbl, term_name):
  row = anova_tbl.loc[anova_tbl["Source"] == term_name]
  if row.empty:
    return np.nan, np.nan
  return float(row["F"].iloc[0]), float(row["p_holm"].iloc[0])


def _factorial_anova_df(data, dv, factor_order_s):
  factor_s = list(factor_order_s.keys())
  out_df = (
    data[[dv] + factor_s]
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=[dv] + factor_s)
    .copy()
  )
  for factor, order_s in factor_order_s.items():
    out_df = out_df[out_df[factor].astype(str).isin(order_s)].copy()
    out_df[factor] = pd.Categorical(out_df[factor].astype(str), categories=order_s, ordered=True)
  return out_df.reset_index(drop=True)


def _missing_factorial_cells(data, factor_order_s):
  factor_s = list(factor_order_s.keys())
  missing_s = []
  for combo in itertools.product(*[factor_order_s[factor] for factor in factor_s]):
    mask = np.ones(len(data), dtype=bool)
    for factor, level in zip(factor_s, combo):
      mask &= (data[factor].astype(str) == str(level))
    if not np.any(mask):
      missing_s.append(" x ".join(str(level) for level in combo))
  return missing_s


def print_factorial_anova(data, dv, factor_order_s, title):
  anova_df = _factorial_anova_df(data, dv, factor_order_s)
  missing_s = _missing_factorial_cells(anova_df, factor_order_s)
  print(f"\n{title}:")
  if len(missing_s) > 0:
    print("skipped (missing factorial cell(s): " + "; ".join(missing_s) + ")")
    return pd.DataFrame()
  anova_tbl = anova_with_holm(anova_df, dv=dv, between=list(factor_order_s.keys()))
  print(anova_tbl[["Source", "F", "np2", "p_unc"]])
  return anova_tbl


def print_tsu_cond_anova(data, dv, title):
  return print_factorial_anova(
    data,
    dv,
    {"tsu_type": ORDER, "cond": HUE_ORDER},
    title,
  )


def posthoc_2x2_holm(stat_df, dv):
  out_s = []
  for within in ["org", "cab"]:
    one_df = stat_df[stat_df["tsu_type"] == within]
    g1 = one_df.loc[one_df["cond"] == "spon", dv].dropna()
    g2 = one_df.loc[one_df["cond"] == "stim", dv].dropna()
    tt = pg.ttest(g1, g2, paired=False)
    tt.insert(0, "comparison", f"spon vs stim | {within}")
    out_s.append(tt)

  for within in ["spon", "stim"]:
    one_df = stat_df[stat_df["cond"] == within]
    g1 = one_df.loc[one_df["tsu_type"] == "org", dv].dropna()
    g2 = one_df.loc[one_df["tsu_type"] == "cab", dv].dropna()
    tt = pg.ttest(g1, g2, paired=False)
    tt.insert(0, "comparison", f"org vs cab | {within}")
    out_s.append(tt)

  ph_tbl = pd.concat(out_s, ignore_index=True)
  ph_tbl["p_holm"] = multipletests(
    ph_tbl["p_val"].to_numpy(dtype=float),
    method="holm",
  )[1]
  return ph_tbl


def posthoc_3group_holm(source_df):
  ph_tbl = pg.pairwise_tests(
    data=source_df,
    dv="value",
    between="source",
    padjust="none",
    parametric=True,
    correction=True,
  )
  ph_tbl = ph_tbl[["A", "B", "T", "dof", "p_unc"]].copy()
  ph_tbl["p_holm"] = multipletests(
    ph_tbl["p_unc"].to_numpy(dtype=float),
    method="holm",
  )[1]
  return ph_tbl


def _group_mask(df, group_col, group_val):
  if isinstance(group_col, (list, tuple)):
    gv = group_val if isinstance(group_val, (list, tuple)) else (group_val,)
    mask = np.ones(len(df), dtype=bool)
    for col, val in zip(group_col, gv):
      mask &= (df[col] == val)
    return mask
  return (df[group_col] == group_val)


def holm_pairwise_pvals(df, group_col, value_col, pair_list):
  p_raw_s = []
  key_s = []

  for g1, g2 in pair_list:
    m1 = _group_mask(df, group_col, g1)
    m2 = _group_mask(df, group_col, g2)

    v1 = df.loc[m1, value_col].to_numpy(dtype=float)
    v2 = df.loc[m2, value_col].to_numpy(dtype=float)
    v1 = v1[np.isfinite(v1)]
    v2 = v2[np.isfinite(v2)]

    p_val = np.nan
    if (v1.size >= 2) and (v2.size >= 2):
      tt = sp.stats.ttest_ind(v1, v2, equal_var=False, nan_policy="omit")
      p_val = float(tt.pvalue)

    key_s.append((tuple(g1) if isinstance(g1, (list, tuple)) else g1,
                  tuple(g2) if isinstance(g2, (list, tuple)) else g2))
    p_raw_s.append(p_val)

  out = {k: np.nan for k in key_s}
  p_arr = np.asarray(p_raw_s, dtype=float)
  valid = np.isfinite(p_arr)
  if np.any(valid):
    p_adj = multipletests(p_arr[valid], method="holm")[1]
    valid_idx = np.where(valid)[0]
    for idx0, p in zip(valid_idx, p_adj):
      out[key_s[idx0]] = float(p)

  return out



def print_pairwise_annotation_table(df, group_col, value_col, pair_list, p_map, title):
  row_s = []
  for g1, g2 in pair_list:
    m1 = _group_mask(df, group_col, g1)
    m2 = _group_mask(df, group_col, g2)
    v1 = df.loc[m1, value_col].to_numpy(dtype=float)
    v2 = df.loc[m2, value_col].to_numpy(dtype=float)
    v1 = v1[np.isfinite(v1)]
    v2 = v2[np.isfinite(v2)]

    t_stat = np.nan
    p_unc = np.nan
    if (v1.size >= 2) and (v2.size >= 2):
      tt = sp.stats.ttest_ind(v1, v2, equal_var=False, nan_policy="omit")
      t_stat = float(tt.statistic)
      p_unc = float(tt.pvalue)

    k12 = (tuple(g1) if isinstance(g1, (list, tuple)) else g1,
           tuple(g2) if isinstance(g2, (list, tuple)) else g2)
    k21 = (k12[1], k12[0])
    row_s.append({
      "comparison": f"{g1} vs {g2}",
      "n_A": int(v1.size),
      "n_B": int(v2.size),
      "T": t_stat,
      "p_unc": p_unc,
      "p_holm": float(p_map.get(k12, p_map.get(k21, np.nan))),
    })

  print(f"\n{title}:")
  print(pd.DataFrame(row_s)[["comparison", "n_A", "n_B", "T", "p_unc", "p_holm"]])

def p_to_stars(p):
  if not np.isfinite(p):
    return "ns"
  if p <= 1e-4:
    return "****"
  if p <= 1e-3:
    return "***"
  if p <= 1e-2:
    return "**"
  if p <= 5e-2:
    return "*"
  return "ns"


def format_corr_r_stars(r, p):
  if not np.isfinite(r):
    return r"$r=\mathrm{nan}^{\mathrm{n.s.}}$"

  stars = p_to_stars(float(p))
  if stars == "ns":
    stars = "n.s."

  return rf"$r={float(r):.2f}^{{\mathrm{{{stars}}}}}$"


def annotate_onesample_stars(ax, df, group_col, value_col, order, p_by_group,
                             fontsize=8, ns_text=r"$\mathrm{n.s.}$"):
  """Mark each box with its own one-sample test result.

  Placed just above that group's largest value, so the marks sit inside the axes and
  leave the space above free for a pairwise bracket drawn afterwards by
  annotate_box_stars, which extends the ylim itself.
  """
  y0, y1 = ax.get_ylim()
  span = float(y1 - y0) if np.isfinite(y1 - y0) and (y1 > y0) else 1.0
  for grp in order:
    p = float(p_by_group.get(grp, np.nan))
    if not np.isfinite(p):
      continue
    stars = p_to_stars(p)
    if stars == "ns":
      stars = ns_text
    grp_v = df.loc[df[group_col] == grp, value_col].to_numpy(dtype=float)
    grp_v = grp_v[np.isfinite(grp_v)]
    if grp_v.size == 0:
      continue
    ax.text(float(order.index(grp)), float(np.max(grp_v)) + 0.04 * span, stars,
            ha="center", va="bottom", fontsize=fontsize)


def annotate_box_stars(ax, pair_list, p_map, order=None, hue_order=None, hide_ns=True, fontsize=8, ns_text=r"$\mathrm{n.s.}$"):
  if order is None:
    order = []
  if hue_order is None:
    hue_order = []

  def _key(v):
    return tuple(v) if isinstance(v, (list, tuple)) else v

  def _xpos(group_val):
    if hue_order:
      base, hue = group_val
      x0 = float(order.index(base))
      n_h = len(hue_order)
      h_idx = hue_order.index(hue)
      off = (h_idx - (n_h - 1) / 2.0) * (0.8 / n_h)
      return x0 + off
    return float(order.index(group_val))

  y0, y1 = ax.get_ylim()
  span = float(y1 - y0) if np.isfinite(y1 - y0) and (y1 > y0) else 1.0
  base_y = y1 + 0.03 * span
  h = 0.03 * span
  step = 0.08 * span

  used = 0
  for g1, g2 in pair_list:
    k12 = (_key(g1), _key(g2))
    k21 = (_key(g2), _key(g1))
    p = p_map.get(k12, p_map.get(k21, np.nan))
    stars = p_to_stars(p)
    if stars == "ns":
      if hide_ns:
        continue
      stars = ns_text

    x1 = _xpos(g1)
    x2 = _xpos(g2)
    if x2 < x1:
      x1, x2 = x2, x1

    y = base_y + used * step
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], color="k", lw=1.0, clip_on=False)
    ax.text((x1 + x2) / 2.0, y + h + 0.01 * span, stars, ha="center", va="bottom", fontsize=fontsize)
    used += 1

  if used > 0:
    ax.set_ylim(y0, base_y + used * step + h + 0.05 * span)




In [ ]:
# fig:mode-n

# _build_multi_plot_df keeps only multi-burst movies; figure:mode-n summary stats intentionally exclude non-bursting movies.
plot_df = _build_multi_plot_df()

order = ORDER
hue_order = HUE_ORDER

demo_mv_name = MODE_N_DEMO_MV
if demo_mv_name not in mv_smry_s.get("mv_name", {}):
  raise KeyError(f"{demo_mv_name} missing from mv_smry_s; run per-movie analysis first")

demo_cnmf_traj = np.asarray(mv_smry_s["cnmf_traj"][demo_mv_name], dtype=float)
demo_stk_pc = mv_smry_s["stk_pc"][demo_mv_name]
demo_stk_pc0 = np.asarray(demo_stk_pc[0], dtype=float)
demo_stk_num_rmt_spct = np.asarray(mv_smry_s["stk_num_rmt_spct"][demo_mv_name], dtype=float)
demo_mode_n_raw = float(mv_smry_s["mode_n"].get(demo_mv_name, np.nan))
demo_mode_n = int(demo_mode_n_raw) if np.isfinite(demo_mode_n_raw) and (demo_mode_n_raw > 0) else 0
demo_mode_sign = np.asarray(mv_smry_s["mode_sign"].get(demo_mv_name, []), dtype=float)
demo_mode_traj = np.asarray(mv_smry_s["mode_traj"][demo_mv_name], dtype=float)
if demo_mode_traj.ndim == 1:
  demo_mode_traj = demo_mode_traj[None, :]
demo_mode_vec = np.asarray(demo_stk_pc[1], dtype=float)
demo_stk_n = int(mv_smry_s["stk_n"][demo_mv_name])
demo_frm_sep = int(mv_smry_s["frm_sep"][demo_mv_name])
if demo_mode_vec.shape[0] % demo_stk_n != 0:
  raise ValueError(f"Cannot reshape dynamical PC vectors for figure:mode-n in {demo_mv_name}")
demo_mode_neur_n = int(demo_mode_vec.shape[0] // demo_stk_n)
mode_n_sig = max(0, min(int(demo_mode_n), int(demo_mode_sign.size), int(demo_mode_traj.shape[0]), int(demo_mode_vec.shape[1])))

# Second recording shown in panel E, the CCN example whose snapshot is panel A right.
demo_cab_mv_name = MODE_N_DEMO_MV_CAB
if demo_cab_mv_name not in mv_smry_s.get("mv_name", {}):
  raise KeyError(f"{demo_cab_mv_name} missing from mv_smry_s; run per-movie analysis first")
demo_cab_stk_pc0 = np.asarray(mv_smry_s["stk_pc"][demo_cab_mv_name][0], dtype=float)
demo_cab_mode_n_raw = float(mv_smry_s["mode_n"].get(demo_cab_mv_name, np.nan))
demo_cab_mode_n = int(demo_cab_mode_n_raw) if np.isfinite(demo_cab_mode_n_raw) and (demo_cab_mode_n_raw > 0) else 0

# Denoised activity of the same recording as the CNMF traces, for panel B bottom.
demo_dns_traj = np.asarray(mv_smry_s["rlb_traj"][demo_mv_name], dtype=float)

neur_n, frm_n = demo_cnmf_traj.shape
frame_s = np.arange(frm_n, dtype=int)
time_s = frame_s * 0.264


fig = plt.figure(figsize=(6, 6.0), constrained_layout=True)

# Two rows now: the A/B/C/D block, then a single summary row of E/F/G.
# The denoised trace is its own panel D, while C retains the three-step pipeline.
fig_gs = fig.add_gridspec(2, 1, height_ratios=[3.9, 1.5])

# Top area: 2 columns [1.4:1]
top_gs = fig_gs[0, 0].subgridspec(1, 2, width_ratios=[1.4, 1])
# Left column: A over the vertically aligned B and D trace panels
top_left_gs = top_gs[0, 0].subgridspec(2, 1, height_ratios=[1.0, 1.15])
# Panel A: two image axes side by side
panel_a_gs = top_left_gs[0, 0].subgridspec(1, 2)
ax_a_left  = fig.add_subplot(panel_a_gs[0, 0])   # BTO
ax_a_right = fig.add_subplot(panel_a_gs[0, 1])   # CCN
# Panels B and D: CNMF and denoised activity, same recording, shared time axis
# The top axis draws no ticks or xlabel, so constrained_layout reserves almost
# nothing between the two and hspace is close to literal gap; keep it small.
panel_b_gs = top_left_gs[1, 0].subgridspec(2, 1, hspace=0.03)
ax_r1_c23     = fig.add_subplot(panel_b_gs[0, 0])                    # CNMF
ax_r1_c23_dns = fig.add_subplot(panel_b_gs[1, 0], sharex=ax_r1_c23)  # denoised
# Panel C: mini-cartoon container
ax_cartoon_c = fig.add_subplot(top_gs[0, 1])
ax_cartoon_c.set_axis_off()
ax_cartoon_c.set_frame_on(False)
ax_cartoon_c.patch.set_visible(False)
for _spine_c in ax_cartoon_c.spines.values():
  _spine_c.set_visible(False)
ax_c = ax_cartoon_c
ax_c.set_xlim(0.0, 1.0)
ax_c.set_ylim(0.0, 1.0)

# 2x2 layout (column-major order: 1,2 then 3,4), using top+bottom rows.
x_col_s = [0.01, 0.51]
cell_w = 0.46

# Keep prior edge margins and prior inter-row gap from skeleton version.
top_margin = 0.03
bottom_margin = 0.03
center_gap = 0.045
cell_h = (1.0 - top_margin - bottom_margin - center_gap) / 2.0
y_row_top = 1.0 - top_margin - cell_h
y_row_bottom = bottom_margin
y_row_s = [y_row_top, y_row_bottom]

# Explicit movie selection for cell 1 content.
movie_name_s = MODE_N_SNAPSHOT_MV_S
print("cartoon_movies:", movie_name_s[0], movie_name_s[1])

payload_s = [_load_figure_imshow(mv_name) for mv_name in movie_name_s]

# Explicit BL (cell 2) delay-embed source.
bl_mv_name = MODE_N_CARTOON_DELAY_MV
bl_stk_traj = mv_smry_s.get("stk_traj", {}).get(bl_mv_name)
if bl_stk_traj is None:
  raise RuntimeError(f"Missing stk_traj for BL movie: {bl_mv_name}")
bl_stk_traj = np.asarray(bl_stk_traj, dtype=float)
if bl_stk_traj.ndim != 2:
  raise RuntimeError(f"stk_traj is not 2D for BL movie: {bl_mv_name}")
bl_delay_embed_crop = bl_stk_traj[:, :50]

bl_stk_n_raw = mv_smry_s.get("stk_n", {}).get(bl_mv_name)
bl_stk_n = int(bl_stk_n_raw) if bl_stk_n_raw is not None else 1
if bl_stk_n <= 0:
  bl_stk_n = 1
bl_neur_n = bl_delay_embed_crop.shape[0] // bl_stk_n
if bl_neur_n <= 0:
  bl_neur_n = bl_delay_embed_crop.shape[0]
bl_delay_left = bl_delay_embed_crop[:bl_neur_n, :]

# ---- Panel A: fluorescence images with contour overlays ----
cmap_a = plt.get_cmap("tab20")
for ax_img_panel, ridx_a, title_a in [
  (ax_a_left,  1, "BTO"),
  (ax_a_right, 0, "CCN"),
]:
  payload_img = _load_figure_imshow(MODE_N_SNAPSHOT_MV_S[ridx_a])
  ax_img_panel.imshow(payload_img["mean_img"], cmap="gray", aspect="equal")
  spat_img = payload_img["spat"]
  if spat_img.size > 0:
    color_s_a = cmap_a(np.linspace(0, 1, max(spat_img.shape[0], 1)))
    for sidx_a, one_spat_a in enumerate(spat_img):
      vmax_a = float(np.nanmax(one_spat_a))
      if not np.isfinite(vmax_a) or vmax_a <= 0:
        continue
      try:
        ax_img_panel.contour(
          one_spat_a,
          levels=[0.4 * vmax_a],
          colors=[color_s_a[sidx_a % len(color_s_a)]],
          linewidths=0.5 * (LW_CONTOUR if "LW_CONTOUR" in globals() else 0.8),
          alpha=ALPHA_SECONDARY if "ALPHA_SECONDARY" in globals() else 0.70,
        )
      except ValueError:
        continue
  ax_img_panel.set_title(rf"$\mathrm{{{title_a}}}$", fontsize=FIG_FONT_PT)
  ax_img_panel.set_xticks([])
  ax_img_panel.set_yticks([])
  for _sp_a in ax_img_panel.spines.values():
    _sp_a.set_visible(False)

# ---- Panel C: three-step analysis pipeline inside one shared frame ----
frame_y0_c   = 0.04
frame_y1_c   = 0.985
gap_c        = 0.012
top_margin_c = 0.02
bot_margin_c = 0.02
cell_w_c     = 0.98
x_start_c    = 0.01
cell_h_c     = (frame_y1_c - frame_y0_c - top_margin_c - bot_margin_c - 2.0 * gap_c) / 3.0
y_blk4       = frame_y0_c + bot_margin_c
y_blk3       = y_blk4 + cell_h_c + gap_c
y_blk2       = y_blk3 + cell_h_c + gap_c

# One outer frame groups the three steps; individual step frames would imply
# that they are separate panels rather than one analysis pipeline.
ax_c.add_patch(Rectangle(
  (0.0, frame_y0_c), 1.0, frame_y1_c - frame_y0_c,
  fill=False, edgecolor="0.2",
  linewidth=LW_MAIN if "LW_MAIN" in globals() else 1.0,
  transform=ax_c.transAxes, clip_on=False, zorder=10,
))

# With text.usetex=True, leading and trailing math spacing is cropped from the
# text extent used by bbox. Draw explicit title rectangles so horizontal padding
# is controlled geometrically rather than through invisible TeX characters.
def _add_pipeline_title(x, y, cell_w, cell_h, title, box_width_frac, title_y_frac=0.95):
  title_x = x + 0.5 * cell_w
  title_y = y + title_y_frac * cell_h
  box_w = box_width_frac * cell_w
  box_h = 0.13 * cell_h
  ax_c.add_patch(Rectangle(
    (title_x - 0.5 * box_w, title_y - 0.105 * cell_h), box_w, box_h,
    fill=True, facecolor="white", edgecolor="0.2", linewidth=LW_MAIN,
    transform=ax_c.transAxes, zorder=11,
  ))
  ax_c.text(
    title_x, title_y, title, ha="center", va="top",
    fontsize=FIG_FONT_PT if "FIG_FONT_PT" in globals() else 8,
    transform=ax_c.transAxes, zorder=12,
  )

# The numbered titles establish the sequence without the former filled triangles,
# allowing the three steps and their shared frame to be vertically tighter.
x, y = x_start_c, y_blk2
cell_w, cell_h = cell_w_c, cell_h_c
# Custom cell 2 content: top title + delay-embed transform mini-diagram.
_add_pipeline_title(
  x, y, cell_w, cell_h,
  r"$\mathrm{1.\ standardize\ and\ delay\mbox{-}embed}$",
  box_width_frac=0.78,
)

# Content region below title.
cx = x + 0.02 * cell_w
# Keep the title fixed while shifting the complete diagram body downward.
cy = y - 0.005 * cell_h
cw = 0.96 * cell_w
ch = 0.78 * cell_h

# Five columns with ratio 3 : 0.4 : 1 : 3 : 0.4
ratio_s = np.array([3.0, 0.4, 1.0, 3.0, 0.4], dtype=float)
ratio_s = ratio_s / np.sum(ratio_s)
w_l_img, w_l_dot, w_mid, w_r_img, w_r_dot = cw * ratio_s

x_l_img = cx
x_l_dot = x_l_img + w_l_img
x_mid = x_l_dot + w_l_dot
x_r_img = x_mid + w_mid
x_r_dot = x_r_img + w_r_img

vmax = float(np.nanmax(np.abs(bl_delay_embed_crop))) if bl_delay_embed_crop.size else 1.0
if (not np.isfinite(vmax)) or (vmax <= 0):
  vmax = 1.0

# No reserved label row: the body fills the cell and each label is placed directly
# under the thing it names, so the left caption sits close to its own matrix.
label_gap = 0.035 * ch
body_h = ch - 0.14 * ch
body_y = cy + 0.14 * ch

# Right panel uses full body height; left panel uses 1/stk_n of that height and is vertically centered.
h_right = body_h
h_left = h_right / max(int(bl_stk_n), 1)
y_right = body_y
y_left = body_y + 0.5 * (body_h - h_left)

# Keep both stack-trajectory images un-interpolated.
ax_l_img = ax_c.inset_axes([x_l_img, y_left, w_l_img, h_left], transform=ax_c.transAxes)
ax_l_img.imshow(bl_delay_left, cmap="bwr", vmin=-vmax, vmax=vmax, aspect="auto", interpolation="none", resample=False)
ax_l_img.set_xticks([])
ax_l_img.set_yticks([])
ax_l_img.spines["left"].set_visible(True)
ax_l_img.spines["top"].set_visible(True)
ax_l_img.spines["bottom"].set_visible(True)
ax_l_img.spines["right"].set_visible(False)

ax_r_img = ax_c.inset_axes([x_r_img, y_right, w_r_img, h_right], transform=ax_c.transAxes)
ax_r_img.imshow(bl_delay_embed_crop, cmap="bwr", vmin=-vmax, vmax=vmax, aspect="auto", interpolation="none", resample=False)
ax_r_img.set_xticks([])
ax_r_img.set_yticks([])
ax_r_img.spines["left"].set_visible(True)
ax_r_img.spines["top"].set_visible(True)
ax_r_img.spines["bottom"].set_visible(True)
ax_r_img.spines["right"].set_visible(False)

# Dot columns (explicit 3 horizontal dots each).
x_dot_s = [0.25, 0.50, 0.75]
ax_dot_l = ax_c.inset_axes([x_l_dot, body_y, w_l_dot, body_h], transform=ax_c.transAxes)
ax_dot_r = ax_c.inset_axes([x_r_dot, body_y, w_r_dot, body_h], transform=ax_c.transAxes)
for dot_ax in [ax_dot_l, ax_dot_r]:
  dot_ax.set_xticks([])
  dot_ax.set_yticks([])
  for spine in dot_ax.spines.values():
    spine.set_visible(False)
# Left dot column: single 3-dot group. Right dot column: stk_n stacked groups.
for x_dot in x_dot_s:
  ax_dot_l.plot([x_dot], [0.5], marker='o', linestyle='None', markersize=0.4, color='0.2', transform=ax_dot_l.transAxes)
n_dot_grp = max(int(bl_stk_n), 1)
y_grp_s = [(i + 0.5) / n_dot_grp for i in range(n_dot_grp)]
for y_grp in y_grp_s:
  for x_dot in x_dot_s:
    ax_dot_r.plot([x_dot], [y_grp], marker='o', linestyle='None', markersize=0.4, color='0.2', transform=ax_dot_r.transAxes)


# "standardize CNMF" directly beneath the left matrix, not in a shared caption row.
ax_c.text(
  x_l_img + 0.5 * (w_l_img + w_l_dot), y_left - label_gap,
  r"$\mathrm{standardize}$" + "\n" + r"$\mathrm{CNMF}$", ha="center", va="top",
  fontsize=FIG_FONT_PT, transform=ax_c.transAxes)

# Two vertical lines across the embedded matrix: each vertical slice is one state,
# so marking two of them says what a column of X is.
state_frac_s = [0.28, 0.80]
# labels nudged outward from their own lines so the two do not crowd each other
state_lbl_frac_s = [0.16, 0.86]
state_lbl_s = [r"$X_{(h,i)}(t)$", r"$X_{(h,i)}(t')$"]
state_line_extension = 0.05
for state_frac, state_lbl_frac, state_lbl in zip(state_frac_s, state_lbl_frac_s, state_lbl_s):
  # Extend equally beyond the top and bottom of the embedded matrix.
  ax_r_img.axvline(
    state_frac * bl_delay_embed_crop.shape[1],
    ymin=-state_line_extension, ymax=1.0 + state_line_extension,
    color="0.15", lw=LW_MAIN, zorder=5, clip_on=False)
  ax_c.text(
    x_r_img + state_lbl_frac * w_r_img, y_right - label_gap, state_lbl,
    ha="center", va="top", fontsize=FIG_FONT_PT, transform=ax_c.transAxes)

# stk_n fan arrows: single center start -> equally spaced endpoints on right panel edge.
n_arrow = max(int(bl_stk_n), 1)
x_start = x_l_dot + w_l_dot
y_start = y_left + 0.5 * h_left
x_end = x_r_img
for i in range(n_arrow):
  y_end = y_right + h_right - (i + 0.5) * (h_right / n_arrow)
  ax_c.annotate(
    "",
    xy=(x_end, y_end),
    xytext=(x_start, y_start),
    xycoords=ax_c.transAxes,
    textcoords=ax_c.transAxes,
    arrowprops=dict(
      arrowstyle="->",
      lw=LW_MAIN if "LW_MAIN" in globals() else 1.0,
      color="0.2",
      shrinkA=0,
      shrinkB=0,
    ),
  )



x, y = x_start_c, y_blk3
cell_w, cell_h = cell_w_c, cell_h_c
# Custom cell 3 content: top title + SVD equations (no folded title).
_add_pipeline_title(
  x, y, cell_w, cell_h,
  r"$\mathrm{2.\ dynamical\ PCA\ and\ PC\ significance}$",
  box_width_frac=0.92,
)

cx = x + 0.06 * cell_w
cy = y + 0.12 * cell_h
cw = 0.88 * cell_w
ch = 0.72 * cell_h

eq_ax = ax_c.inset_axes([cx, cy, cw, ch], transform=ax_c.transAxes)
eq_ax.set_xticks([])
eq_ax.set_yticks([])
for spine in eq_ax.spines.values():
  spine.set_visible(False)

eq_font = 0.7 * 2 * (FIG_FONT_PT if "FIG_FONT_PT" in globals() else 8)
# Symbol x positions are approximate, set to match the rendered equation; they only
# have to be close enough for the arrows to point at the right factor.
eq_y = 0.78
eq_ax.text(
  0.5, eq_y,
  r"$X_{(h,i),t} = \sum_k U_{(h,i),k}\, \Lambda_k\, V_{k,t}$",
  ha="center", va="center", fontsize=eq_font, transform=eq_ax.transAxes,
)

desc_font = FIG_FONT_PT if "FIG_FONT_PT" in globals() else 8
# The labels are spread evenly across the width and each arrow runs at an angle up to
# the symbol it names, so the caption row stays regular while the tips stay on
# X, U, Lambda and V.
# tip_x values are read off the rendered equation rather than guessed: X sits far
# left, and U/Lambda/V are pushed right by the sum and the long (h,i),k subscripts.
tip_y = eq_y - 0.11
txt_y = tip_y - 0.8 * 0.21
sym_s = [
  (0.11, r"$\mathrm{embedding}$"),
  (0.54, r"$\mathrm{PC}$"),
  (0.75, r"$\mathrm{standard}$" + "\n" + r"$\mathrm{deviation}$"),
  (0.855, r"$\mathrm{time}$" + "\n" + r"$\mathrm{course}$"),
]
lbl_x_s = np.linspace(0.5 / len(sym_s), 1.0 - 0.5 / len(sym_s), len(sym_s))
for lbl_x, (tip_x, sym_lbl) in zip(lbl_x_s, sym_s):
  eq_ax.annotate(
    "", xy=(tip_x, tip_y), xytext=(float(lbl_x), txt_y),
    xycoords=eq_ax.transAxes, textcoords=eq_ax.transAxes,
    arrowprops=dict(arrowstyle="->", lw=0.7 * LW_MAIN, color="0.35",
                    shrinkA=2.0, shrinkB=0),
  )
  eq_ax.text(float(lbl_x), txt_y, sym_lbl, ha="center", va="top",
             fontsize=desc_font, transform=eq_ax.transAxes)

eq_ax.text(
  0.5, 0.16,
  r"$h\ \mathrm{delay},\ i\ \mathrm{neuron},\ t\ \mathrm{time},\ k\ \mathrm{PC\ index}$",
  ha="center", va="center", fontsize=desc_font, transform=eq_ax.transAxes,
)
eq_ax.text(
  0.5, -0.04,
  r"$\mathrm{PC\ significance\ identified\ using\ null\ model}$",
  ha="center", va="center", fontsize=desc_font, transform=eq_ax.transAxes,
)


x, y = x_start_c, y_blk4
cell_w, cell_h = cell_w_c, cell_h_c
# Custom cell 4 content: the one-step regression that completes the decomposition.
_add_pipeline_title(
  x, y, cell_w, cell_h,
  r"$\mathrm{3.\ denoising\ and\ propagator\ fit}$",
  box_width_frac=0.76, title_y_frac=0.93,
)
# Two columns under the title. The left one carries no equation, because keeping a
# A subset of dynamical PCs needs no algebra to state; the propagator step does.
ax_c.text(
  x + 0.21 * cell_w, y + 0.66 * cell_h,
  r"$\mathrm{denoising:}$",
  ha="center", va="center", fontsize=FIG_FONT_PT, transform=ax_c.transAxes,
)
ax_c.text(
  x + 0.21 * cell_w, y + 0.40 * cell_h,
  r"$\mathrm{keep\ significant}$" + "\n" + r"$\mathrm{PCs\ only}$",
  ha="center", va="center", fontsize=FIG_FONT_PT, transform=ax_c.transAxes,
)
ax_c.text(
  x + 0.71 * cell_w, y + 0.66 * cell_h,
  r"$\mathrm{propagator\ fit:}$",
  ha="center", va="center", fontsize=FIG_FONT_PT, transform=ax_c.transAxes,
)
ax_c.text(
  x + 0.71 * cell_w, y + 0.46 * cell_h,
  r"$\tilde{V}_{\cdot,\,t+1} = A\,\tilde{V}_{\cdot,\,t}$",
  ha="center", va="center", fontsize=FIG_FONT_PT, transform=ax_c.transAxes,
)
ax_c.text(
  x + 0.71 * cell_w, y + 0.18 * cell_h,
  r"$\mathrm{eigenvalues\ of\ }A$" + "\n"
  + r"$\mathrm{give\ rate\ and\ frequency}$" + "\n"
  + r"$\mathrm{of\ PC\ interaction}$",
  ha="center", va="center", fontsize=FIG_FONT_PT, transform=ax_c.transAxes,
)
ax_c.plot(
  [x + 0.42 * cell_w] * 2, [y + 0.08 * cell_h, y + 0.78 * cell_h],
  color="0.75", lw=0.7 * LW_MAIN, transform=ax_c.transAxes,
)


# ---- Bottom row: E the spectrum, F and G the two cross-cytoarchitecture summaries ----
# Plain three-way split, so the widths are equal without tuning ratios. wspace is
# additional gap on top of what constrained_layout already reserves for the axis
# labels, so it stays small or the three panels lose a fifth of the row to padding.
bot_gs = fig_gs[1, 0].subgridspec(1, 3, wspace=0.06)
ax_r2_c1 = fig.add_subplot(bot_gs[0, 0])   # E
ax_r4_c2 = fig.add_subplot(bot_gs[0, 1])   # F
ax_r4_c3 = fig.add_subplot(bot_gs[0, 2])   # H

color_map = plt.get_cmap("tab20")
if neur_n <= 1:
  neur_color_s = np.array([color_map(0.0)])
else:
  neur_color_s = color_map(np.linspace(0, 1, neur_n))

# row 1 (full width): CNMF trajectory normalized by max activity (one color per neuron)
cnmf_max = float(np.nanmax(demo_cnmf_traj))
if (not np.isfinite(cnmf_max)) or (cnmf_max <= 0):
  cnmf_max = 1.0
demo_cnmf_traj_norm = demo_cnmf_traj / cnmf_max

for neur_idx in range(neur_n):
  ax_r1_c23.plot(
    time_s,
    demo_cnmf_traj_norm[neur_idx],
    color=neur_color_s[neur_idx],
    lw=LW_SECONDARY,
    alpha=ALPHA_MAIN,
  )
# Bottom half: the same recording after reconstruction from significant dynamical PCs.
# The denoised trajectory lives on the standardized scale, orders of magnitude below
# raw CNMF units, so it carries its own normalization; the two axes share time, not
# amplitude.
demo_dns_traj_crop = demo_dns_traj[:neur_n, :frm_n]
dns_max = float(np.nanmax(np.abs(demo_dns_traj_crop))) if demo_dns_traj_crop.size else np.nan
if (not np.isfinite(dns_max)) or (dns_max <= 0):
  dns_max = 1.0
demo_dns_traj_norm = demo_dns_traj_crop / dns_max
for neur_idx in range(min(neur_n, demo_dns_traj_norm.shape[0])):
  ax_r1_c23_dns.plot(
    time_s,
    demo_dns_traj_norm[neur_idx],
    color=neur_color_s[neur_idx],
    lw=LW_SECONDARY,
    alpha=ALPHA_MAIN,
  )

ax_r1_c23.set_xlim(0.0, float(time_s[-1]))
ax_r1_c23.set_ylim(-0.4, 1.05)
ax_r1_c23.set_ylabel(r"$\mathrm{CNMF\ activity}$" + "\n" + r"$\mathrm{(a.u.)}$")
ax_r1_c23.set_yticks([0, 1])
# shared x: the top axis shows no ticks or label, the bottom carries both once
ax_r1_c23.tick_params(labelbottom=False)
ax_r1_c23.set_xlabel("")

ax_r1_c23_dns.set_ylim(-0.5, 1.2)
ax_r1_c23_dns.set_yticks([0, 1])
ax_r1_c23_dns.set_xlabel(r"$\mathrm{time\ (s)}$")
# Not "a.u.": this is reconstructed from standardized activity, so it has no
# fluorescence units to be arbitrary about, and it is then scaled by its own maximum.
ax_r1_c23_dns.set_ylabel(r"$\mathrm{denoised\ activity}$" + "\n" + r"$\mathrm{(norm.)}$")

# ---- Panel E: component-variance fractions for one BTO and one CCN recording.
# Global style: color is condition, marker direction is cytoarchitecture. Markers are
# hollow throughout; a component inside its own recording's surrogate range is drawn gray, and
# each series is joined by a faint line so the spectrum reads as a curve.
S_DYNVAR = 16
e_series_s = [
  ("org", demo_mv_name, demo_stk_pc0, demo_mode_n, r"$\mathrm{BTO}$"),
  ("cab", demo_cab_mv_name, demo_cab_stk_pc0, demo_cab_mode_n, r"$\mathrm{CCN}$"),
]
e_max_mode_n = 1
e_any = False
for e_tsu, e_mv, e_pc0, e_mode_n, e_label in e_series_s:
  e_finite = e_pc0[np.isfinite(e_pc0)]
  e_desc = e_finite[::-1]
  e_total = float(np.sum(e_desc))
  if (e_desc.size == 0) or (not np.isfinite(e_total)) or (e_total <= 0):
    continue
  e_any = True
  e_frac = e_desc / e_total
  e_idx = np.arange(1, len(e_frac) + 1)
  e_max_mode_n = max(e_max_mode_n, len(e_frac))
  e_sig = e_idx <= max(0, min(int(e_mode_n), len(e_idx)))
  e_pos = e_frac > 0
  # cond in the summary dict is the raw label ("bic", "spon"); COND_MAP is what the
  # rest of the paper keys on, so map before looking up the color.
  e_cond_raw = str(mv_smry_s["cond"].get(e_mv, "spon"))
  e_cond = COND_MAP.get(e_cond_raw, e_cond_raw)
  e_color = COND_COLOR.get(e_cond, "0.4")
  e_marker = TSU_MARKER.get(e_tsu, "o")
  ax_r2_c1.plot(
    e_idx[e_pos], e_frac[e_pos], linestyle="-", lw=0.6 * LW_SECONDARY,
    color=e_color, alpha=ALPHA_BACKGROUND, zorder=1)
  ax_r2_c1.scatter(
    e_idx[~e_sig & e_pos], e_frac[~e_sig & e_pos],
    facecolors="none", edgecolors="0.7", marker=e_marker,
    linewidths=LW_SECONDARY, s=S_DYNVAR, alpha=ALPHA_SECONDARY, zorder=2)
  ax_r2_c1.scatter(
    e_idx[e_sig & e_pos], e_frac[e_sig & e_pos],
    facecolors="none", edgecolors=e_color, marker=e_marker,
    linewidths=LW_MAIN, s=S_DYNVAR, alpha=ALPHA_MAIN, zorder=3, label=e_label)

if e_any:
  # Focus the linear axis on the low-index dynamical PCs where the variance is concentrated.
  ax_r2_c1.set_xlim(0.0, 10.0)
  ax_r2_c1.set_xscale("linear")
  ax_r2_c1.set_xticks([1, 5, 9])
  ax_r2_c1.set_ylim(-0.05, 0.7)
  ax_r2_c1.set_yticks(np.arange(0, 0.61, 0.2))
  # only the zero line: a fraction of 1 is now above the visible range
  ax_r2_c1.hlines([0], *ax_r2_c1.get_xlim(), colors="0.5", linestyles="--", linewidth=LW_REFERENCE)
else:
  ax_r2_c1.text(0.5, 0.5, "no valid spectrum", ha="center", va="center", transform=ax_r2_c1.transAxes)
ax_r2_c1.set_xlabel(r"$\mathrm{sorted\ dynamical\ PC\ index\ } k$")
ax_r2_c1.set_ylabel(r"$\mathrm{dynamical\ PC\ var.\ }\Lambda^2_k$" + "\n"
                    + r"$\mathrm{as\ fraction\ of\ total\ var.}$")
if e_any:
  ax_r2_c1.legend(loc="upper center", ncol=2, fontsize=FIG_FONT_PT, frameon=False,
                  borderpad=0.0, handletextpad=0.4, columnspacing=1.0)

# Summary statistics: panel F effective dimensionality and panel G fundamental frequency.

box_kw = {
  "boxprops": {"facecolor": "none", "edgecolor": "0.45", "linewidth": LW_MAIN},
  "whiskerprops": {"color": "k", "linewidth": LW_MAIN},
  "capprops": {"color": "k", "linewidth": LW_MAIN},
  "medianprops": {"color": "k", "linewidth": LW_MAIN},
}

def _print_onesample_vs_one(one_df, value_col, panel_name, popmean=1.0, alternative="two-sided"):
  # Holm-correct across the cytoarchitectures, matching fig:cluster and fig:physical
  row_s = []
  for grp in ORDER:
    grp_v = one_df.loc[one_df["tsu_type"] == grp, value_col].to_numpy(dtype=float)
    grp_v = grp_v[np.isfinite(grp_v)]
    if grp_v.size < 2:
      row_s.append({"test": grp, "n": int(grp_v.size), "mean": np.nan, "std": np.nan,
                    "stat": np.nan, "p_unc": np.nan, "p_holm": np.nan})
      continue
    t_res = sp.stats.ttest_1samp(grp_v, popmean=popmean, nan_policy="omit", alternative=alternative)
    row_s.append({
      "test": grp,
      "n": int(grp_v.size),
      "mean": float(np.mean(grp_v)),
      "std": float(np.std(grp_v, ddof=1)),
      "stat": float(t_res.statistic),
      "p_unc": float(t_res.pvalue),
      "p_holm": np.nan,
    })
  one_sample_tbl = pd.DataFrame(row_s)
  valid_p = np.isfinite(one_sample_tbl["p_unc"].to_numpy(dtype=float))
  if np.any(valid_p):
    one_sample_tbl.loc[valid_p, "p_holm"] = multipletests(
      one_sample_tbl.loc[valid_p, "p_unc"].to_numpy(dtype=float),
      method="holm",
    )[1]
  print(f"\nOne-sample t-test ({alternative}) vs {popmean:.3g} for {panel_name}:")
  print(one_sample_tbl[["test", "n", "mean", "std", "stat", "p_unc", "p_holm"]])
  return dict(zip(one_sample_tbl["test"], one_sample_tbl["p_holm"]))

# r4_c2: mode_n * mode_pr by org/cab; plot_df excludes non-bursting movies.
mode_np_df = (
  plot_df[["tsu_type", "cond", "mode_n", "mode_pr"]]
  .replace([np.inf, -np.inf], np.nan)
  .dropna()
  .copy()
)
mode_np_df["mode_n_mode_pr"] = mode_np_df["mode_n"] * mode_np_df["mode_pr"]
mode_np_df = mode_np_df.replace([np.inf, -np.inf], np.nan).dropna(subset=["mode_n_mode_pr"]).copy()

sns.boxplot(
  data=mode_np_df,
  x="tsu_type", y="mode_n_mode_pr",
  order=ORDER,
  ax=ax_r4_c2,
  **box_kw,
)
sns.stripplot(
  data=mode_np_df[(mode_np_df["tsu_type"] == "org") & (mode_np_df["cond"] == "spon")],
  x="tsu_type", y="mode_n_mode_pr",
  order=ORDER,
  jitter=False, alpha=ALPHA_MAIN, color=COND_COLOR["spon"], marker=TSU_MARKER["org"], size=S_STRIP, linewidth=0,
  ax=ax_r4_c2,
)
sns.stripplot(
  data=mode_np_df[(mode_np_df["tsu_type"] == "org") & (mode_np_df["cond"] == "stim")],
  x="tsu_type", y="mode_n_mode_pr",
  order=ORDER,
  jitter=False, alpha=ALPHA_MAIN, color=COND_COLOR["stim"], marker=TSU_MARKER["org"], size=S_STRIP, linewidth=0,
  ax=ax_r4_c2,
)
sns.stripplot(
  data=mode_np_df[(mode_np_df["tsu_type"] == "cab") & (mode_np_df["cond"] == "spon")],
  x="tsu_type", y="mode_n_mode_pr",
  order=ORDER,
  jitter=False, alpha=ALPHA_MAIN, color=COND_COLOR["spon"], marker=TSU_MARKER["cab"], size=S_STRIP, linewidth=0,
  ax=ax_r4_c2,
)
sns.stripplot(
  data=mode_np_df[(mode_np_df["tsu_type"] == "cab") & (mode_np_df["cond"] == "stim")],
  x="tsu_type", y="mode_n_mode_pr",
  order=ORDER,
  jitter=False, alpha=ALPHA_MAIN, color=COND_COLOR["stim"], marker=TSU_MARKER["cab"], size=S_STRIP, linewidth=0,
  ax=ax_r4_c2,
)
if ax_r4_c2.get_legend() is not None:
  ax_r4_c2.get_legend().remove()
legend_handles = [
  plt.Line2D([], [], linestyle="", marker="o", markersize=LEGEND_MARKER_SMALL, color=COND_COLOR["spon"], label=r"$\mathrm{spont.}$"),
  plt.Line2D([], [], linestyle="", marker="o", markersize=LEGEND_MARKER_SMALL, color=COND_COLOR["stim"], label=r"$\mathrm{excited}$"),
]
ax_r4_c2.set_xlabel("")
ax_r4_c2.set_ylabel(r"$\mathrm{effective\ dimensionality}$")
ax_r4_c2.set_yticks([0, 1, 3, 6, 9])
ax_r4_c2.set_ylim(-0.2, 7.2)
ax_r4_c2.set_xlim(-0.75, 1.75)
reference_color = "0.5"
ax_r4_c2.hlines(1.0, *ax_r4_c2.get_xlim(), colors=reference_color, linestyles="--", linewidth=LW_REFERENCE)
# A small open circle marks the line-axis intersection. The reference value is
# also a real y tick, colored to match the dashed line.
ax_r4_c2.plot(
  [0.0], [1.0], transform=ax_r4_c2.get_yaxis_transform(),
  marker="o", markersize=3.0, markerfacecolor="white",
  markeredgecolor=reference_color, markeredgewidth=LW_REFERENCE,
  linestyle="none", clip_on=False, zorder=6,
)
for tick_value, tick_label in zip(ax_r4_c2.get_yticks(), ax_r4_c2.get_yticklabels()):
  tick_label.set_color(reference_color if np.isclose(tick_value, 1.0) else "k")
ax_r4_c2.legend(handles=legend_handles, loc="upper center", ncol=len(legend_handles),
                fontsize=FIG_FONT_PT, frameon=False, borderpad=0.0,
                handletextpad=0.4, columnspacing=1.0)
mode_np_pair_s = [("org", "cab")]
p_map = holm_pairwise_pvals(mode_np_df, "tsu_type", "mode_n_mode_pr", mode_np_pair_s)
print_pairwise_annotation_table(
  mode_np_df, "tsu_type", "mode_n_mode_pr", mode_np_pair_s, p_map,
  "figure:mode-n effective dimensionality annotation Welch t-test",
)
print_tsu_cond_anova(
  mode_np_df,
  "mode_n_mode_pr",
  "ANOVA for figure:mode-n panel F effective dimensionality by tissue x condition",
)
# One-sample tests against 1 are the claim panel F actually makes. Draw each as
# a vertical side bracket from the labelled reference line to that group, leaving
# the horizontal bracket above for the between-cytoarchitecture comparison.
def _annotate_onesample_reference_brackets(
    ax, df, group_col, value_col, order, p_by_group, reference=1.0,
    fontsize=FIG_FONT_PT, ns_text=r"$\mathrm{n.s.}$"):
  y0, y1 = ax.get_ylim()
  span = float(y1 - y0) if np.isfinite(y1 - y0) and (y1 > y0) else 1.0
  bracket_offset = 0.55
  cap_w = 0.08
  star_gap = 0.05
  for group_idx, grp in enumerate(order):
    p = float(p_by_group.get(grp, np.nan))
    if not np.isfinite(p):
      continue
    grp_v = df.loc[df[group_col] == grp, value_col].to_numpy(dtype=float)
    grp_v = grp_v[np.isfinite(grp_v)]
    if grp_v.size == 0:
      continue
    stars = p_to_stars(p)
    if stars == "ns":
      stars = ns_text
    side = -1.0 if group_idx == 0 else 1.0
    x_bracket = float(group_idx) + side * bracket_offset
    x_cap = x_bracket - side * cap_w
    # The one-sample t-test is a test of the group mean against the reference.
    y_group = float(np.mean(grp_v))
    y_lo, y_hi = sorted([float(reference), y_group])
    star_rotation = 90 if side < 0 else -90
    ax.plot(
      [x_cap, x_bracket, x_bracket, x_cap], [y_lo, y_lo, y_hi, y_hi],
      color="k", lw=LW_MAIN, clip_on=False, zorder=5,
    )
    ax.text(
      x_bracket + side * star_gap, 0.5 * (y_lo + y_hi), stars,
      ha="center", va="center", rotation=star_rotation, rotation_mode="anchor",
      fontsize=fontsize, clip_on=False, zorder=6,
    )

mode_np_one_p = _print_onesample_vs_one(
  mode_np_df, "mode_n_mode_pr", "effective\ndimensionality", popmean=1.0, alternative="greater")
_annotate_onesample_reference_brackets(
  ax_r4_c2, mode_np_df, "tsu_type", "mode_n_mode_pr", ORDER, mode_np_one_p, reference=1.0)
annotate_box_stars(ax_r4_c2, mode_np_pair_s, p_map, order=ORDER)
# annotate_box_stars raises the top to fit its bracket, so the intended limit is set
# after it rather than before, or it would be silently overwritten.
ax_r4_c2.set_ylim(-0.2, 11.0)

# r4_c3: fundamental frequency by org/cab (same style as r4_c2); plot_df excludes
# non-bursting movies. Recordings that resolve no oscillation above the two-cycle
# floor carry NaN and drop out via dropna(); both are two-burst CCN recordings.
# Derived here rather than read straight from mv_smry_s so the panel also runs
# against a pickle written before fit_mode_propagator() stored prop_fund_freq.
if "prop_fund_freq" not in plot_df.columns or plot_df["prop_fund_freq"].isna().all():
  def _fund_freq(mv):
    f = np.asarray(mv_smry_s["prop_freq"][mv], dtype=float)
    f = f[f >= mv_smry_s["prop_freq_floor"][mv]]
    return float(np.min(f)) if f.size else np.nan
  plot_df["prop_fund_freq"] = [_fund_freq(mv) for mv in plot_df["mv_name"]]

stat_name = "prop_fund_freq"
stat_df = (
  plot_df[["tsu_type", "cond", stat_name]]
  .replace([np.inf, -np.inf], np.nan)
  .dropna()
  .copy()
)

sns.boxplot(
  data=stat_df,
  x="tsu_type", y=stat_name,
  order=ORDER,
  ax=ax_r4_c3,
  **box_kw,
)
sns.stripplot(
  data=stat_df[(stat_df["tsu_type"] == "org") & (stat_df["cond"] == "spon")],
  x="tsu_type", y=stat_name,
  order=ORDER,
  jitter=False, alpha=ALPHA_MAIN, color=COND_COLOR["spon"], marker=TSU_MARKER["org"], size=S_STRIP, linewidth=0,
  ax=ax_r4_c3,
)
sns.stripplot(
  data=stat_df[(stat_df["tsu_type"] == "org") & (stat_df["cond"] == "stim")],
  x="tsu_type", y=stat_name,
  order=ORDER,
  jitter=False, alpha=ALPHA_MAIN, color=COND_COLOR["stim"], marker=TSU_MARKER["org"], size=S_STRIP, linewidth=0,
  ax=ax_r4_c3,
)
sns.stripplot(
  data=stat_df[(stat_df["tsu_type"] == "cab") & (stat_df["cond"] == "spon")],
  x="tsu_type", y=stat_name,
  order=ORDER,
  jitter=False, alpha=ALPHA_MAIN, color=COND_COLOR["spon"], marker=TSU_MARKER["cab"], size=S_STRIP, linewidth=0,
  ax=ax_r4_c3,
)
sns.stripplot(
  data=stat_df[(stat_df["tsu_type"] == "cab") & (stat_df["cond"] == "stim")],
  x="tsu_type", y=stat_name,
  order=ORDER,
  jitter=False, alpha=ALPHA_MAIN, color=COND_COLOR["stim"], marker=TSU_MARKER["cab"], size=S_STRIP, linewidth=0,
  ax=ax_r4_c3,
)
if ax_r4_c3.get_legend() is not None:
  ax_r4_c3.get_legend().remove()
legend_handles = [
  plt.Line2D([], [], linestyle="", marker="o", markersize=LEGEND_MARKER_SMALL, color=COND_COLOR["spon"], label=r"$\mathrm{spont.}$"),
  plt.Line2D([], [], linestyle="", marker="o", markersize=LEGEND_MARKER_SMALL, color=COND_COLOR["stim"], label=r"$\mathrm{excited}$"),
]
ax_r4_c3.legend(handles=legend_handles, loc="upper center", ncol=len(legend_handles),
                fontsize=FIG_FONT_PT, frameon=False, borderpad=0.0,
                handletextpad=0.4, columnspacing=1.0)
ax_r4_c3.set_xlabel("")
ax_r4_c3.set_ylabel(r"$\mathrm{fundamental\ freq.\ (Hz)}$")
ax_r4_c3.set_yticks(np.arange(0, 0.241, 0.06))
ax_r4_c3.set_ylim(-0.006, 0.24)
# no reference line: a frequency has no natural null value to test against
avg_dyn_pair_s = [("org", "cab")]
print("\nfigure:mode-n fundamental frequency group means:")
print(
  stat_df.groupby("tsu_type", observed=True)[stat_name]
  .agg(n="count", mean="mean").reindex(ORDER).reset_index()
)
p_map = holm_pairwise_pvals(stat_df, "tsu_type", stat_name, avg_dyn_pair_s)
print_pairwise_annotation_table(
  stat_df, "tsu_type", stat_name, avg_dyn_pair_s, p_map,
  "figure:mode-n fundamental frequency annotation Welch t-test",
)
print_tsu_cond_anova(
  stat_df,
  stat_name,
  "ANOVA for figure:mode-n panel G fundamental frequency by tissue x condition",
)
# The absence of a difference is the result here, so it is labelled rather than hidden.
annotate_box_stars(ax_r4_c3, avg_dyn_pair_s, p_map, order=ORDER, hide_ns=False,
                   ns_text=r"$\mathrm{n.s.}$")
ax_r4_c3.set_yticks(np.arange(0, 0.241, 0.06))
ax_r4_c3.set_ylim(-0.006, 0.35)
ax_r4_c3.set_xlim(-0.75, 1.75)

ax_r4_c2.set_ylabel(r"$\mathrm{effective\ dimensionality}$")
ax_r4_c3.set_ylabel(r"$\mathrm{fundamental\ freq.\ (Hz)}$")
ax_r4_c2.set_xticks(np.arange(len(ORDER)))
ax_r4_c2.set_xticklabels([r"$\mathrm{BTO}$", r"$\mathrm{CCN}$"])
ax_r4_c3.set_xticks(np.arange(len(ORDER)))
ax_r4_c3.set_xticklabels([r"$\mathrm{BTO}$", r"$\mathrm{CCN}$"])
for _ax in fig.axes:
  _ax.xaxis.label.set_size(FIG_FONT_PT)
  _ax.yaxis.label.set_size(FIG_FONT_PT)
  _ax.xaxis.labelpad = FIG_XLABELPAD
  _ax.yaxis.labelpad = FIG_YLABELPAD
  _ax.title.set_size(FIG_FONT_PT)
  _ax.tick_params(labelsize=FIG_FONT_PT)
  for _txt in _ax.texts:
    _txt.set_fontsize(FIG_FONT_PT)
  _leg = _ax.get_legend()
  if _leg is not None:
    _leg.get_title().set_fontsize(FIG_FONT_PT)
    for _txt in _leg.get_texts():
      _txt.set_fontsize(FIG_FONT_PT)

# C sits inside its shared pipeline frame; all other panel letters remain outside.
ax_c.text(
  0.018, frame_y1_c - 0.005, r"$\mathrm{C}$", ha="left", va="top",
  fontsize=FIG_PANEL_LABEL_FONT_PT, transform=ax_c.transAxes, zorder=20,
)
add_panel_labels(fig, [
  ("A", ax_a_left, 1.6, 0.5),
  ("B", ax_r1_c23, -2.1, 0.0),
  ("D", ax_r1_c23_dns, -2.1, 0.0),
  ("E", ax_r2_c1, -2.2, 0.2),
  ("F", ax_r4_c2, 1.0, 0.2),
  ("G", ax_r4_c3, 0.0, 0.2),
])

(main_path / "analysis/result").mkdir(parents=True, exist_ok=True)
fig.savefig(main_path / "analysis/result/m_mode_n.pdf", format="pdf", bbox_inches="tight")

plt.show()




In [ ]:
# fig:seq-part

plot_df = _build_multi_plot_df([
  "flt_kendall",
  "stb_neur_n", "itrm_neur_n", "oth_neur_n",
])


order = ORDER
hue_order = HUE_ORDER

demo_mv_name = SEQ_PART_DEMO_MV
if demo_mv_name not in mv_smry_s.get("mv_name", {}):
  raise KeyError(f"{demo_mv_name} missing from mv_smry_s; run per-movie analysis first")

demo_flt_traj = np.asarray(mv_smry_s["flt_traj"][demo_mv_name], dtype=float)
demo_traj = np.asarray(mv_smry_s["traj"][demo_mv_name], dtype=float)
demo_cnmf_traj = np.asarray(mv_smry_s["cnmf_traj"][demo_mv_name], dtype=float)
demo_cnmf_max = float(np.nanmax(demo_cnmf_traj)) if demo_cnmf_traj.size else np.nan
if (not np.isfinite(demo_cnmf_max)) or (demo_cnmf_max <= 0):
  demo_cnmf_max = 1.0
demo_cnmf_traj_norm = demo_cnmf_traj / demo_cnmf_max
demo_traj_nonneg = demo_flt_traj - np.nanmin(demo_flt_traj, axis=1, keepdims=True)
demo_rlb_traj = np.asarray(mv_smry_s["rlb_traj"][demo_mv_name], dtype=float)
demo_all_brst_win = np.asarray(mv_smry_s["all_brst_win"][demo_mv_name], dtype=int)
demo_flt_all_brst_traj = np.asarray(mv_smry_s["flt_all_brst_traj"][demo_mv_name], dtype=float)
demo_all_brst_traj = np.asarray(mv_smry_s["all_brst_traj"][demo_mv_name], dtype=float)
demo_rlb_res_std = np.std(demo_rlb_traj - demo_traj, axis=1)
demo_rlb_res_std = np.where(
  np.isfinite(demo_rlb_res_std) & (demo_rlb_res_std > 0),
  demo_rlb_res_std,
  np.nan,
)
demo_all_brst_traj_snr = np.full_like(demo_all_brst_traj, np.nan, dtype=float)
for _brst_idx, _brst_traj in enumerate(demo_all_brst_traj):
  _baseline = np.linspace(_brst_traj[:, 0], _brst_traj[:, -1], _brst_traj.shape[1]).T
  _dev = _brst_traj - _baseline
  demo_all_brst_traj_snr[_brst_idx] = np.divide(
    _dev,
    demo_rlb_res_std[:, None],
    out=np.full_like(_dev, np.nan, dtype=float),
    where=np.isfinite(demo_rlb_res_std[:, None]),
  )

# Fixed activity-to-noise scale for every neuron and burst in Figure 2A.
# White marks the participation amplitude threshold at one residual SD.
SEQ_HEAT_VMIN = -1.5
SEQ_HEAT_THRESHOLD = 1.0
SEQ_HEAT_VMAX = 7.5
seq_heat_cmap = plt.get_cmap("bwr")
seq_heat_norm = mcolors.TwoSlopeNorm(
  vmin=SEQ_HEAT_VMIN,
  vcenter=SEQ_HEAT_THRESHOLD,
  vmax=SEQ_HEAT_VMAX,
)
demo_cnmf_all_brst_traj = np.array([
  demo_cnmf_traj_norm[:, one_win[0]:one_win[1]] for one_win in demo_all_brst_win
], dtype=float) if demo_all_brst_win.size else np.empty((0, demo_cnmf_traj.shape[0], 0), dtype=float)
demo_cnmf_all_brst_traj_detrended = np.full_like(
  demo_cnmf_all_brst_traj,
  np.nan,
  dtype=float,
)
for _brst_idx, _brst_traj in enumerate(demo_cnmf_all_brst_traj):
  _baseline = np.linspace(_brst_traj[:, 0], _brst_traj[:, -1], _brst_traj.shape[1]).T
  demo_cnmf_all_brst_traj_detrended[_brst_idx] = _brst_traj - _baseline
demo_spk_time_ord = np.asarray(mv_smry_s["spk_time_ord"][demo_mv_name], dtype=int)
demo_flt_all_spk_time = np.asarray(mv_smry_s["flt_all_spk_time"][demo_mv_name], dtype=float)
demo_all_spk_time = np.asarray(mv_smry_s["all_spk_time"][demo_mv_name], dtype=float)
demo_mean_spk_time = np.asarray(mv_smry_s["mean_spk_time"][demo_mv_name], dtype=float)
demo_brst_neur_n = int(mv_smry_s["brst_neur_n"][demo_mv_name])
demo_brst_part = np.asarray(mv_smry_s["brst_part"][demo_mv_name], dtype=bool)
demo_brst_p = np.asarray(mv_smry_s["brst_p"][demo_mv_name], dtype=float)
demo_stb_neur = np.asarray(mv_smry_s["stb_neur"][demo_mv_name], dtype=bool)
demo_itrm_neur = np.asarray(mv_smry_s["itrm_neur"][demo_mv_name], dtype=bool)
demo_oth_neur = np.asarray(mv_smry_s["oth_neur"][demo_mv_name], dtype=bool)
neur_n, frm_n = demo_traj_nonneg.shape
frame_s = np.arange(frm_n, dtype=int)
time_s = frame_s * 0.264



source_order_part = ["unrecruited", "partial", "recruited"]
mode_count_order = ["no-burst", "unrecruited"]
cond_order = ["spon", "stim", "musc"]
for _part_key, _frac_key in [
  ("oth_sig_mode_n", "oth_sig_mode_frac"),
  ("itrm_sig_mode_n", "itrm_sig_mode_frac"),
  ("stb_sig_mode_n", "stb_sig_mode_frac"),
]:
  plot_df[_frac_key] = np.divide(
    plot_df[_part_key].to_numpy(dtype=float),
    plot_df["mode_n"].to_numpy(dtype=float),
    out=np.full(len(plot_df), np.nan, dtype=float),
    where=(plot_df["mode_n"].to_numpy(dtype=float) > 0),
  )


all_mv_name_s = sorted(mv_smry_s["mv_name"].keys())
non_brst_mv_s = [
  mv_name for mv_name in all_mv_name_s
  if mv_smry_s["brst_type"].get(mv_name) == "none"
]


def _build_part_source_df(part_key_map):
  row_s = []
  for source_name, key_name in part_key_map:
    one_df = (
      plot_df[["mv_name", "tsu_type", "cond", key_name]]
      .replace([np.inf, -np.inf], np.nan)
      .dropna(subset=[key_name])
      .copy()
    )
    for _, row in one_df.iterrows():
      row_s.append({
        "mv_name": row["mv_name"],
        "tsu_type": row["tsu_type"],
        "cond": row["cond"],
        "source": source_name,
        "value": float(row[key_name]),
      })

  source_df = pd.DataFrame(row_s)
  if source_df.empty:
    return pd.DataFrame(columns=["mv_name", "tsu_type", "cond", "source", "value"])
  source_df["source"] = pd.Categorical(source_df["source"], categories=source_order_part, ordered=True)
  source_df["tsu_type"] = pd.Categorical(source_df["tsu_type"].astype(str), categories=ORDER, ordered=True)
  source_df["cond"] = pd.Categorical(source_df["cond"].astype(str), categories=HUE_ORDER, ordered=True)
  return source_df.reset_index(drop=True)


def _build_mode_count_source_df():
  non_vals = np.asarray([
    mv_smry_s["mode_n"].get(mv_name, np.nan)
    for mv_name in non_brst_mv_s
  ], dtype=float)
  non_vals = non_vals[np.isfinite(non_vals)]

  oth_vals = plot_df["oth_sig_mode_n"].to_numpy(dtype=float)
  oth_vals = oth_vals[np.isfinite(oth_vals)]
  source_df = pd.DataFrame({
    "source": np.concatenate([
      np.repeat("no-burst", len(non_vals)),
      np.repeat("unrecruited", len(oth_vals)),
    ]),
    "value": np.concatenate([non_vals, oth_vals]),
  })
  source_df["source"] = pd.Categorical(
    source_df["source"],
    categories=mode_count_order,
    ordered=True,
  )
  return source_df


def _build_dyn_var_source_df():
  non_vals = np.asarray([
    mv_smry_s["avg_dyn_var"].get(mv_name, np.nan)
    for mv_name in non_brst_mv_s
  ], dtype=float)
  non_vals = non_vals[np.isfinite(non_vals)]

  oth_vals = plot_df["oth_avg_dyn_var"].to_numpy(dtype=float)
  oth_vals = oth_vals[np.isfinite(oth_vals)]
  source_df = pd.DataFrame({
    "source": np.concatenate([
      np.repeat("no-burst", len(non_vals)),
      np.repeat("unrecruited", len(oth_vals)),
    ]),
    "value": np.concatenate([non_vals, oth_vals]),
  })
  source_df["source"] = pd.Categorical(
    source_df["source"],
    categories=mode_count_order,
    ordered=True,
  )
  return source_df


def _anova_and_posthoc(source_df, stat_name):
  anova_tbl = anova_with_holm(source_df, dv="value", between="source")
  print(f"\nANOVA for {stat_name}:")
  print(anova_tbl[["Source", "F", "np2", "p_unc"]])

  f_src, p_src = _anova_term_fp(anova_tbl, "source")
  if np.isfinite(p_src) and (p_src < ALPHA):
    ph_tbl = posthoc_3group_holm(source_df)
    print(f"\nPOSTHOC for {stat_name}:")
    print(ph_tbl[["A", "B", "T", "dof", "p_unc", "p_holm"]])
  else:
    print(f"\nPOSTHOC for {stat_name}: skipped (ANOVA Holm p >= {ALPHA})")

  return f_src, p_src


def _source_tsu_anova(source_df, stat_name):
  print_factorial_anova(
    source_df,
    "value",
    {"source": source_order_part, "tsu_type": ORDER},
    f"ANOVA for {stat_name} by partition x tissue",
  )
  return np.nan, np.nan


def _onesample_vs_zero_holm(source_df, stat_name, popmean=0.0, group_col="source", group_order=None):
  if group_order is None:
    group_order = list(pd.unique(source_df[group_col]))

  row_s = []
  for grp_name in group_order:
    grp_v = source_df.loc[source_df[group_col] == grp_name, "value"].to_numpy(dtype=float)
    grp_v = grp_v[np.isfinite(grp_v)]

    row = {
      group_col: grp_name,
      "n": int(grp_v.size),
      "mean": float(np.mean(grp_v)) if grp_v.size > 0 else np.nan,
      "T": np.nan,
      "p_unc": np.nan,
      "p_holm": np.nan,
      "note": "",
    }

    if grp_v.size < 2:
      row["note"] = "skipped: n<2"
    else:
      tt = pg.ttest(grp_v, float(popmean), paired=False, alternative="greater")
      p_col = "p_val" if "p_val" in tt.columns else "p-val"
      row["T"] = float(tt["T"].iloc[0]) if "T" in tt.columns else np.nan
      row["p_unc"] = float(tt[p_col].iloc[0]) if p_col in tt.columns else np.nan

    row_s.append(row)

  res_df = pd.DataFrame(row_s)
  p_arr = res_df["p_unc"].to_numpy(dtype=float)
  valid = np.isfinite(p_arr)
  if np.any(valid):
    res_df.loc[valid, "p_holm"] = multipletests(p_arr[valid], method="holm")[1]

  print(f"\nONE-SAMPLE t-test greater than {float(popmean):.1f} for {stat_name}:")
  print(res_df[[group_col, "n", "mean", "T", "p_unc", "p_holm", "note"]])
  return res_df


no_burst_plot_df = _build_multi_plot_df([
  "mode_n", "avg_dyn_var",
  "oth_sig_mode_n", "oth_avg_dyn_var",
])

all_mv_name_s = sorted(mv_smry_s["mv_name"].keys())
non_brst_mv_s = [
  mv_name for mv_name in all_mv_name_s
  if mv_smry_s["brst_type"].get(mv_name) == "none"
]


def _build_no_burst_mode_count_source_df():
  non_vals = np.asarray([
    mv_smry_s["mode_n"].get(mv_name, np.nan)
    for mv_name in non_brst_mv_s
  ], dtype=float)
  non_vals = non_vals[np.isfinite(non_vals)]

  oth_vals = no_burst_plot_df["oth_sig_mode_n"].to_numpy(dtype=float)
  oth_vals = oth_vals[np.isfinite(oth_vals)]
  source_df = pd.DataFrame({
    "source": np.concatenate([
      np.repeat("no-burst", len(non_vals)),
      np.repeat("unrecruited", len(oth_vals)),
    ]),
    "value": np.concatenate([non_vals, oth_vals]),
  })
  source_df["source"] = pd.Categorical(
    source_df["source"],
    categories=mode_count_order,
    ordered=True,
  )
  return source_df


def _build_no_burst_dyn_var_source_df():
  non_vals = np.asarray([
    mv_smry_s["avg_dyn_var"].get(mv_name, np.nan)
    for mv_name in non_brst_mv_s
  ], dtype=float)
  non_vals = non_vals[np.isfinite(non_vals)]

  oth_vals = no_burst_plot_df["oth_avg_dyn_var"].to_numpy(dtype=float)
  oth_vals = oth_vals[np.isfinite(oth_vals)]
  source_df = pd.DataFrame({
    "source": np.concatenate([
      np.repeat("no-burst", len(non_vals)),
      np.repeat("unrecruited", len(oth_vals)),
    ]),
    "value": np.concatenate([non_vals, oth_vals]),
  })
  source_df["source"] = pd.Categorical(
    source_df["source"],
    categories=mode_count_order,
    ordered=True,
  )
  return source_df


def _no_burst_onesample_vs_zero_holm(source_df, stat_name, popmean=0.0, group_col="source", group_order=None):
  if group_order is None:
    group_order = list(pd.unique(source_df[group_col]))

  row_s = []
  for grp_name in group_order:
    grp_v = source_df.loc[source_df[group_col] == grp_name, "value"].to_numpy(dtype=float)
    grp_v = grp_v[np.isfinite(grp_v)]

    row = {
      group_col: grp_name,
      "n": int(grp_v.size),
      "mean": float(np.mean(grp_v)) if grp_v.size > 0 else np.nan,
      "T": np.nan,
      "p_unc": np.nan,
      "p_holm": np.nan,
      "note": "",
    }

    if grp_v.size < 2:
      row["note"] = "skipped: n<2"
    else:
      tt = pg.ttest(grp_v, float(popmean), paired=False, alternative="greater")
      p_col = "p_val" if "p_val" in tt.columns else "p-val"
      row["T"] = float(tt["T"].iloc[0]) if "T" in tt.columns else np.nan
      row["p_unc"] = float(tt[p_col].iloc[0]) if p_col in tt.columns else np.nan

    row_s.append(row)

  res_df = pd.DataFrame(row_s)
  p_arr = res_df["p_unc"].to_numpy(dtype=float)
  valid = np.isfinite(p_arr)
  if np.any(valid):
    res_df.loc[valid, "p_holm"] = multipletests(p_arr[valid], method="holm")[1]

  print(f"\nONE-SAMPLE t-test greater than {float(popmean):.1f} for {stat_name}:")
  print(res_df[[group_col, "n", "mean", "T", "p_unc", "p_holm", "note"]])
  return res_df


def _plot_no_burst_no_burst_nonparticipating_panel(ax, source_df, ylabel, stat_name):
  sns.boxplot(
    data=source_df,
    x="source",
    y="value",
    hue="source",
    order=mode_count_order,
    palette=mode_count_palette,
    dodge=False,
    legend=False,
    boxprops={"alpha": ALPHA_BOX},
    ax=ax,
  )
  sns.stripplot(
    data=source_df,
    x="source",
    y="value",
    order=mode_count_order,
    hue="source",
    palette=mode_count_palette,
    dodge=False,
    jitter=False,
    alpha=ALPHA_MAIN,
    size=S_STRIP,
    linewidth=0,
    ax=ax,
  )
  if ax.get_legend() is not None:
    ax.get_legend().remove()

  pair_s = [("no-burst", "unrecruited")]
  p_map = holm_pairwise_pvals(source_df, "source", "value", pair_s)
  print_pairwise_annotation_table(
    source_df, "source", "value", pair_s, p_map,
    f"figure:seq-part {stat_name} annotation Welch t-test",
  )
  annotate_box_stars(ax, pair_s, p_map, order=mode_count_order)
  _no_burst_onesample_vs_zero_holm(
    source_df,
    stat_name,
    popmean=0.0,
    group_order=mode_count_order,
  )

  ax.set(
    xlabel="",
    ylabel=ylabel,
  )
  ax.set_xticks(np.arange(len(mode_count_order)))
  ax.set_xticklabels([r"$\mathrm{no\mbox{-}burst}$", r"$\mathrm{non\mbox{-}participating}$"])

part_mask_key_s = [
  ("unrecruited", "oth_neur"),
  ("partial", "itrm_neur"),
  ("recruited", "stb_neur"),
]


def _weighted_partition_mode_power_rows(mv_name):
  itrm_adj_spct = np.asarray(mv_smry_s["itrm_adj_spct"].get(mv_name, []), dtype=float)
  mode_n = int(itrm_adj_spct.shape[0])
  if mode_n <= 0:
    return []

  if mv_name not in mv_smry_s.get("stk_pc", {}):
    return []
  mode_var = np.asarray(mv_smry_s["stk_pc"][mv_name][0], dtype=float)[-mode_n:]
  stk_pc_vec = np.asarray(mv_smry_s["stk_pc"][mv_name][1], dtype=float)
  stk_n = int(mv_smry_s["stk_n"].get(mv_name, 0))
  traj = np.asarray(mv_smry_s["traj"].get(mv_name, []), dtype=float)
  if traj.ndim < 2:
    return []
  neur_n = int(traj.shape[0])
  if (stk_n <= 0) or (neur_n <= 0) or (stk_n * neur_n != stk_pc_vec.shape[0]):
    return []

  mode_col_s = np.arange(stk_pc_vec.shape[1] - mode_n, stk_pc_vec.shape[1], dtype=int)
  stk_mode = stk_pc_vec[:, mode_col_s].reshape(stk_n, neur_n, mode_n)
  neur_mode_pwr = np.mean(stk_mode ** 2, axis=0) * float(neur_n * stk_n)

  row_s = []
  for source_name, mask_key in part_mask_key_s:
    mask = np.asarray(mv_smry_s[mask_key].get(mv_name, []), dtype=bool)
    if (mask.size != neur_n) or (not np.any(mask)):
      continue
    part_mode_pwr = np.nanmean(neur_mode_pwr[mask, :], axis=0)
    good = np.isfinite(part_mode_pwr) & np.isfinite(mode_var) & (mode_var > 0)
    if not np.any(good):
      continue
    value = float(np.sum(part_mode_pwr[good] * mode_var[good]) / np.sum(mode_var[good]))
    row_s.append({
      "mv_name": mv_name,
      "tsu_type": TSU_TYPE_MAP.get(mv_smry_s["tsu_type"].get(mv_name), mv_smry_s["tsu_type"].get(mv_name)),
      "cond": COND_MAP.get(mv_smry_s["cond"].get(mv_name), mv_smry_s["cond"].get(mv_name)),
      "source": source_name,
      "value": value,
    })
  return row_s


row_s = []
for mv_name in sorted(mv_smry_s["mv_name"].keys()):
  if mv_smry_s["brst_type"].get(mv_name) != "multi":
    continue
  row_s.extend(_weighted_partition_mode_power_rows(mv_name))

weighted_power_df = pd.DataFrame(row_s)
if weighted_power_df.empty:
  raise ValueError("No valid movies for figure:seq-part weighted partition mode power plot")
weighted_power_df["source"] = pd.Categorical(
  weighted_power_df["source"],
  categories=source_order_part,
  ordered=True,
)
weighted_power_df = weighted_power_df[
  weighted_power_df["tsu_type"].astype(str).isin(ORDER)
  & weighted_power_df["cond"].astype(str).isin(HUE_ORDER)
].copy()
weighted_power_df["tsu_type"] = pd.Categorical(weighted_power_df["tsu_type"].astype(str), categories=ORDER, ordered=True)
weighted_power_df["cond"] = pd.Categorical(weighted_power_df["cond"].astype(str), categories=HUE_ORDER, ordered=True)

mode_count_palette = SOURCE_PALETTE


def _annotate_partition_brackets(ax, pair_list, p_map, order, level_s, fontsize=8, shared_gap=0.04):
  # Figure-local compact packing: adjacent comparisons may share a level,
  # while the spanning comparison sits above them. This keeps this cell
  # runnable without changing or rerunning the shared plotting helper.
  y0, y1 = ax.get_ylim()
  span = float(y1 - y0) if np.isfinite(y1 - y0) and (y1 > y0) else 1.0
  base_y = y1 + 0.03 * span
  h = 0.03 * span
  step = 0.12 * span
  max_level = -1
  for pair_idx, ((g1, g2), level) in enumerate(zip(pair_list, level_s)):
    p = p_map.get((g1, g2), p_map.get((g2, g1), np.nan))
    stars = p_to_stars(p)
    annotation = r"$\mathrm{n.s.}$" if stars == "ns" else stars
    x1 = float(order.index(g1))
    x2 = float(order.index(g2))
    text_x = (x1 + x2) / 2.0
    level = int(level)
    # Separate coincident inner endpoints when adjacent brackets share a level.
    for other_idx, ((other_g1, other_g2), other_level) in enumerate(zip(pair_list, level_s)):
      if (other_idx == pair_idx) or (int(other_level) != level):
        continue
      other_x1 = float(order.index(other_g1))
      other_x2 = float(order.index(other_g2))
      if np.isclose(x2, other_x1):
        x2 -= float(shared_gap)
      if np.isclose(x1, other_x2):
        x1 += float(shared_gap)
    max_level = max(max_level, level)
    y = base_y + level * step
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], color="k", lw=1.0, clip_on=False)
    ax.text(
      text_x, y + h + 0.025 * span, annotation,
      ha="center", va="baseline", fontsize=fontsize,
    )
  if max_level >= 0:
    ax.set_ylim(y0, base_y + (max_level + 1) * step + h + 0.05 * span)


cnmf_heat_cmap = plt.get_cmap("bwr")
cnmf_heat_v = np.asarray(demo_cnmf_all_brst_traj_detrended, dtype=float)
cnmf_heat_v = cnmf_heat_v[np.isfinite(cnmf_heat_v)]
cnmf_heat_pos = cnmf_heat_v[cnmf_heat_v > 0]
cnmf_heat_vmax = float(np.nanpercentile(cnmf_heat_pos, 95)) if cnmf_heat_pos.size else 1.0
if (not np.isfinite(cnmf_heat_vmax)) or (cnmf_heat_vmax <= 0):
  cnmf_heat_vmax = 1.0
cnmf_heat_vmin = -cnmf_heat_vmax / 3.0
cnmf_heat_norm = mcolors.Normalize(vmin=cnmf_heat_vmin, vmax=cnmf_heat_vmax, clip=True)

fig = plt.figure(figsize=(6, 6.33))

# Explicit axes preserve the current composition while locking the two left
# rows to identical horizontal bounds. No layout engine moves them afterward.
ax_r1_heat = fig.add_axes([0.085, 0.862, 0.590, 0.112])
ax_r1_trace = fig.add_axes([0.085, 0.698, 0.590, 0.151])
ax_r1_right = fig.add_axes([0.750, 0.698, 0.200, 0.253])
ax_r2_heat = fig.add_axes([0.085, 0.518, 0.590, 0.112])
ax_r2_trace = fig.add_axes([0.085, 0.354, 0.590, 0.151])
ax_r2_right = fig.add_axes([0.750, 0.354, 0.200, 0.253])
ax_bot_c2 = fig.add_axes([0.070, 0.046, 0.225, 0.239])
ax_bot_c3 = fig.add_axes([0.365, 0.046, 0.250, 0.239])
ax_bot_c4 = fig.add_axes([0.685, 0.046, 0.250, 0.239])

if demo_spk_time_ord.size == neur_n:
  sort_idx = demo_spk_time_ord
else:
  sort_idx = np.arange(neur_n, dtype=int)

def _plot_overlay_row(ax, one_traj, one_spk_time_sec, lat_norm, lat_cmap, win_fill_rgba, ylabel_text):
  # Shapes come from the trace passed in, never from module globals. fig:cond
  # reassigns time_s, neur_n and frm_n for its own recordings and runs before
  # fig:proj-order, so by then those globals belong to a different movie.
  one_traj = np.asarray(one_traj, dtype=float)
  row_neur_n, row_frm_n = one_traj.shape
  row_time_s = np.arange(row_frm_n, dtype=float) * 0.264
  y_lo, y_hi = np.nanmin(one_traj), np.nanmax(one_traj)
  y_span = y_hi - y_lo

  for one_win in demo_all_brst_win:
    lft = int(one_win[0])
    rgt = int(one_win[1])
    lft_t = lft * 0.264
    rgt_t = rgt * 0.264
    ax.axvspan(lft_t, rgt_t, color=win_fill_rgba, zorder=0)
    ax.add_patch(plt_ptc.Rectangle(
      (lft_t, y_lo - 0.06 * y_span), rgt_t - lft_t, 1.12 * y_span,
      fill=False, edgecolor="0.45", linewidth=LW_SECONDARY, alpha=ALPHA_SECONDARY,
      linestyle="--"
    ))

  for neur_idx in range(row_neur_n):
    y = one_traj[neur_idx]
    ax.plot(row_time_s, y, color="0.6", lw=LW_SECONDARY, alpha=ALPHA_BACKGROUND, zorder=1)

    for brst_idx, one_win in enumerate(demo_all_brst_win):
      lft = int(one_win[0])
      rgt = int(one_win[1])
      lat_val = one_spk_time_sec[brst_idx, neur_idx]
      seg_color = lat_cmap(lat_norm(lat_val)) if np.isfinite(lat_val) else "0.6"
      ax.plot(
        row_time_s[lft:rgt], y[lft:rgt],
        color=seg_color, lw=LW_MAIN, alpha=ALPHA_SECONDARY, zorder=2
      )

  ax.set_xlim(0.0, float(row_frm_n) * 0.264)
  ax.set_ylim(y_lo - 0.08 * y_span, y_hi + 0.08 * y_span)
  ax.set_xlabel(r"$\mathrm{time\ (s)}$")
  ax.set_ylabel(ylabel_text)
  ax.tick_params(axis="x", which="both", bottom=True, labelbottom=True)

def _plot_burst_inset_row(ax, one_all_brst_traj, row_frm_n, heat_cmap, heat_norm):
  # Draw heatmaps in the same data coordinates as the trace below. This keeps
  # every heatmap edge locked to its burst-window edge after layout adjustment.
  one_all_brst_traj = np.asarray(one_all_brst_traj, dtype=float)
  row_duration_s = float(row_frm_n) * 0.264
  row_neur_n = int(one_all_brst_traj.shape[1]) if one_all_brst_traj.ndim == 3 else int(len(sort_idx))
  ax.set_xlim(0.0, row_duration_s)
  ax.set_ylim(float(row_neur_n), 0.0)
  ax.tick_params(axis="x", which="both", bottom=False, labelbottom=False)
  ax.set_yticks([])
  ax.set_ylabel(r"$\mathrm{sorted\ neuron}$")
  for spine in ax.spines.values():
    spine.set_visible(False)

  if one_all_brst_traj.ndim == 3 and one_all_brst_traj.shape[0] > 0:
    if demo_all_brst_win.shape[0] == one_all_brst_traj.shape[0]:
      brst_ord = np.argsort(demo_all_brst_win[:, 0])
    else:
      brst_ord = np.arange(one_all_brst_traj.shape[0], dtype=int)

    for brst_idx in brst_ord:
      lft = int(demo_all_brst_win[brst_idx, 0])
      rgt = int(demo_all_brst_win[brst_idx, 1])
      lft_t = float(lft) * 0.264
      rgt_t = float(rgt) * 0.264
      brst_traj = np.asarray(one_all_brst_traj[brst_idx], dtype=float)[sort_idx]
      x_edge_s = np.linspace(lft_t, rgt_t, brst_traj.shape[1] + 1)
      y_edge_s = np.arange(row_neur_n + 1, dtype=float)
      ax.pcolormesh(
        x_edge_s,
        y_edge_s,
        brst_traj,
        cmap=heat_cmap,
        norm=heat_norm,
        shading="flat",
        rasterized=True,
      )
      ax.add_patch(plt_ptc.Rectangle(
        (lft_t, 0.0),
        rgt_t - lft_t,
        float(row_neur_n),
        fill=False,
        edgecolor="k",
        linewidth=LW_MAIN,
        clip_on=False,
      ))

def _plot_latency_panel(ax, lat_mtx, vmin, vmax, show_part=True):
  lat_img = np.ma.masked_invalid(lat_mtx)
  im = ax.imshow(
    lat_img.T,
    aspect="auto",
    interpolation="nearest",
    cmap=lat_cmap,
    vmin=vmin,
    vmax=vmax,
  )
  divider = make_axes_locatable(ax)
  # participation sidebar: one row per neuron, coloured as in panels B and C
  part_name_s = np.where(
    demo_stb_neur, "recruited",
    np.where(demo_itrm_neur, "partial", "unrecruited"),
  )
  part_rgb = np.array([mcolors.to_rgb(PART_COLOR[n]) for n in part_name_s[sort_idx]])
  if not show_part:
    part_rgb = np.ones_like(part_rgb)
  pax = divider.append_axes("right", size="4%", pad=0.0)
  pax.imshow(part_rgb[:, None, :], aspect="auto", interpolation="nearest")
  pax.set_xticks([])
  pax.set_yticks([])
  for _side in ("top", "right", "bottom"):
    pax.spines[_side].set_visible(False)
  cax = divider.append_axes("right", size="6%", pad=0.04)
  cbar = plt.colorbar(im, cax=cax)
  cbar.ax.tick_params(labelsize=8)
  ax.set_xlabel(r"$\mathrm{burst}$")
  ax.set_ylabel(r"$\mathrm{sorted\ neuron}$")
  ax.set_title(r"$\mathrm{activation\ latency\ (s)}$", fontsize=8)

# right-column latency matrices with shared scale (seconds)
flt_spk_time_sec = np.asarray(demo_flt_all_spk_time, dtype=float) * 0.264
rlb_spk_time_sec = np.asarray(demo_all_spk_time, dtype=float) * 0.264
flt_lat_mtx = np.asarray(flt_spk_time_sec[:, sort_idx], dtype=float)
rlb_lat_mtx = np.asarray(rlb_spk_time_sec[:, sort_idx], dtype=float)
all_lat = np.concatenate([
  flt_lat_mtx[np.isfinite(flt_lat_mtx)],
  rlb_lat_mtx[np.isfinite(rlb_lat_mtx)],
])
if all_lat.size > 0:
  lat_abs = float(np.nanmax(np.abs(all_lat)))
  if (not np.isfinite(lat_abs)) or (lat_abs <= 0):
    lat_abs = 1.0
  lat_vmin, lat_vmax = -lat_abs, lat_abs
else:
  lat_vmin, lat_vmax = -1.0, 1.0

lat_cmap = plt.get_cmap("summer").copy()
lat_cmap.set_bad(color="0.7")
lat_norm = mcolors.Normalize(vmin=lat_vmin, vmax=lat_vmax)
zero_lat_rgba = lat_cmap(lat_norm(0.0))
win_fill_rgba = (zero_lat_rgba[0], zero_lat_rgba[1], zero_lat_rgba[2], 0.07)

# top: CNMF sequence view, using the same plotting functions and bounds as the denoised view
_plot_burst_inset_row(
  ax_r1_heat,
  demo_cnmf_all_brst_traj_detrended,
  demo_cnmf_traj_norm.shape[1],
  cnmf_heat_cmap,
  cnmf_heat_norm,
)
_plot_overlay_row(ax_r1_trace, demo_cnmf_traj_norm, flt_spk_time_sec, lat_norm, lat_cmap, win_fill_rgba, r"$\mathrm{CNMF}$" + "\n" + r"$\mathrm{activity\ (a.u.)}$")
ax_r1_trace.set_yticks([0, 1])
_plot_latency_panel(ax_r1_right, flt_lat_mtx, lat_vmin, lat_vmax, show_part=False)

# middle: burst heatmaps above the time-aligned denoised traces
_plot_burst_inset_row(
  ax_r2_heat,
  demo_all_brst_traj_snr,
  demo_rlb_traj.shape[1],
  seq_heat_cmap,
  seq_heat_norm,
)
_plot_overlay_row(ax_r2_trace, demo_rlb_traj, rlb_spk_time_sec, lat_norm, lat_cmap, win_fill_rgba, r"$\mathrm{denoised\ activity}$")

_plot_latency_panel(ax_r2_right, rlb_lat_mtx, lat_vmin, lat_vmax)

# bottom row stat panels
box_kw = {
  "boxprops": {"facecolor": "none", "edgecolor": "0.45", "linewidth": LW_MAIN},
  "whiskerprops": {"color": "k", "linewidth": LW_MAIN},
  "capprops": {"color": "k", "linewidth": LW_MAIN},
  "medianprops": {"color": "k", "linewidth": LW_MAIN},
}

def _plot_kendall_style_panel(ax, stat_name, ylabel_text):
  stat_df = (
    plot_df[["tsu_type", "cond", stat_name]]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .copy()
  )

  sns.boxplot(
    data=stat_df,
    x="tsu_type", y=stat_name,
    order=ORDER,
    ax=ax,
    **box_kw,
  )
  sns.stripplot(
    data=stat_df[(stat_df["tsu_type"] == "org") & (stat_df["cond"] == "spon")],
    x="tsu_type", y=stat_name,
    order=ORDER,
    jitter=False, alpha=ALPHA_MAIN, color=COND_COLOR["spon"], marker=TSU_MARKER["org"], size=S_STRIP, linewidth=0,
    ax=ax,
  )
  sns.stripplot(
    data=stat_df[(stat_df["tsu_type"] == "org") & (stat_df["cond"] == "stim")],
    x="tsu_type", y=stat_name,
    order=ORDER,
    jitter=False, alpha=ALPHA_MAIN, color=COND_COLOR["stim"], marker=TSU_MARKER["org"], size=S_STRIP, linewidth=0,
    ax=ax,
  )
  sns.stripplot(
    data=stat_df[(stat_df["tsu_type"] == "cab") & (stat_df["cond"] == "spon")],
    x="tsu_type", y=stat_name,
    order=ORDER,
    jitter=False, alpha=ALPHA_MAIN, color=COND_COLOR["spon"], marker=TSU_MARKER["cab"], size=S_STRIP, linewidth=0,
    ax=ax,
  )
  sns.stripplot(
    data=stat_df[(stat_df["tsu_type"] == "cab") & (stat_df["cond"] == "stim")],
    x="tsu_type", y=stat_name,
    order=ORDER,
    jitter=False, alpha=ALPHA_MAIN, color=COND_COLOR["stim"], marker=TSU_MARKER["cab"], size=S_STRIP, linewidth=0,
    ax=ax,
  )

  if ax.get_legend() is not None:
    ax.get_legend().remove()

  p_map = holm_pairwise_pvals(stat_df, "tsu_type", stat_name, [("org", "cab")])

  org_v = stat_df.loc[stat_df["tsu_type"] == "org", stat_name].to_numpy(dtype=float)
  cab_v = stat_df.loc[stat_df["tsu_type"] == "cab", stat_name].to_numpy(dtype=float)
  org_v = org_v[np.isfinite(org_v)]
  cab_v = cab_v[np.isfinite(cab_v)]

  t_stat = np.nan
  p_unc = np.nan
  if (org_v.size >= 2) and (cab_v.size >= 2):
    tt = sp.stats.ttest_ind(org_v, cab_v, equal_var=False, nan_policy="omit")
    t_stat = float(tt.statistic)
    p_unc = float(tt.pvalue)

  p_holm = float(p_map.get(("org", "cab"), p_map.get(("cab", "org"), np.nan)))
  test_tbl = pd.DataFrame([
    {
      "comparison": "org vs cab",
      "T": t_stat,
      "p_unc": p_unc,
      "p_holm": p_holm,
    }
  ])
  print(f"\n{stat_name} org vs cab t-test:")
  print(test_tbl[["comparison", "T", "p_unc", "p_holm"]])

  annotate_box_stars(ax, [("org", "cab")], p_map, order=ORDER)

  ax.set_xlabel("")
  ax.set_ylabel(ylabel_text)
  ax.set_xticks(np.arange(len(ORDER)))
  ax.set_xticklabels([r"$\mathrm{BTO}$", r"$\mathrm{CCN}$"])
  return stat_df


ax_s = [ax_bot_c4]
# Variance-weighted participation-type power across dynamical PCs.
sns.boxplot(
  data=weighted_power_df,
  x="source",
  y="value",
  hue="source",
  order=source_order_part,
  palette=SOURCE_PALETTE,
  dodge=False,
  legend=False,
  boxprops={"alpha": ALPHA_BOX},
  ax=ax_bot_c3,
)
sns.stripplot(
  data=weighted_power_df,
  x="source",
  y="value",
  order=source_order_part,
  hue="source",
  palette=SOURCE_PALETTE,
  dodge=False,
  jitter=False,
  alpha=ALPHA_MAIN,
  size=S_STRIP,
  linewidth=0,
  ax=ax_bot_c3,
)
if ax_bot_c3.get_legend() is not None:
  ax_bot_c3.get_legend().remove()

pair_s = [
  (source_order_part[i], source_order_part[j])
  for i in range(len(source_order_part))
  for j in range(i + 1, len(source_order_part))
]
p_map = holm_pairwise_pvals(weighted_power_df, "source", "value", pair_s)
print_pairwise_annotation_table(
  weighted_power_df, "source", "value", pair_s, p_map,
  "figure:seq-part variance-weighted partition mode power annotation Welch t-test",
)
_annotate_partition_brackets(
  ax_bot_c3, pair_s, p_map, source_order_part, level_s=[0, 1, 0],
)

_source_tsu_anova(weighted_power_df, "variance-weighted partition mode power")
_onesample_vs_zero_holm(
  weighted_power_df,
  "variance-weighted partition mode power",
  popmean=0.0,
  group_order=source_order_part,
)

ax_bot_c3.set(
  xlabel="",
  ylabel=r"$\mathrm{power\ relative\ to\ average}$",
)
ax_bot_c3.set_xticks(np.arange(len(source_order_part)))
ax_bot_c3.set_xticklabels([r"$\mathrm{non\mbox{-}}$" + "\n" + r"$\mathrm{part.}$", r"$\mathrm{inter\mbox{-}}$" + "\n" + r"$\mathrm{mittent}$", r"$\mathrm{consistent}$"])

def _spread_nonparticipating_intermittent_vs_consistent(source_df, stat_name):
  av_v = source_df.loc[
    source_df["source"].isin(["unrecruited", "partial"]),
    "value",
  ].to_numpy(dtype=float)
  rel_v = source_df.loc[source_df["source"] == "recruited", "value"].to_numpy(dtype=float)
  av_v = av_v[np.isfinite(av_v)]
  rel_v = rel_v[np.isfinite(rel_v)]

  stat = np.nan
  p_val = np.nan
  if (av_v.size >= 2) and (rel_v.size >= 2):
    stat, p_val = sp.stats.levene(av_v, rel_v, center="median")
    stat = float(stat)
    p_val = float(p_val)

  spread_tbl = pd.DataFrame([
    {
      "comparison": "other vs consistent",
      "n_other": int(av_v.size),
      "n_consistent": int(rel_v.size),
      "std_other": float(np.std(av_v, ddof=1)) if av_v.size >= 2 else np.nan,
      "std_consistent": float(np.std(rel_v, ddof=1)) if rel_v.size >= 2 else np.nan,
      "W": stat,
      "p_unc": p_val,
    }
  ])
  print(f"\nBrown-Forsythe spread test for {stat_name}:")
  print(spread_tbl[["comparison", "std_other", "std_consistent", "W", "p_unc"]])
  return spread_tbl


panel_s = [
  (
    [("unrecruited", "oth_avg_dyn_var"), ("partial", "itrm_avg_dyn_var"), ("recruited", "stb_avg_dyn_var")],
    r"$\mathrm{dynamical\ var.\ ratio}$",
    "partition_avg_dyn_var",
  ),
]

for ax, (part_map, y_label, stat_name) in zip(ax_s, panel_s):
  source_df = _build_part_source_df(part_map)

  sns.boxplot(
    data=source_df,
    x="source",
    y="value",
    hue="source",
    order=source_order_part,
    palette=SOURCE_PALETTE,
    dodge=False,
    legend=False,
    boxprops={"alpha": ALPHA_BOX},
    ax=ax,
  )
  sns.stripplot(
    data=source_df,
    x="source",
    y="value",
    order=source_order_part,
    hue="source",
    palette=SOURCE_PALETTE,
    dodge=False,
    jitter=False,
    alpha=ALPHA_MAIN,
    size=S_STRIP,
    linewidth=0,
    ax=ax,
  )
  if ax.get_legend() is not None:
    ax.get_legend().remove()

  pair_s = [
    (source_order_part[i], source_order_part[j])
    for i in range(len(source_order_part))
    for j in range(i + 1, len(source_order_part))
  ]
  p_map = holm_pairwise_pvals(source_df, "source", "value", pair_s)
  print_pairwise_annotation_table(
    source_df, "source", "value", pair_s, p_map,
    f"figure:seq-part {stat_name} annotation Welch t-test",
  )
  _annotate_partition_brackets(
    ax, pair_s, p_map, source_order_part, level_s=[0, 1, 0],
  )

  f_val, p_val = _source_tsu_anova(source_df, stat_name)
  _onesample_vs_zero_holm(source_df, stat_name, popmean=0.0)
  _spread_nonparticipating_intermittent_vs_consistent(source_df, stat_name)

  ax.set(
    xlabel="",
    ylabel=y_label,
  )
  if stat_name == "partition_sig_mode_frac":
    ax.set_yticks([0, 0.5, 1.0])
  ax.set_xticks(np.arange(len(source_order_part)))
  ax.set_xticklabels([r"$\mathrm{non\mbox{-}}$" + "\n" + r"$\mathrm{part.}$", r"$\mathrm{inter\mbox{-}}$" + "\n" + r"$\mathrm{mittent}$", r"$\mathrm{consistent}$"])
  ax.tick_params(labelsize=8)

# Fraction classified as intermittent in each recording.
intermittent_df = (
  plot_df[["mv_name", "tsu_type", "cond", "stb_neur_n", "itrm_neur_n", "oth_neur_n"]]
  .replace([np.inf, -np.inf], np.nan)
  .dropna(subset=["stb_neur_n", "itrm_neur_n", "oth_neur_n"])
  .drop_duplicates(subset=["mv_name"])
  .copy()
)
intermittent_df["neur_n"] = intermittent_df[["stb_neur_n", "itrm_neur_n", "oth_neur_n"]].sum(axis=1)
intermittent_df = intermittent_df[intermittent_df["neur_n"] > 0].copy()
intermittent_df["intermittent_frac"] = intermittent_df["itrm_neur_n"] / intermittent_df["neur_n"]
intermittent_summary = (
  intermittent_df.groupby("tsu_type", observed=True)["intermittent_frac"]
  .agg(
    n="size",
    n_with_intermittent=lambda value_s: int(np.sum(value_s > 0)),
    mean="mean",
    sd="std",
    median="median",
  )
  .reset_index()
)
print("\nIntermittent-neuron fraction by cytoarchitecture:")
print(intermittent_summary[["tsu_type", "n", "n_with_intermittent", "mean", "sd", "median"]])
intermittent_pair_s = [("org", "cab")]
intermittent_p_map = holm_pairwise_pvals(
  intermittent_df, "tsu_type", "intermittent_frac", intermittent_pair_s)
print_pairwise_annotation_table(
  intermittent_df, "tsu_type", "intermittent_frac",
  intermittent_pair_s, intermittent_p_map,
  "figure:seq-part intermittent-neuron fraction Welch t-test",
)
sns.boxplot(
  data=intermittent_df,
  x="tsu_type", y="intermittent_frac",
  order=ORDER,
  ax=ax_bot_c2,
  **box_kw,
)
for tsu_type in ORDER:
  for cond in HUE_ORDER:
    one_df = intermittent_df[
      (intermittent_df["tsu_type"].astype(str) == tsu_type)
      & (intermittent_df["cond"].astype(str) == cond)
    ]
    if one_df.empty:
      continue
    ax_bot_c2.scatter(
      np.full(len(one_df), ORDER.index(tsu_type), dtype=float),
      one_df["intermittent_frac"].to_numpy(dtype=float),
      marker=TSU_MARKER.get(tsu_type, "o"),
      color=COND_COLOR.get(cond, "0.4"),
      s=S_SCATTER_HIGHLIGHT,
      alpha=ALPHA_MAIN,
      linewidths=0,
      zorder=3,
    )
annotate_box_stars(
  ax_bot_c2, intermittent_pair_s, intermittent_p_map,
  order=ORDER, hide_ns=False,
)
ax_bot_c2.set_xlabel("")
ax_bot_c2.set_ylabel(r"$\mathrm{fraction\ intermittent}$")
ax_bot_c2.set_xticks(np.arange(len(ORDER)))
ax_bot_c2.set_xticklabels([r"$\mathrm{BTO}$", r"$\mathrm{CCN}$"])
ax_bot_c2.set_yticks(np.arange(0.0, 0.41, 0.1))
ax_bot_c2.set_ylim(-0.03, 0.50)

for _ax in fig.axes:
  _ax.xaxis.label.set_size(FIG_FONT_PT)
  _ax.yaxis.label.set_size(FIG_FONT_PT)
  _ax.xaxis.labelpad = FIG_XLABELPAD
  _ax.yaxis.labelpad = FIG_YLABELPAD
  _ax.title.set_size(FIG_FONT_PT)
  _ax.tick_params(labelsize=FIG_FONT_PT)
  for _txt in _ax.texts:
    _txt.set_fontsize(FIG_FONT_PT)
  _leg = _ax.get_legend()
  if _leg is not None:
    _leg.get_title().set_fontsize(FIG_FONT_PT)
    for _txt in _leg.get_texts():
      _txt.set_fontsize(FIG_FONT_PT)

# Keep the heatmap labels close to the inset rows.
ax_r1_heat.yaxis.set_label_coords(0.025, 0.5)
ax_r2_heat.yaxis.set_label_coords(0.025, 0.5)

add_panel_labels(fig, [
  ("A", [ax_r1_heat, ax_r1_trace, ax_r1_right], 0.5, 0.0),
  ("B", [ax_r2_heat, ax_r2_trace, ax_r2_right], 0.5, 0.0),
  ("C", ax_bot_c2, 0.0, -0.05),
  ("D", ax_bot_c3, 0.0, -0.05),
  ("E", ax_bot_c4, -0.5, -0.35),
])

(main_path / "analysis/result").mkdir(parents=True, exist_ok=True)
fig.savefig(main_path / "analysis/result/m_seq_part.pdf", format="pdf", bbox_inches="tight")

plt.show()

# Table II (tab:cond-anova) row: sequence participation fraction.
# Reported in the manuscript but previously not backed by a printed ANOVA (V16).
part_frac_anova_df = (
  _build_multi_plot_df(["brst_neur_frac"])[["mv_name", "tsu_type", "cond", "brst_neur_frac"]]
  .replace([np.inf, -np.inf], np.nan)
  .dropna(subset=["tsu_type", "cond", "brst_neur_frac"])
  .copy()
)
print_tsu_cond_anova(
  part_frac_anova_df,
  "brst_neur_frac",
  "ANOVA for table:cond-anova sequence participation fraction by tissue x condition",
)


In [ ]:
# fig:cluster

cluster_plot_df = _build_multi_plot_df(["clstr_scr", "part_clstr_scr", "brst_neur_frac"])

order = ORDER
hue_order = HUE_ORDER
box_kw_nonhue = {
  "boxprops": {"facecolor": "none", "edgecolor": "0.45", "linewidth": LW_MAIN},
  "whiskerprops": {"color": "k", "linewidth": LW_MAIN},
  "capprops": {"color": "k", "linewidth": LW_MAIN},
  "medianprops": {"color": "k", "linewidth": LW_MAIN},
}


def _require_mv_field(field_name, mv_name):
  if field_name not in mv_smry_s or mv_name not in mv_smry_s[field_name]:
    raise KeyError(
      f"{field_name} missing for {mv_name}; rerun per-movie analysis after the figure:cluster clustering update"
    )
  return mv_smry_s[field_name][mv_name]


def _latency_sort_key(latency):
  latency = np.asarray(latency, dtype=float)
  return np.where(np.isfinite(latency), latency, np.inf)


def _ordered_idx_by_group(label_arr, group_order, latency):
  label_arr = np.asarray(label_arr)
  lat_key = _latency_sort_key(latency)
  ord_s = []
  for group in group_order:
    idx = np.where(label_arr == group)[0]
    if idx.size == 0:
      continue
    idx = idx[np.argsort(lat_key[idx], kind="mergesort")]
    ord_s.append(idx)
  if len(ord_s) == 0:
    return np.arange(label_arr.size, dtype=int)
  return np.concatenate(ord_s).astype(int)


def _hdbscan_group_order(clstr_lbl):
  uniq_lbl_s = np.sort(np.unique(clstr_lbl))
  non_noise = [int(lbl) for lbl in uniq_lbl_s if int(lbl) != -1]
  return non_noise + ([-1] if np.any(clstr_lbl == -1) else [])


def _cluster_color_map(clstr_lbl):
  uniq_lbl_s = np.sort(np.unique(clstr_lbl))
  lbl_color = {-1: "gray"}
  tab10_colors = plt.get_cmap("tab10").colors
  cluster_color_s = [tab10_colors[idx] for idx in [0, 1, 3, 4, 5, 6, 7, 8, 9]]
  non_noise_lbl_s = [int(lbl) for lbl in uniq_lbl_s if int(lbl) != -1]
  for idx, lbl in enumerate(non_noise_lbl_s):
    lbl_color[lbl] = cluster_color_s[idx % len(cluster_color_s)]
  return lbl_color


def _plot_labeled_dist_matrix(
    ax, dist_mat, ord_idx, label_arr, label_order, label_color, label_text,
    axis_label, legend_title, matrix_title):
  ord_idx = np.asarray(ord_idx, dtype=int)
  ord_dist = np.asarray(dist_mat, dtype=float)[np.ix_(ord_idx, ord_idx)]
  ord_lbl = np.asarray(label_arr)[ord_idx]

  im = ax.imshow(ord_dist, cmap="Greys", vmin=0, vmax=2, interpolation="nearest")
  divider = make_axes_locatable(ax)
  bot_ax = divider.append_axes("bottom", size="2.5%", pad=0.005, sharex=ax)
  left_ax = divider.append_axes("left", size="2.5%", pad=0.005, sharey=ax)
  cax = divider.append_axes("right", size="6%", pad=0.08)
  plt.colorbar(im, cax=cax, ticks=[0, 1, 2])

  present_order = [lbl for lbl in label_order if np.any(ord_lbl == lbl)]
  extra_order = [lbl for lbl in np.sort(np.unique(ord_lbl)) if lbl not in present_order]
  present_order = present_order + extra_order
  if len(present_order) > 0:
    lbl_to_code = {lbl: idx for idx, lbl in enumerate(present_order)}
    strip_code = np.asarray([lbl_to_code[lbl] for lbl in ord_lbl], dtype=float)
    strip_cmap = mcolors.ListedColormap([label_color.get(int(lbl), "gray") for lbl in present_order])
    strip_norm = mcolors.BoundaryNorm(np.arange(-0.5, len(present_order) + 0.5), strip_cmap.N)
    bot_ax.imshow(strip_code[None, :], cmap=strip_cmap, norm=strip_norm, aspect="auto", interpolation="nearest")
    left_ax.imshow(strip_code[:, None], cmap=strip_cmap, norm=strip_norm, aspect="auto", interpolation="nearest")
    bot_ax.set_xlim(ax.get_xlim())
    left_ax.set_ylim(ax.get_ylim())

  for strip_ax in [bot_ax, left_ax]:
    strip_ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    strip_ax.set_xticks([])
    strip_ax.set_yticks([])
    for spine in strip_ax.spines.values():
      spine.set_visible(False)

  ax.set_title(matrix_title, fontsize=FIG_FONT_PT)
  ax.set_xlabel(axis_label)
  ax.set_ylabel(axis_label)
  ax.tick_params(labelbottom=False, labelleft=False, bottom=False, left=False)

  return im


def _format_score(score):
  if not np.isfinite(score):
    return r"$\mathrm{DBCV=nan}$"
  return rf"$\mathrm{{DBCV={float(score):.3g}}}$"


def _set_mds_limits(ax, mds_xy):
  ax.autoscale(enable=True, axis="both", tight=False)
  x0, x1 = ax.get_xlim()
  y0, y1 = ax.get_ylim()
  x_span = x1 - x0 if np.isfinite(x1 - x0) and (x1 > x0) else 1.0
  y_span = y1 - y0 if np.isfinite(y1 - y0) and (y1 > y0) else 1.0
  ax.set_xlim(x0 - 0.08 * x_span, x1 + 0.08 * x_span)
  ax.set_ylim(y0 - 0.08 * y_span, y1 + 0.08 * y_span)
  ax.set_aspect("equal", adjustable="box")
  ax.set_xlabel(r"$\mathrm{MDS\ 1}$")
  ax.set_ylabel(r"$\mathrm{MDS\ 2}$")


def _add_score_text(ax, score):
  ax.text(
    0.02, 0.02,
    _format_score(score),
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontsize=FIG_FONT_PT,
    bbox={"facecolor": "none", "edgecolor": "none", "alpha": 0.0},
  )


def _plot_partition_mds(ax, mds_xy, part_lbl, score):
  part_info_s = [
    (2, r"$\mathrm{consistent}$", PART_COLOR["recruited"]),
    (1, r"$\mathrm{intermittent}$", PART_COLOR["partial"]),
    (0, r"$\mathrm{non\mbox{-}participating}$", PART_COLOR["unrecruited"]),
    (-1, r"$\mathrm{noise}$", "gray"),
  ]
  for lbl, _, color in part_info_s:
    m = part_lbl == lbl
    if not np.any(m):
      continue
    ax.scatter(
      mds_xy[m, 0], mds_xy[m, 1],
      color=color,
      alpha=ALPHA_MAIN,
      s=S_SCATTER,
      linewidths=0,
    )
  handles = [
    Line2D([], [], linestyle="", marker="o", markersize=LEGEND_MARKER_SMALL,
           markerfacecolor=color, markeredgecolor="none", alpha=ALPHA_MAIN, label=label)
    for lbl, label, color in part_info_s if np.any(part_lbl == lbl)
  ]
  ax.legend(handles=handles, title=r"$\mathrm{partition}$", loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
  ax.set_title(r"$\mathrm{recruitment\ partition}$", fontsize=FIG_FONT_PT)
  _set_mds_limits(ax, mds_xy)
  _add_score_text(ax, score)


def _plot_hdbscan_mds(ax, mds_xy, clstr_lbl, score):
  lbl_color = _cluster_color_map(clstr_lbl)
  uniq_lbl_s = np.sort(np.unique(clstr_lbl))
  for lbl in uniq_lbl_s:
    lbl_i = int(lbl)
    m = clstr_lbl == lbl_i
    ax.scatter(
      mds_xy[m, 0], mds_xy[m, 1],
      color=lbl_color.get(lbl_i, "gray"),
      alpha=ALPHA_MAIN,
      s=S_SCATTER,
      linewidths=0,
    )
  handles = []
  for lbl in uniq_lbl_s:
    lbl_i = int(lbl)
    lbl_txt = r"$\mathrm{noise}$" if lbl_i == -1 else rf"$\mathrm{{cluster\ {lbl_i}}}$"
    handles.append(Line2D(
      [], [], linestyle="", marker="o", markersize=LEGEND_MARKER_SMALL,
      markerfacecolor=lbl_color.get(lbl_i, "gray"), markeredgecolor="none",
      alpha=ALPHA_MAIN, label=lbl_txt,
    ))
  ax.legend(handles=handles, title=r"$\mathrm{HDBSCAN}$", loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
  ax.set_title(r"$\mathrm{HDBSCAN\ cluster}$", fontsize=FIG_FONT_PT)
  _set_mds_limits(ax, mds_xy)
  _add_score_text(ax, score)


def _plot_combined_mds(ax, mds_xy, part_lbl, clstr_lbl, part_score, clstr_score):
  part_info_s = [
    (2, r"$\mathrm{consistent}$", PART_COLOR["recruited"]),
    (1, r"$\mathrm{intermittent}$", PART_COLOR["partial"]),
    (0, r"$\mathrm{non\mbox{-}participating}$", PART_COLOR["unrecruited"]),
    (-1, r"$\mathrm{noise}$", "gray"),
  ]
  for lbl, _, color in part_info_s:
    m = part_lbl == lbl
    if not np.any(m):
      continue
    ax.scatter(
      mds_xy[m, 0], mds_xy[m, 1],
      color=color,
      alpha=ALPHA_MAIN,
      s=S_SCATTER,
      linewidths=0,
      zorder=2,
    )

  hdb_color = _cluster_color_map(clstr_lbl)
  hdb_lbl_s = np.sort(np.unique(clstr_lbl))
  for lbl in hdb_lbl_s:
    lbl_i = int(lbl)
    m = clstr_lbl == lbl_i
    ax.scatter(
      mds_xy[m, 0], mds_xy[m, 1],
      facecolors="none",
      edgecolors=hdb_color.get(lbl_i, "gray"),
      s=S_SCATTER_HIGHLIGHT,
      linewidths=1.7 * LW_MAIN,
      alpha=0.55,
      zorder=1,
    )

  part_handles = [
    Line2D([], [], linestyle="", marker="o", markersize=LEGEND_MARKER_SMALL,
           markerfacecolor=color, markeredgecolor="none", alpha=ALPHA_MAIN, label=label)
    for lbl, label, color in part_info_s if np.any(part_lbl == lbl)
  ]
  hdb_handles = []
  for lbl in hdb_lbl_s:
    lbl_i = int(lbl)
    lbl_txt = r"$\mathrm{noise}$" if lbl_i == -1 else rf"$\mathrm{{cluster\ {lbl_i}}}$"
    hdb_handles.append(Line2D(
      [], [], linestyle="", marker="o", markersize=LEGEND_MARKER_SMALL,
      markerfacecolor="none", markeredgecolor=hdb_color.get(lbl_i, "gray"),
      markeredgewidth=LW_MAIN, alpha=ALPHA_MAIN, label=lbl_txt,
    ))

  legend_handles = []
  if len(part_handles) > 0:
    legend_handles.append(Line2D([], [], linestyle="", label=r"$\mathrm{sequence}$" + "\n" + r"$\mathrm{participation}$"))
    legend_handles.extend(part_handles)
  if (len(part_handles) > 0) and (len(hdb_handles) > 0):
    legend_handles.append(Line2D([], [], linestyle="", label=""))
  if len(hdb_handles) > 0:
    legend_handles.append(Line2D([], [], linestyle="", label=r"$\mathrm{HDBSCAN}$"))
    legend_handles.extend(hdb_handles)
  if len(legend_handles) > 0:
    ax.legend(
      handles=legend_handles,
      loc="center left",
      bbox_to_anchor=(1.02, 0.5),
      frameon=False,
      borderaxespad=0.0,
      handlelength=1.0,
      labelspacing=0.45,
    )

  ax.set_title(r"$\mathrm{sequence\ participation}$", fontsize=FIG_FONT_PT)
  _set_mds_limits(ax, mds_xy)
  ax.text(
    0.02, 0.02,
    rf"$\mathrm{{part.={part_score:.2g}}}$" + "\n" + rf"$\mathrm{{HDBSCAN={clstr_score:.2g}}}$",
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontsize=FIG_FONT_PT,
    bbox={"facecolor": "none", "edgecolor": "none", "alpha": 0.0},
  )


def _plot_org_cab_cluster_score(ax, stat_df):
  sns.boxplot(
    data=stat_df,
    x="tsu_type",
    y="clstr_scr",
    order=order,
    ax=ax,
    **box_kw_nonhue,
  )
  for tsu_type in order:
    for cond in hue_order:
      one_df = stat_df[(stat_df["tsu_type"] == tsu_type) & (stat_df["cond"] == cond)]
      if one_df.empty:
        continue
      sns.stripplot(
        data=one_df,
        x="tsu_type",
        y="clstr_scr",
        order=order,
        jitter=False,
        alpha=ALPHA_MAIN,
        color=COND_COLOR[str(cond)],
        marker=TSU_MARKER[str(tsu_type)],
        size=S_STRIP,
        linewidth=0,
        ax=ax,
      )
  if ax.get_legend() is not None:
    ax.get_legend().remove()
  legend_handles = [
    Line2D([], [], linestyle="", marker="o", markersize=LEGEND_MARKER_SMALL, color=COND_COLOR["spon"], label=r"$\mathrm{spont.}$"),
    Line2D([], [], linestyle="", marker="o", markersize=LEGEND_MARKER_SMALL, color=COND_COLOR["stim"], label=r"$\mathrm{excited}$"),
  ]
  ax.legend(handles=legend_handles, loc="lower right")

  p_map = holm_pairwise_pvals(stat_df, "tsu_type", "clstr_scr", [("org", "cab")])
  annotate_box_stars(ax, [("org", "cab")], p_map, order=order)

  org_v = stat_df.loc[stat_df["tsu_type"] == "org", "clstr_scr"].to_numpy(dtype=float)
  cab_v = stat_df.loc[stat_df["tsu_type"] == "cab", "clstr_scr"].to_numpy(dtype=float)
  org_v = org_v[np.isfinite(org_v)]
  cab_v = cab_v[np.isfinite(cab_v)]
  if (org_v.size >= 2) and (cab_v.size >= 2):
    tt = pg.ttest(org_v, cab_v, paired=False)
    p_col = "p_val" if "p_val" in tt.columns else "p-val"
    t_val = float(tt["T"].iloc[0]) if "T" in tt.columns else np.nan
    p_unc = float(tt[p_col].iloc[0]) if p_col in tt.columns else np.nan
    print("\nclstr_scr org vs cab t-test:")
    print(
      f"org n={org_v.size}, mean={np.mean(org_v):.3g}; "
      f"cab n={cab_v.size}, mean={np.mean(cab_v):.3g}; "
      f"T={t_val:.3g}, p={p_unc:.3g}"
    )
  else:
    print("\nclstr_scr org vs cab t-test: skipped (need >=2 per group)")

  print_tsu_cond_anova(
    stat_df,
    "clstr_scr",
    "ANOVA for figure:cluster panel E HDBSCAN DBCV by tissue x condition",
  )

  ax.set_xlabel("")
  ax.set_ylabel(r"$\mathrm{HDBSCAN\ DBCV}$")
  ax.set_yticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
  ax.set_xticks(np.arange(len(order)))
  ax.set_xticklabels([r"$\mathrm{BTO}$", r"$\mathrm{CCN}$"])


def _plot_paired_scores(ax, pair_df):
  score_order = ["sequence participation", "HDBSCAN"]
  x_pos = {"recruitment": 0.0, "HDBSCAN": 1.0}
  score_plot_df = pd.concat([
    pair_df[["mv_name", "tsu_type", "cond", "part_clstr_scr"]].rename(columns={"part_clstr_scr": "score"}).assign(score_type="sequence participation"),
    pair_df[["mv_name", "tsu_type", "cond", "clstr_scr"]].rename(columns={"clstr_scr": "score"}).assign(score_type="HDBSCAN"),
  ], ignore_index=True).replace([np.inf, -np.inf], np.nan).dropna(subset=["score"])
  sns.boxplot(
    data=score_plot_df,
    x="score_type",
    y="score",
    order=score_order,
    ax=ax,
    **box_kw_nonhue,
  )
  for _, row in pair_df.iterrows():
    y_pair = [float(row["part_clstr_scr"]), float(row["clstr_scr"])]
    if not np.all(np.isfinite(y_pair)):
      continue
    ax.plot([x_pos["recruitment"], x_pos["HDBSCAN"]], y_pair, color="0.70", lw=0.55, zorder=1)
    color = COND_COLOR.get(str(row["cond"]), "0.4")
    marker = TSU_MARKER.get(str(row["tsu_type"]), "o")
    ax.scatter([x_pos["recruitment"], x_pos["HDBSCAN"]], y_pair, color=color, marker=marker,
               s=S_SCATTER_HIGHLIGHT, alpha=ALPHA_MAIN, linewidths=0, zorder=2)

  ax.set_xlim(-0.50, 1.50)
  ax.set_xlabel("")
  ax.set_xticks([x_pos["recruitment"], x_pos["HDBSCAN"]])
  ax.set_xticklabels([r"$\mathrm{sequence}$" + "\n" + r"$\mathrm{participation}$", r"$\mathrm{HDBSCAN}$"])
  ax.set_ylabel(r"$\mathrm{DBCV}$")

  part_v = pair_df["part_clstr_scr"].to_numpy(dtype=float)
  hdb_v = pair_df["clstr_scr"].to_numpy(dtype=float)
  one_sample_row_s = []
  for test_name, score_v, alt in [
      ("sequence participation < 0", part_v, "less"),
      ("HDBSCAN > 0", hdb_v, "greater")]:
    finite_v = score_v[np.isfinite(score_v)]
    effect_size = np.nan
    p_unc = np.nan
    p_holm = np.nan
    if finite_v.size >= 2:
      sd_v = float(np.std(finite_v, ddof=1))
      if np.isfinite(sd_v) and (sd_v > 0):
        effect_size = float(np.mean(finite_v) / sd_v)
      tt_one = sp.stats.ttest_1samp(finite_v, popmean=0.0, alternative=alt, nan_policy="omit")
      p_unc = float(tt_one.pvalue)
    one_sample_row_s.append({"test": test_name, "effect_size": effect_size, "p_unc": p_unc, "p_holm": p_holm})
  one_sample_df = pd.DataFrame(one_sample_row_s)
  valid_p = np.isfinite(one_sample_df["p_unc"].to_numpy(dtype=float))
  if np.any(valid_p):
    one_sample_df.loc[valid_p, "p_holm"] = multipletests(one_sample_df.loc[valid_p, "p_unc"].to_numpy(dtype=float), method="holm")[1]
  print("\nDBCV one-sample tests vs 0:")
  print(one_sample_df[["test", "effect_size", "p_unc", "p_holm"]])
  valid = np.isfinite(part_v) & np.isfinite(hdb_v)
  if np.sum(valid) >= 2:
    tt = sp.stats.ttest_rel(part_v[valid], hdb_v[valid], alternative="less", nan_policy="omit")
    t_val = float(tt.statistic)
    p_val = float(tt.pvalue)
    print("\npaired DBCV test: sequence/participation < HDBSCAN")
    print(
      f"n={np.sum(valid)}, sequence/participation mean={np.mean(part_v[valid]):.3g}; "
      f"HDBSCAN mean={np.mean(hdb_v[valid]):.3g}; T={t_val:.3g}, p={p_val:.3g}"
    )
    y0, y1 = ax.get_ylim()
    span = float(y1 - y0) if np.isfinite(y1 - y0) and (y1 > y0) else 1.0
    y = y1 + 0.05 * span
    h = 0.035 * span
    ax.plot([0, 0, 1, 1], [y, y + h, y + h, y], color="k", lw=LW_MAIN, clip_on=False)
    ax.text(0.5, y + h + 0.01 * span, p_to_stars(p_val), ha="center", va="bottom", fontsize=FIG_FONT_PT)
    ax.set_ylim(y0 - 0.08 * span, y + h + 0.16 * span)
  else:
    print("\npaired DBCV test: skipped (need >=2 paired movies)")

cluster_mds_df = _build_multi_plot_df([
  "x_t_cor", "x_t_cor_p", "clstr_x_t_cor", "clstr_x_t_cor_p",
  "clstr_lbl", "clstr_scr", "clstr_scr_s", "t_clstr_scr",
  "clstr_time_eta2", "clstr_time_eta2_p", "kendall", "t_self_clstr_scr",
])

order = ORDER
hue_order = HUE_ORDER
box_kw_white = {
  "color": "white",
  "boxprops": {"alpha": ALPHA_BOX, "edgecolor": "0.45", "linewidth": LW_MAIN},
  "whiskerprops": {"color": "k", "linewidth": LW_MAIN},
  "capprops": {"color": "k", "linewidth": LW_MAIN},
  "medianprops": {"color": "k", "linewidth": LW_MAIN},
  "showfliers": False,
}


def _weighted_dict_r(row, value_key):
  r_s = row[value_key]
  clstr_lbl = np.asarray(row["clstr_lbl"])
  if not isinstance(r_s, dict):
    return np.nan

  has_non_noise = np.any(clstr_lbl != -1)
  weighted_sum = 0.0
  weight_sum = 0.0
  for lbl, val in r_s.items():
    lbl = int(lbl)
    if (lbl == -1) and has_non_noise:
      continue
    val = float(val)
    if not np.isfinite(val):
      continue
    weight = int(np.sum(clstr_lbl == lbl))
    if weight <= 0:
      continue
    weighted_sum += val * weight
    weight_sum += weight

  if weight_sum <= 0:
    return np.nan
  return float(weighted_sum / weight_sum)


def _format_r_p(r, p):
  if not (np.isfinite(r) and np.isfinite(p)):
    return r"$r=\mathrm{nan}$"
  return rf"$r={float(r):.2f}^{{\mathrm{{{p_to_stars(float(p))}}}}}$"


def _format_named_corr_line(label, r, p):
  if not (np.isfinite(r) and np.isfinite(p)):
    return rf"$\mathrm{{{label}}}:\ r=\mathrm{{nan}}$"
  stars = p_to_stars(float(p))
  if stars == "ns":
    stars = "n.s."
  return rf"$\mathrm{{{label}}}:\ r={float(r):.2f}^{{\mathrm{{{stars}}}}}$"


def _paired_stars(ax, x0, x1, y, p_val, h):
  ax.plot([x0, x0, x1, x1], [y, y + h, y + h, y], color="k", linewidth=LW_MAIN)
  ax.text(
    (x0 + x1) / 2, y + h,
    p_to_stars(float(p_val)),
    ha="center", va="bottom", fontsize=FIG_FONT_PT,
  )


def _make_paired_r_df(src_df, global_key, clstr_key, weighted_key):
  out_df = src_df.copy()
  out_df[weighted_key] = out_df.apply(lambda row: _weighted_dict_r(row, clstr_key), axis=1)
  out_df = (
    out_df[["mv_name", "tsu_type", "cond", global_key, weighted_key]]
    .replace([np.inf, -np.inf], np.nan)
    .copy()
  )
  finite_any = (
    np.isfinite(out_df[global_key].to_numpy(dtype=float))
    | np.isfinite(out_df[weighted_key].to_numpy(dtype=float))
  )
  return out_df.loc[finite_any].copy()


def _paired_ttest_from_df(paired_df, left_key, right_key):
  left_v = paired_df[left_key].to_numpy(dtype=float)
  right_v = paired_df[right_key].to_numpy(dtype=float)
  pair_mask = np.isfinite(left_v) & np.isfinite(right_v)
  if np.sum(pair_mask) >= 2:
    tt_res = sp.stats.ttest_rel(
      right_v[pair_mask], left_v[pair_mask],
      nan_policy="omit", alternative="greater")
    return int(np.sum(pair_mask)), float(tt_res.statistic), float(tt_res.pvalue)
  return int(np.sum(pair_mask)), np.nan, np.nan


def _one_sample_greater_zero_holm(df, group_col, value_col, group_order, label_map):
  row_s = []
  for group in group_order:
    v = df.loc[df[group_col].astype(str) == str(group), value_col].to_numpy(dtype=float)
    v = v[np.isfinite(v)]
    if v.size >= 2:
      tt_res = sp.stats.ttest_1samp(v, popmean=0.0, alternative="greater")
      stat = float(tt_res.statistic)
      p_unc = float(tt_res.pvalue)
    else:
      stat = np.nan
      p_unc = np.nan
    row_s.append({
      "test": f"{label_map.get(str(group), str(group))} > 0",
      "stat": stat,
      "p_unc": p_unc,
      "p_holm": np.nan,
    })
  out_tbl = pd.DataFrame(row_s)
  valid = np.isfinite(out_tbl["p_unc"].to_numpy(dtype=float))
  if np.any(valid):
    out_tbl.loc[valid, "p_holm"] = multipletests(out_tbl.loc[valid, "p_unc"].to_numpy(dtype=float), method="holm")[1]
  return out_tbl


def _build_cluster_delta_df(src_df, clstr_r_key, global_r_key, delta_key):
  row_s = []
  for _, row in src_df.iterrows():
    clstr_scr_s = row.get("clstr_scr_s", np.nan)
    clstr_r_s = row.get(clstr_r_key, np.nan)
    global_r = float(row.get(global_r_key, np.nan))
    if not (isinstance(clstr_scr_s, dict) and isinstance(clstr_r_s, dict) and np.isfinite(global_r)):
      continue
    for lbl, scr in clstr_scr_s.items():
      lbl = int(lbl)
      if lbl == -1:
        continue
      scr = float(scr)
      r_val = float(clstr_r_s.get(lbl, np.nan))
      delta = r_val - global_r
      if not (np.isfinite(scr) and np.isfinite(delta)):
        continue
      row_s.append({
        "mv_name": row["mv_name"],
        "tsu_type": row["tsu_type"],
        "cond": row["cond"],
        "label": lbl,
        "clstr_scr": scr,
        delta_key: delta,
      })
  return pd.DataFrame(row_s)


def _prep_delta_group_df(delta_df, delta_key):
  out_df = (
    delta_df[["mv_name", "tsu_type", "cond", "label", delta_key]]
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=[delta_key])
    .copy()
  )
  out_df["group"] = out_df["tsu_type"].astype(str) + "_" + out_df["cond"].astype(str)
  return out_df


def _top_cluster_labels(clstr_lbl, n=2):
  clstr_lbl = np.asarray(clstr_lbl, dtype=int)
  lbl_s = [int(lbl) for lbl in np.unique(clstr_lbl) if int(lbl) != -1]
  if len(lbl_s) < n:
    raise ValueError(f"Need at least {n} non-noise HDBSCAN clusters; found {len(lbl_s)}")
  lbl_s = sorted(lbl_s, key=lambda lbl: (-int(np.sum(clstr_lbl == lbl)), lbl))
  return lbl_s[:n]


def _latency_limits(latency):
  finite_lat = np.asarray(latency, dtype=float)
  finite_lat = finite_lat[np.isfinite(finite_lat)]
  lat_abs = float(np.nanmax(np.abs(finite_lat))) if finite_lat.size > 0 else np.nan
  if (not np.isfinite(lat_abs)) or (lat_abs <= 0):
    lat_abs = 1.0
  return -lat_abs, lat_abs


early_n = 2


def _best_latency_reference(space_xy, latency, neur_mask, early_n):
  space_xy = np.asarray(space_xy, dtype=float)
  latency = np.asarray(latency, dtype=float)
  neur_mask = np.asarray(neur_mask, dtype=bool)
  stable_idx = np.where(neur_mask & np.isfinite(latency))[0]
  if stable_idx.size < 3:
    raise ValueError("Need at least 3 recruited neurons for distance-latency correlation demo")

  early_cnt = min(int(early_n), stable_idx.size)
  early_idx = stable_idx[np.argsort(latency[stable_idx])[:early_cnt]]
  ref_xy = np.nanmean(space_xy[early_idx], axis=0)
  ref_time = float(np.nanmean(latency[early_idx]))

  dist = np.linalg.norm(space_xy - ref_xy, axis=-1)
  dt = latency - ref_time
  one_dist = dist[stable_idx]
  one_dt = dt[stable_idx]
  good = np.isfinite(one_dist) & np.isfinite(one_dt)
  if np.sum(good) < 3:
    raise ValueError("Could not compute any finite distance-latency correlation for demo movie")
  if np.isclose(np.nanstd(one_dist[good]), 0) or np.isclose(np.nanstd(one_dt[good]), 0):
    raise ValueError("Could not compute any finite distance-latency correlation for demo movie")

  dist_time = np.vstack([one_dist[good], one_dt[good]])
  cor, p_val = sp.stats.pearsonr(*dist_time)
  return dict(
    cor=float(cor),
    p=float(p_val),
    dist_time=dist_time,
    ref_idx=int(early_idx[0]),
    ref_xy=np.asarray(ref_xy, dtype=float),
    ref_time=ref_time,
  )


def _add_cluster_halos(ax, xy, clstr_lbl, zorder=1):
  xy = np.asarray(xy, dtype=float)
  clstr_lbl = np.asarray(clstr_lbl, dtype=int)
  lbl_color = _cluster_color_map(clstr_lbl)
  for lbl in np.sort(np.unique(clstr_lbl)):
    lbl = int(lbl)
    mask = clstr_lbl == lbl
    if not np.any(mask):
      continue
    ax.scatter(
      xy[mask, 0], xy[mask, 1],
      facecolors="none",
      edgecolors=lbl_color.get(lbl, "gray"),
      s=S_SCATTER_HIGHLIGHT,
      linewidths=1.7 * LW_MAIN,
      alpha=0.55,
      zorder=zorder,
    )

def _plot_latency_embedding(ax, mv_name, space_key="tot_mds", add_colorbar=False, add_halos=True, annotate_ref=False, colorbar_location="right", *, early_n):
  space_xy = np.asarray(mv_smry_s[space_key][mv_name], dtype=float)
  if space_key == "com":
    space_xy = space_xy * (50 / 55)
  latency = np.asarray(mv_smry_s["mean_spk_time"][mv_name], dtype=float) * 0.264
  clstr_lbl = np.asarray(mv_smry_s["clstr_lbl"][mv_name], dtype=int)
  brst_neur = np.asarray(mv_smry_s.get("brst_neur", {}).get(mv_name, np.isfinite(latency)), dtype=bool)
  vmin, vmax = _latency_limits(latency)
  finite = brst_neur & np.isfinite(latency)

  sc = ax.scatter(
    space_xy[finite, 0], space_xy[finite, 1],
    c=latency[finite], vmin=vmin, vmax=vmax, cmap="summer",
    s=S_SCATTER, alpha=ALPHA_MAIN, linewidths=0, zorder=2,
  )
  other = ~finite
  if np.any(other):
    ax.scatter(
      space_xy[other, 0], space_xy[other, 1],
      color="gray", marker="x", s=S_SCATTER_SMALL,
      alpha=ALPHA_SECONDARY, linewidths=LW_SECONDARY, zorder=2,
    )
  if add_halos:
    _add_cluster_halos(ax, space_xy, clstr_lbl, zorder=1)

  if annotate_ref:
    cluster_lbl_s = _top_cluster_labels(clstr_lbl, n=2)
    lbl_color = _cluster_color_map(clstr_lbl)
    centroid_obs_s = [(brst_neur, "k")]
    centroid_obs_s.extend([
      (brst_neur & (clstr_lbl == lbl), lbl_color.get(int(lbl), "gray"))
      for lbl in cluster_lbl_s
    ])
    for mask, color in centroid_obs_s:
      obs = _best_latency_reference(space_xy, latency, mask, early_n)
      ax.scatter(
        obs["ref_xy"][0], obs["ref_xy"][1],
        color=color,
        marker="*", s=0.55 * S_SCATTER_HIGHLIGHT, linewidths=LW_MAIN, zorder=4,
      )

  if space_key == "tot_mds":
    _set_mds_limits(ax, space_xy)
  else:
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel(r"$\mathrm{x\ }(\mu\mathrm{m})$")
    ax.set_ylabel(r"$\mathrm{y\ }(\mu\mathrm{m})$")
    absmax = float(np.nanmax(np.abs(space_xy[:, :2]))) if space_xy.size > 0 else np.nan
    if (not np.isfinite(absmax)) or (absmax <= 0):
      absmax = 1.0
    x_lim = 1.15 * absmax
    y_lim = 1.08 * absmax
    x_max = float(np.nanmax(space_xy[:, 0])) if space_xy.size > 0 else np.nan
    y_max = float(np.nanmax(space_xy[:, 1])) if space_xy.size > 0 else np.nan
    if not np.isfinite(x_max):
      x_max = 0.0
    if not np.isfinite(y_max):
      y_max = 0.0
    ax.set_xlim(x_max - x_lim, x_lim)
    ax.set_ylim(y_max - y_lim, y_lim)
  title_y = 1.24 if (add_colorbar and colorbar_location == "top") else None
  ax.set_title(r"$\mathrm{activation\ latency\ (s)}$", fontsize=FIG_FONT_PT, y=title_y)
  if add_colorbar:
    divider = make_axes_locatable(ax)
    if colorbar_location == "top":
      cax = divider.append_axes("top", size="6%", pad=0.06)
      plt.colorbar(sc, cax=cax, orientation="horizontal")
      cax.xaxis.set_ticks_position("top")
      cax.xaxis.set_label_position("top")
    else:
      cax = divider.append_axes("right", size="6%", pad=0.06)
      plt.colorbar(sc, cax=cax)
  return sc


def _plot_current_eta(ax, eta_df, eta_group_present, eta_test_tbl):
  sns.boxplot(data=eta_df, x="cond", y="clstr_time_eta2", order=eta_group_present, ax=ax, **box_kw_white)
  eta_x_s = {grp: idx for idx, grp in enumerate(eta_group_present)}
  for cond in eta_group_present:
    for tsu_type in ORDER:
      one_df = eta_df[(eta_df["cond"].astype(str) == cond) & (eta_df["tsu_type"].astype(str) == tsu_type)]
      if one_df.empty:
        continue
      grp_idx = eta_x_s[cond]
      ax.scatter(
        np.full(len(one_df), grp_idx, dtype=float),
        one_df["clstr_time_eta2"].to_numpy(dtype=float),
        marker=TSU_MARKER.get(str(tsu_type), "o"),
        color=COND_COLOR.get(str(cond), "0.4"),
        s=S_SCATTER_HIGHLIGHT, alpha=ALPHA_MAIN, linewidths=0, zorder=3,
      )
  ax.set_xlabel("")
  ax.set_ylabel(r"$\mathrm{Cluster\ effect\ }\eta^2\mathrm{\ for\ latency}$")
  ax.set_xticks(np.arange(len(eta_group_present)))
  ax.set_xticklabels([{"spon": r"$\mathrm{spontaneous}$", "stim": r"$\mathrm{excited}$"}.get(str(g), rf"$\mathrm{{{g}}}$") for g in eta_group_present])
  ax.set_ylim(-0.05, 1.15)
  ax.set_yticks(np.arange(0.0, 1.01, 0.2))
  eta_scale_p_holm = float(eta_test_tbl.loc[eta_test_tbl["test"] == "scale", "p_holm"].iloc[0])
  if (len(eta_group_present) >= 2) and np.isfinite(eta_scale_p_holm):
    x0, x1 = 0, 1
    y = 1.04
    h = 0.04
    ax.plot([x0, x0, x1, x1], [y, y + h, y + h, y], color="k", linewidth=LW_MAIN)
    ax.text(
      (x0 + x1) / 2, y + h,
      rf"$\mathrm{{scale}}:\ {p_to_stars(eta_scale_p_holm)}$",
      ha="center", va="bottom", fontsize=FIG_FONT_PT,
    )


def _finish_paper_figure(fig):
  for _ax in fig.axes:
    _ax.xaxis.label.set_size(FIG_FONT_PT)
    _ax.yaxis.label.set_size(FIG_FONT_PT)
    _ax.xaxis.labelpad = FIG_XLABELPAD
    _ax.yaxis.labelpad = FIG_YLABELPAD
    _ax.title.set_size(FIG_FONT_PT)
    _ax.tick_params(labelsize=FIG_FONT_PT)
    for _txt in _ax.texts:
      _txt.set_fontsize(FIG_FONT_PT)
    _leg = _ax.get_legend()
    if _leg is not None:
      _leg.get_title().set_fontsize(FIG_FONT_PT)
      for _txt in _leg.get_texts():
        _txt.set_fontsize(FIG_FONT_PT)


paired_df = _make_paired_r_df(cluster_mds_df, "x_t_cor", "clstr_x_t_cor", "clstr_x_t_cor_wgt")
if paired_df.empty:
  raise ValueError("Need at least 1 finite movie for figure:cluster real-space correlation paired plot")
clstr_vs_global_n, clstr_vs_global_t, clstr_vs_global_p = _paired_ttest_from_df(paired_df, "x_t_cor", "clstr_x_t_cor_wgt")
print("figure:cluster paired comparison: weighted cluster real-space correlation > global real-space correlation")
print(f"cluster vs global: n={clstr_vs_global_n}, paired t={clstr_vs_global_t:.3g}, p={clstr_vs_global_p:.3g}")

x_delta_df = _build_cluster_delta_df(cluster_mds_df, "clstr_x_t_cor", "x_t_cor", "x_t_cor_delta")
x_delta_group_df = _prep_delta_group_df(x_delta_df, "x_t_cor_delta")
x_delta_tsu_present = [tsu_type for tsu_type in ORDER if tsu_type in set(x_delta_group_df["tsu_type"].astype(str))]
x_delta_pair_s = [("org", "cab")] if set(ORDER).issubset(set(x_delta_tsu_present)) else []
x_delta_p_map = holm_pairwise_pvals(x_delta_group_df, "tsu_type", "x_t_cor_delta", x_delta_pair_s) if len(x_delta_pair_s) > 0 else {}
eta_df = cluster_mds_df[["mv_name", "tsu_type", "cond", "clstr_time_eta2", "clstr_time_eta2_p"]].replace([np.inf, -np.inf], np.nan).dropna(subset=["clstr_time_eta2"]).copy()
eta_group_present = [grp for grp in HUE_ORDER if grp in set(eta_df["cond"].astype(str))]
spon_eta_v = eta_df.loc[eta_df["cond"].astype(str) == "spon", "clstr_time_eta2"].to_numpy(dtype=float)
stim_eta_v = eta_df.loc[eta_df["cond"].astype(str) == "stim", "clstr_time_eta2"].to_numpy(dtype=float)
eta_scale_stat = np.nan
eta_scale_p = np.nan
if (spon_eta_v.size >= 2) and (stim_eta_v.size >= 2):
  eta_scale_res = sp.stats.levene(spon_eta_v, stim_eta_v, center="median")
  eta_scale_stat = float(eta_scale_res.statistic)
  eta_scale_p = float(eta_scale_res.pvalue)
eta_test_tbl = pd.DataFrame([{"test": "scale", "stat": eta_scale_stat, "p_unc": eta_scale_p, "p_holm": eta_scale_p, "n_spon": int(spon_eta_v.size), "n_stim": int(stim_eta_v.size)}])
print("\nclstr_time_eta2 spon vs stim scale test:")
print(eta_test_tbl[["test", "stat", "p_unc", "p_holm", "n_spon", "n_stim"]])

latency_score_label = r"$\mathrm{DBCV\ on\ latency}$"
latency_corr_df = (
  cluster_mds_df[["mv_name", "tsu_type", "cond", "t_self_clstr_scr", "clstr_scr"]]
  .replace([np.inf, -np.inf], np.nan)
  .dropna(subset=["t_self_clstr_scr", "clstr_scr"])
  .copy()
)
if len(latency_corr_df) >= 3:
  latency_scr_r, latency_scr_p = sp.stats.pearsonr(
    latency_corr_df["t_self_clstr_scr"].to_numpy(dtype=float),
    latency_corr_df["clstr_scr"].to_numpy(dtype=float),
  )
else:
  latency_scr_r, latency_scr_p = np.nan, np.nan
print(
  "DBCV on latency vs DBCV on activity forms: "
  f"n={len(latency_corr_df)}, pearson r={latency_scr_r:.3g}, p={latency_scr_p:.3g}"
)

# Combined cluster / MDS figure.
cluster_demo_mv = CLUSTER_DEMO_MV
if cluster_demo_mv not in mv_smry_s.get("mv_name", {}):
  raise KeyError(f"{cluster_demo_mv} missing from mv_smry_s; run per-movie analysis first")
for _field in ["tot_mds", "mean_spk_time", "clstr_lbl", "part_lbl", "clstr_scr", "part_clstr_scr"]:
  _require_mv_field(_field, cluster_demo_mv)

cluster_demo_tot_mds = np.asarray(mv_smry_s["tot_mds"][cluster_demo_mv], dtype=float)
cluster_demo_clstr_lbl = np.asarray(mv_smry_s["clstr_lbl"][cluster_demo_mv], dtype=int)
cluster_demo_part_lbl = np.asarray(mv_smry_s["part_lbl"][cluster_demo_mv], dtype=int)
cluster_demo_clstr_scr = float(mv_smry_s["clstr_scr"].get(cluster_demo_mv, np.nan))
cluster_demo_part_clstr_scr = float(mv_smry_s["part_clstr_scr"].get(cluster_demo_mv, np.nan))

paired_df = (
  cluster_plot_df[["mv_name", "tsu_type", "cond", "part_clstr_scr", "clstr_scr"]]
  .replace([np.inf, -np.inf], np.nan)
  .dropna(subset=["part_clstr_scr", "clstr_scr"])
  .drop_duplicates(subset=["mv_name"])
  .copy()
)
if paired_df.empty:
  raise ValueError("No valid paired DBCV rows; rerun per-movie analysis after the clustering update")
stat_df = (
  cluster_plot_df[["tsu_type", "cond", "clstr_scr"]]
  .replace([np.inf, -np.inf], np.nan)
  .dropna()
  .copy()
)
if stat_df.empty:
  raise ValueError("No valid rows for clstr_scr stats panel")

latency_demo_mv = CLUSTER_LATENCY_DEMO_MV
if latency_demo_mv not in mv_smry_s.get("mv_name", {}):
  raise KeyError(f"{latency_demo_mv} missing from mv_smry_s; run per-movie analysis first")
for _field in ["tot_mds", "mean_spk_time", "brst_neur", "clstr_lbl"]:
  _require_mv_field(_field, latency_demo_mv)

fig = plt.figure(figsize=(6, 4.15), constrained_layout=True)
outer_gs = fig.add_gridspec(2, 1, height_ratios=[1.3, 1])
row1_gs = outer_gs[0, 0].subgridspec(1, 3, width_ratios=[1, 0.2, 1])
stat_gs = outer_gs[1, 0].subgridspec(1, 3, width_ratios=[1, 1, 1])

ax_cluster_embed = fig.add_subplot(row1_gs[0, 0])
ax_cluster_legend = fig.add_subplot(row1_gs[0, 1])
ax_latency_mds = fig.add_subplot(row1_gs[0, 2])
ax_cluster_scores = fig.add_subplot(stat_gs[0, 0])
ax_pair_scores = fig.add_subplot(stat_gs[0, 1])
ax_latency_score = fig.add_subplot(stat_gs[0, 2])

_plot_combined_mds(
  ax_cluster_embed, cluster_demo_tot_mds, cluster_demo_part_lbl, cluster_demo_clstr_lbl,
  cluster_demo_part_clstr_scr, cluster_demo_clstr_scr)
cluster_leg = ax_cluster_embed.get_legend()
if cluster_leg is not None:
  cluster_handles = cluster_leg.legend_handles if hasattr(cluster_leg, "legend_handles") else cluster_leg.legendHandles
  cluster_labels = [txt.get_text() for txt in cluster_leg.get_texts()]
  cluster_leg.remove()
  ax_cluster_legend.legend(
    handles=cluster_handles,
    labels=cluster_labels,
    loc="center left",
    frameon=False,
    borderaxespad=0.0,
    handlelength=1.0,
    labelspacing=0.45,
  )
ax_cluster_legend.axis("off")

latency_sc = _plot_latency_embedding(
  ax_latency_mds,
  latency_demo_mv,
  space_key="tot_mds",
  add_colorbar=False,
  add_halos=True,
  early_n=early_n,
)
latency_cbar = fig.colorbar(latency_sc, ax=ax_latency_mds, location="right", fraction=0.046, pad=0.04)

for _ax in [ax_cluster_embed, ax_latency_mds]:
  _ax.set_xlim(-1.2, 1.7)
  _ax.set_ylim(-1.5, 1.5)
  _ax.set_xticks([-1, 0, 1])
  _ax.set_yticks([-1, 0, 1])

_plot_paired_scores(ax_pair_scores, paired_df)
ax_pair_scores.set_yticks([-1.0, -0.5, 0.0, 0.5, 1.0])
ax_pair_scores.hlines(0.0, *ax_pair_scores.get_xlim(), colors="0.5", linestyles="--", linewidth=LW_REFERENCE)
_plot_org_cab_cluster_score(ax_cluster_scores, stat_df)
if ax_cluster_scores.get_legend() is not None:
  ax_cluster_scores.get_legend().remove()
cluster_condition_handles = [
  Line2D([], [], linestyle="", marker="o", markersize=LEGEND_MARKER_SMALL, color=COND_COLOR["spon"], label=r"$\mathrm{spont.}$"),
  Line2D([], [], linestyle="", marker="o", markersize=LEGEND_MARKER_SMALL, color=COND_COLOR["stim"], label=r"$\mathrm{excited}$"),
]
ax_cluster_scores.legend(
  handles=cluster_condition_handles, loc="upper center", ncol=len(cluster_condition_handles),
  fontsize=FIG_FONT_PT, frameon=False, borderpad=0.0, handletextpad=0.4, columnspacing=1.0,
)
ax_cluster_scores.set_ylim(-0.05, 1.0)
ax_cluster_scores.set_yticks(np.arange(0.0, 0.81, 0.2))

for tsu_type in ORDER:
  for cond in HUE_ORDER:
    mask = (
      (latency_corr_df["tsu_type"].astype(str) == tsu_type)
      & (latency_corr_df["cond"].astype(str) == cond)
    )
    if not np.any(mask):
      continue
    ax_latency_score.scatter(
      latency_corr_df.loc[mask, "t_self_clstr_scr"].to_numpy(dtype=float),
      latency_corr_df.loc[mask, "clstr_scr"].to_numpy(dtype=float),
      marker=TSU_MARKER.get(str(tsu_type), "o"),
      color=COND_COLOR.get(str(cond), "0.4"),
      s=S_SCATTER_HIGHLIGHT,
      alpha=ALPHA_MAIN,
      linewidths=0,
    )
ax_latency_score.set_xlabel(latency_score_label)
ax_latency_score.set_ylabel(r"$\mathrm{DBCV\ on\ activity\ forms}$")
ax_latency_score.set_ylim(-0.05, 0.9)
ax_latency_score.set_yticks(np.arange(0.0, 0.81, 0.2))
ax_latency_score.text(
  0.02, 0.02, _format_r_p(latency_scr_r, latency_scr_p),
  transform=ax_latency_score.transAxes, ha="left", va="bottom", fontsize=FIG_FONT_PT,
  bbox={"facecolor": "none", "edgecolor": "none", "alpha": 0.0})


_finish_paper_figure(fig)

add_panel_labels(fig, [
  ("A", ax_cluster_embed),
  ("B", ax_latency_mds),
  ("C", ax_cluster_scores, 0.5, 1.0 / 3.0),
  ("D", ax_pair_scores, 0.5, 1.0 / 3.0),
  ("E", ax_latency_score, 0.5, 1.0 / 3.0),
])

(main_path / "analysis/result").mkdir(parents=True, exist_ok=True)
fig.savefig(main_path / "analysis/result/m_cluster.pdf", format="pdf", bbox_inches="tight")

plt.show()




In [ ]:
# fig:physical

def _plot_distance_latency_series(ax, mv_name, space_key, cluster_n=2, annotation_ax=None, *, early_n):
  space_xy = np.asarray(mv_smry_s[space_key][mv_name], dtype=float)
  latency = np.asarray(mv_smry_s["mean_spk_time"][mv_name], dtype=float) * 0.264
  brst_neur = np.asarray(mv_smry_s["brst_neur"][mv_name], dtype=bool)
  clstr_lbl = np.asarray(mv_smry_s["clstr_lbl"][mv_name], dtype=int)
  cluster_lbl_s = sorted(_top_cluster_labels(clstr_lbl, cluster_n))
  lbl_color = _cluster_color_map(clstr_lbl)

  obs_s = [("global", brst_neur, "k")]
  obs_s.extend([
    (f"cluster {lbl}", brst_neur & (clstr_lbl == lbl), lbl_color.get(int(lbl), "gray"))
    for lbl in cluster_lbl_s
  ])

  line_s = []
  for label, mask, color in obs_s:
    obs = _best_latency_reference(space_xy, latency, mask, early_n)
    ax.scatter(
      obs["dist_time"][0], obs["dist_time"][1],
      color=color, s=S_SCATTER_SMALL, alpha=ALPHA_MAIN,
      linewidths=0, label=rf"$\mathrm{{{label}}}$",
    )
    p_key = "mds_t_cor_p" if space_key == "tot_mds" else "x_t_cor_p"
    p_val = obs["p"] if label != "global" else float(mv_smry_s.get(p_key, {}).get(mv_name, obs["p"]))
    line_s.append({
      "label": label,
      "text": _format_named_corr_line(label.replace(" ", "\\ "), obs["cor"], p_val),
      "color": color,
    })

  cluster_line_s = [line for line in line_s if line["label"].startswith("cluster ")]
  cluster_line_s = sorted(cluster_line_s, key=lambda line: int(line["label"].split()[1]))
  global_line_s = [line for line in line_s if line["label"] == "global"]
  annot_line_s = global_line_s + cluster_line_s

  ax.set_xlabel(r"$\mathrm{MDS\ distance\ to\ early\ centroid}$")
  ax.set_ylabel(r"$\mathrm{latency\ relative\ to\ reference\ (s)}$")
  if annotation_ax is not None:
    annotation_ax.axis("off")
  from matplotlib.offsetbox import AnchoredOffsetbox, TextArea, VPacker
  annot_text_s = [
    TextArea(
      line["text"],
      textprops={"color": line["color"], "fontsize": FIG_FONT_PT},
    )
    for line in annot_line_s
  ]
  annot_box = VPacker(children=annot_text_s, align="left", pad=0, sep=1)
  anchored_annot = AnchoredOffsetbox(
    loc="upper right", child=annot_box, frameon=False, pad=0, borderpad=0,
    bbox_to_anchor=(0.98, 0.98), bbox_transform=ax.transAxes,
  )
  ax.add_artist(anchored_annot)


def _plot_current_x_delta(ax, x_delta_group_df, x_delta_tsu_present, x_delta_pair_s, x_delta_p_map):
  sns.boxplot(data=x_delta_group_df, x="tsu_type", y="x_t_cor_delta", order=x_delta_tsu_present, ax=ax, **box_kw_white)
  for tsu_idx, tsu_type in enumerate(x_delta_tsu_present):
    for cond in HUE_ORDER:
      one_df = x_delta_group_df[(x_delta_group_df["tsu_type"].astype(str) == tsu_type) & (x_delta_group_df["cond"].astype(str) == cond)]
      if one_df.empty:
        continue
      ax.scatter(
        np.full(len(one_df), tsu_idx, dtype=float),
        one_df["x_t_cor_delta"].to_numpy(dtype=float),
        marker=TSU_MARKER.get(str(tsu_type), "o"),
        color=COND_COLOR.get(str(cond), "0.4"),
        s=S_SCATTER_HIGHLIGHT, alpha=ALPHA_MAIN, linewidths=0, zorder=3,
      )
  ax.set_xlabel("")
  ax.set_ylabel(r"$\mathrm{cluster\ -\ global\ cor.}$")
  ax.set_xticks(np.arange(len(x_delta_tsu_present)))
  ax.set_xticklabels([{"org": r"$\mathrm{BTO}$", "cab": r"$\mathrm{CCN}$"}.get(str(g), rf"$\mathrm{{{g}}}$") for g in x_delta_tsu_present])
  print_tsu_cond_anova(
    x_delta_group_df,
    "x_t_cor_delta",
    "ANOVA for figure:physical x_t_cor_delta by tissue x condition",
  )
  one_sample_tbl = _one_sample_greater_zero_holm(x_delta_group_df, "tsu_type", "x_t_cor_delta", x_delta_tsu_present, {"org": "BTO", "cab": "CCN"})
  print("ONE-SAMPLE tests for x_t_cor_delta by tissue (H1: tissue > 0):")
  print(one_sample_tbl[["test", "stat", "p_unc", "p_holm"]])

  if len(x_delta_pair_s) > 0:
    print_pairwise_annotation_table(
      x_delta_group_df, "tsu_type", "x_t_cor_delta", x_delta_pair_s, x_delta_p_map,
      "figure:physical x_t_cor_delta annotation Welch t-test",
    )
    annotate_box_stars(ax, x_delta_pair_s, x_delta_p_map, order=x_delta_tsu_present)
  y0, y1 = ax.get_ylim()
  ax.set_ylim(y0, max(y1, 1.10))


physical_stat_df = _build_multi_plot_df([
  "x_t_cor", "x_t_cor_p", "clstr_x_t_cor", "clstr_x_t_cor_p", "clstr_lbl", "clstr_scr_s",
])
physical_stat_df["clstr_x_t_cor_wgt"] = physical_stat_df.apply(lambda row: _weighted_dict_r(row, "clstr_x_t_cor"), axis=1)
physical_stat_df["x_t_cor_gain"] = physical_stat_df["clstr_x_t_cor_wgt"].to_numpy(dtype=float) - physical_stat_df["x_t_cor"].to_numpy(dtype=float)
physical_pick_df = (
  physical_stat_df[["mv_name", "x_t_cor_gain"]]
  .replace([np.inf, -np.inf], np.nan)
  .dropna(subset=["x_t_cor_gain"])
  .sort_values(["x_t_cor_gain", "mv_name"], ascending=[False, True], kind="mergesort")
)
if len(physical_pick_df) < 2:
  raise ValueError("Need at least 2 finite weighted cluster-global real-space correlation increases for figure:physical")
physical_rank_idx = 0
physical_demo_mv = str(physical_pick_df.iloc[physical_rank_idx]["mv_name"])
print(f"figure:physical selected movie name: {physical_demo_mv}")

x_delta_df = _build_cluster_delta_df(physical_stat_df, "clstr_x_t_cor", "x_t_cor", "x_t_cor_delta")
x_delta_group_df = _prep_delta_group_df(x_delta_df, "x_t_cor_delta")
x_delta_tsu_present = [tsu_type for tsu_type in ORDER if tsu_type in set(x_delta_group_df["tsu_type"].astype(str))]
x_delta_pair_s = [("org", "cab")] if set(ORDER).issubset(set(x_delta_tsu_present)) else []
x_delta_p_map = holm_pairwise_pvals(x_delta_group_df, "tsu_type", "x_t_cor_delta", x_delta_pair_s) if len(x_delta_pair_s) > 0 else {}


def _sig_fraction_from_pvals(p_val_s, alpha=ALPHA):
  p_val_s = np.asarray(p_val_s, dtype=float)
  finite = np.isfinite(p_val_s)
  denom = int(np.sum(finite))
  sig_n = int(np.sum(p_val_s[finite] < float(alpha))) if denom > 0 else 0
  frac = float(sig_n / denom) if denom > 0 else np.nan
  return frac, sig_n, denom


def _cluster_pvals_for_rows(row_s):
  p_val_s = []
  for p_dict in row_s["clstr_x_t_cor_p"]:
    if not isinstance(p_dict, dict):
      continue
    for lbl, p_val in p_dict.items():
      if int(lbl) == -1:
        continue
      p_val_s.append(float(p_val))
  return p_val_s


physical_sig_frac_row_s = []
for tsu_type, tissue_label in [("org", "BTO"), ("cab", "CCN"), ("combined", "combined")]:
  if tsu_type == "combined":
    one_df = physical_stat_df[physical_stat_df["tsu_type"].astype(str).isin(ORDER)]
  else:
    one_df = physical_stat_df[physical_stat_df["tsu_type"].astype(str) == str(tsu_type)]
  global_frac, global_sig_n, global_n = _sig_fraction_from_pvals(one_df["x_t_cor_p"].to_numpy(dtype=float))
  cluster_frac, cluster_sig_n, cluster_n = _sig_fraction_from_pvals(_cluster_pvals_for_rows(one_df))
  physical_sig_frac_row_s.append({
    "tissue": tissue_label,
    "global_sig_frac": global_frac,
    "global_sig_n/n": f"{global_sig_n}/{global_n}",
    "cluster_sig_frac": cluster_frac,
    "cluster_sig_n/n": f"{cluster_sig_n}/{cluster_n}",
  })
print("\nfigure:physical panel D significant real-space correlation fractions (p < 0.05; non-nan denominator):")
print(pd.DataFrame(physical_sig_frac_row_s)[[
  "tissue", "global_sig_frac", "global_sig_n/n",
  "cluster_sig_frac", "cluster_sig_n/n",
]])

for _field in ["com", "mean_spk_time", "brst_neur", "clstr_lbl"]:
  _require_mv_field(_field, physical_demo_mv)


physical_reg_mv = physical_demo_mv
physical_reg_space_xy = np.asarray(mv_smry_s["com"][physical_reg_mv], dtype=float)
physical_reg_latency = np.asarray(mv_smry_s["mean_spk_time"][physical_reg_mv], dtype=float) * 0.264
physical_reg_brst_neur = np.asarray(mv_smry_s["brst_neur"][physical_reg_mv], dtype=bool)
physical_reg_clstr_lbl = np.asarray(mv_smry_s["clstr_lbl"][physical_reg_mv], dtype=int)
physical_reg_cluster_lbl_s = sorted(_top_cluster_labels(physical_reg_clstr_lbl, 2))
physical_reg_lbl_color = _cluster_color_map(physical_reg_clstr_lbl)
physical_reg_obs_s = [
  ("global", physical_reg_brst_neur, "k"),
  (r"cluster\ 1", physical_reg_brst_neur & (physical_reg_clstr_lbl == physical_reg_cluster_lbl_s[0]), physical_reg_lbl_color.get(int(physical_reg_cluster_lbl_s[0]), "gray")),
  (r"cluster\ 2", physical_reg_brst_neur & (physical_reg_clstr_lbl == physical_reg_cluster_lbl_s[1]), physical_reg_lbl_color.get(int(physical_reg_cluster_lbl_s[1]), "gray")),
]

print(f"figure:physical-reg selected movie name: {physical_reg_mv}")
print(f"figure:physical-reg cluster 1/cluster 2 source labels: {physical_reg_cluster_lbl_s[0]}, {physical_reg_cluster_lbl_s[1]}")


def _plot_physical_reg_panel(ax, label, mask, color):
  obs = _best_latency_reference(physical_reg_space_xy, physical_reg_latency, mask, early_n)
  x = np.asarray(obs["dist_time"][0], dtype=float)
  y = np.asarray(obs["dist_time"][1], dtype=float)
  ax.scatter(
    x, y,
    color=color,
    s=S_SCATTER_SMALL,
    alpha=ALPHA_MAIN,
    linewidths=0,
    zorder=2,
  )
  good = np.isfinite(x) & np.isfinite(y)
  if (np.sum(good) >= 2) and (not np.isclose(np.nanstd(x[good]), 0)):
    fit_coef = np.polyfit(x[good], y[good], 1)
    fit_x = np.linspace(np.nanmin(x[good]), np.nanmax(x[good]), 100)
    ax.plot(
      fit_x,
      np.polyval(fit_coef, fit_x),
      color=color,
      linestyle="-",
      linewidth=LW_REFERENCE,
      alpha=ALPHA_MAIN,
      zorder=3,
    )
  star_txt = p_to_stars(float(obs["p"]))
  if star_txt == "ns":
    star_txt = "n.s."
  ax.text(
    0.04,
    0.96,
    rf"$r:\ {float(obs['cor']):.2f}^{{\mathrm{{{star_txt}}}}}$",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=FIG_FONT_PT,
    color=color,
    bbox={"facecolor": "white", "edgecolor": color, "linewidth": 0.5 * LW_REFERENCE, "alpha": 0.85, "pad": 2.0},
  )
  ax.set_title(rf"$\mathrm{{{label}}}$", fontsize=FIG_FONT_PT, pad=3.0)
  ax.set_xlabel(r"$\mathrm{physical\ distance\ to\ reference}$")
  ax.set_ylabel(r"$\mathrm{latency\ relative}$" + "\n" + r"$\mathrm{to\ reference\ (s)}$")

fig = plt.figure(figsize=(6, 5.0), constrained_layout=True)
fig.get_layout_engine().set(h_pad=0, hspace=0)
physical_outer_gs = fig.add_gridspec(1, 2, width_ratios=[1.7, 1.0], wspace=0.0)
physical_left_gs = physical_outer_gs[0, 0].subgridspec(2, 1, height_ratios=[1.45, 1.0], hspace=0.0)
physical_a_gs = physical_left_gs[0, 0].subgridspec(1, 2, width_ratios=[1.0, 0.06], wspace=0.05)
ax_real_lat = fig.add_subplot(physical_a_gs[0, 0])
ax_real_lat_cbar = fig.add_subplot(physical_a_gs[0, 1])
physical_bottom_outer_gs = physical_left_gs[1, 0].subgridspec(2, 1, height_ratios=[0.06, 0.94], hspace=0.0)
ax_physical_bottom_spacer = fig.add_subplot(physical_bottom_outer_gs[0, 0])
ax_physical_bottom_spacer.axis("off")
physical_bottom_gs = physical_bottom_outer_gs[1, 0].subgridspec(1, 2, width_ratios=[1.0, 1.0], wspace=0.0)
ax_x_clstr_scr = fig.add_subplot(physical_bottom_gs[0, 0])
ax_x_delta = fig.add_subplot(physical_bottom_gs[0, 1])
physical_reg_gs = physical_outer_gs[0, 1].subgridspec(3, 1, hspace=0.03)
ax_physical_reg_s = [
  fig.add_subplot(physical_reg_gs[0, 0]),
  fig.add_subplot(physical_reg_gs[1, 0]),
  fig.add_subplot(physical_reg_gs[2, 0]),
]

physical_real_sc = _plot_latency_embedding(ax_real_lat, physical_demo_mv, space_key="com", add_colorbar=False, add_halos=True, annotate_ref=True, early_n=early_n)
plt.colorbar(physical_real_sc, cax=ax_real_lat_cbar)
ax_real_lat.set_anchor("NW")
ax_real_lat.set_aspect("equal")

for ax, (label, mask, color) in zip(ax_physical_reg_s, physical_reg_obs_s):
  _plot_physical_reg_panel(ax, label, mask, color)
physical_reg_shared_ylim = ax_physical_reg_s[0].get_ylim()
for ax in ax_physical_reg_s:
  ax.set_ylim(physical_reg_shared_ylim)
  ax.set_yticks([0, 1, 2])

x_clstr_scr_tsu_df = (
  _build_multi_plot_df(["x_clstr_scr"])[["mv_name", "tsu_type", "cond", "x_clstr_scr"]]
  .replace([np.inf, -np.inf], np.nan)
  .dropna(subset=["tsu_type", "x_clstr_scr"])
  .drop_duplicates(subset=["mv_name"])
  .copy()
)
if x_clstr_scr_tsu_df.empty:
  raise ValueError("No multi-bursting movies with x_clstr_scr for figure:physical physical DBCV plot")

x_clstr_scr_tsu_order = [g for g in ORDER if g in set(x_clstr_scr_tsu_df["tsu_type"].astype(str))]
if len(x_clstr_scr_tsu_order) < 2:
  raise ValueError("Need both BTO and CCN movies for figure:physical physical DBCV plot")

sns.boxplot(
  data=x_clstr_scr_tsu_df,
  x="tsu_type",
  y="x_clstr_scr",
  order=x_clstr_scr_tsu_order,
  ax=ax_x_clstr_scr,
  **box_kw_white,
)
for tsu_type in x_clstr_scr_tsu_order:
  for cond in HUE_ORDER:
    one_df = x_clstr_scr_tsu_df[
      (x_clstr_scr_tsu_df["tsu_type"].astype(str) == tsu_type)
      & (x_clstr_scr_tsu_df["cond"].astype(str) == cond)
    ]
    if one_df.empty:
      continue
    sns.stripplot(
      data=one_df,
      x="tsu_type",
      y="x_clstr_scr",
      order=x_clstr_scr_tsu_order,
      jitter=False,
      alpha=ALPHA_MAIN,
      color=COND_COLOR.get(str(cond), "0.4"),
      marker=TSU_MARKER.get(str(tsu_type), "o"),
      size=S_STRIP,
      linewidth=0,
      ax=ax_x_clstr_scr,
    )
if ax_x_clstr_scr.get_legend() is not None:
  ax_x_clstr_scr.get_legend().remove()

x_clstr_scr_pair_s = [("org", "cab")]
p_map = holm_pairwise_pvals(x_clstr_scr_tsu_df, "tsu_type", "x_clstr_scr", x_clstr_scr_pair_s)
print_pairwise_annotation_table(
  x_clstr_scr_tsu_df, "tsu_type", "x_clstr_scr", x_clstr_scr_pair_s, p_map,
  "figure:physical physical DBCV annotation Welch t-test",
)
annotate_box_stars(ax_x_clstr_scr, x_clstr_scr_pair_s, p_map, order=x_clstr_scr_tsu_order)

anova_tbl_x_clstr_scr_tsu = print_tsu_cond_anova(
  x_clstr_scr_tsu_df,
  "x_clstr_scr",
  "ANOVA for figure:physical physical DBCV by tissue x condition",
)
one_sample_row_s = []
for tsu_type in x_clstr_scr_tsu_order:
  vals = x_clstr_scr_tsu_df.loc[x_clstr_scr_tsu_df["tsu_type"].astype(str) == str(tsu_type), "x_clstr_scr"].to_numpy(dtype=float)
  vals = vals[np.isfinite(vals)]
  row = {
    "test": {"org": "BTO", "cab": "CCN"}.get(str(tsu_type), str(tsu_type)),
    "stat": np.nan,
    "p_unc": np.nan,
    "p_holm": np.nan,
  }
  if vals.size >= 2:
    tt = sp.stats.ttest_1samp(vals, popmean=0.0, alternative="two-sided", nan_policy="omit")
    row["stat"] = float(tt.statistic)
    row["p_unc"] = float(tt.pvalue)
  one_sample_row_s.append(row)
one_sample_tbl_x_clstr_scr = pd.DataFrame(one_sample_row_s)
valid_p = np.isfinite(one_sample_tbl_x_clstr_scr["p_unc"].to_numpy(dtype=float))
if np.any(valid_p):
  one_sample_tbl_x_clstr_scr.loc[valid_p, "p_holm"] = multipletests(
    one_sample_tbl_x_clstr_scr.loc[valid_p, "p_unc"].to_numpy(dtype=float),
    method="holm",
  )[1]
print("ONE-SAMPLE tests for x_clstr_scr by tissue (two-sided vs 0):")
print(one_sample_tbl_x_clstr_scr[["test", "stat", "p_unc", "p_holm"]])

ax_x_clstr_scr.set_xlabel("")
ax_x_clstr_scr.set_ylabel(r"$\mathrm{physical\ DBCV}$")
ax_x_clstr_scr.set_ylim(-0.8, 0.25)
ax_x_clstr_scr.set_yticks(np.arange(-0.8, 0.01, 0.2))
ax_x_clstr_scr.set_xticks(np.arange(len(x_clstr_scr_tsu_order)))
ax_x_clstr_scr.set_xticklabels([
  {"org": r"$\mathrm{BTO}$", "cab": r"$\mathrm{CCN}$"}.get(str(g), rf"$\mathrm{{{g}}}$")
  for g in x_clstr_scr_tsu_order
])
ax_x_clstr_scr.hlines(0.0, *ax_x_clstr_scr.get_xlim(), colors="0.5", linestyles="--", linewidth=LW_REFERENCE)

_plot_current_x_delta(ax_x_delta, x_delta_group_df, x_delta_tsu_present, x_delta_pair_s, x_delta_p_map)

physical_c_pooled_v = x_delta_group_df["x_t_cor_delta"].to_numpy(dtype=float)
physical_c_pooled_v = physical_c_pooled_v[np.isfinite(physical_c_pooled_v)]
if physical_c_pooled_v.size >= 2:
  physical_c_pooled_tt = sp.stats.ttest_1samp(physical_c_pooled_v, popmean=0.0, alternative="greater")
  physical_c_pooled_tbl = pd.DataFrame([{
    "test": "combined > 0",
    "n": int(physical_c_pooled_v.size),
    "mean": float(np.mean(physical_c_pooled_v)),
    "stat": float(physical_c_pooled_tt.statistic),
    "p_unc": float(physical_c_pooled_tt.pvalue),
  }])
else:
  physical_c_pooled_tbl = pd.DataFrame([{
    "test": "combined > 0",
    "n": int(physical_c_pooled_v.size),
    "mean": np.nan,
    "stat": np.nan,
    "p_unc": np.nan,
  }])
print("figure:physical panel D pooled x_t_cor_delta > 0 test:")
print(physical_c_pooled_tbl[["test", "n", "mean", "stat", "p_unc"]])

ax_x_delta.set_ylim(-0.5, 1.0)
ax_x_delta.set_yticks(np.arange(-0.4, 0.81, 0.2))
ax_x_delta.hlines(0.0, *ax_x_delta.get_xlim(), colors="0.5", linestyles="--", linewidth=LW_REFERENCE)

physical_condition_handles = [
  Line2D([], [], linestyle="", marker="o", markersize=LEGEND_MARKER_SMALL, color=COND_COLOR["spon"], label=r"$\mathrm{spont.}$"),
  Line2D([], [], linestyle="", marker="o", markersize=LEGEND_MARKER_SMALL, color=COND_COLOR["stim"], label=r"$\mathrm{excited}$"),
]
for _ax in [ax_x_clstr_scr, ax_x_delta]:
  _ax.legend(
    handles=physical_condition_handles, loc="upper center", bbox_to_anchor=(0.47, 1.0), ncol=len(physical_condition_handles),
    fontsize=FIG_FONT_PT, frameon=False, borderpad=0.0, handletextpad=0.4, columnspacing=1.0,
  )


_finish_paper_figure(fig)
add_panel_labels(fig, [
  ("A", ax_real_lat),
  ("B", ax_physical_reg_s),
  ("C", ax_x_clstr_scr, -1.0),
  ("D", ax_x_delta, -1.0),
])
(main_path / "analysis/result").mkdir(parents=True, exist_ok=True)
fig.savefig(main_path / "analysis/result/m_physical.pdf", format="pdf", bbox_inches="tight")

plt.show()


# supplement

In [ ]:
# fig:mode-more

# One-sample check of propagator growth and decay. Neutral stability is
# |mu| = 1. Each real eigenvalue and each complex-conjugate pair contribute
# one dynamically distinct mode. Taking one median per recording keeps
# recordings with more significant dynamical PCs from receiving more weight.
def _mode_more_component_moduli(mv_name):
  mode_traj = np.asarray(mv_smry_s["mode_traj"][mv_name], dtype=float)[::-1]
  stk_n = int(mv_smry_s["stk_n"][mv_name])
  frm_sep = int(mv_smry_s["frm_sep"][mv_name])
  frm_n = int(np.asarray(mv_smry_s["traj"][mv_name]).shape[1])
  stk_mtd = mv_smry_s["stk_mtd"][mv_name]
  col_lo = (stk_n - 1) * frm_sep if stk_mtd == "padding" else 0
  col_hi = frm_n if stk_mtd == "padding" else mode_traj.shape[1]
  col_s = np.arange(col_lo, col_hi - 1)
  x_1, x_2 = mode_traj[:, col_s], mode_traj[:, col_s + 1]
  a_fwd = x_2 @ np.linalg.pinv(x_1)
  try:
    a_bwd = x_1 @ np.linalg.pinv(x_2)
    val_s, vec_s = np.linalg.eig(a_fwd @ np.linalg.inv(a_bwd))
    a_prop = np.real(
      vec_s @ np.diag(np.sqrt(val_s.astype(complex))) @ np.linalg.inv(vec_s))
  except np.linalg.LinAlgError:
    a_prop = a_fwd
  mu_s = np.linalg.eigvals(a_prop)
  real_s = np.abs(np.imag(mu_s)) <= 1e-10
  component_s = real_s | (np.imag(mu_s) > 1e-10)
  return np.abs(mu_s[component_s])


growth_neutral_row_s = []
for mv_name in sorted(mv_smry_s["mv_name"].keys()):
  if mv_smry_s["brst_type"].get(mv_name) != "multi":
    continue
  component_mu_abs = _mode_more_component_moduli(mv_name)
  growth_neutral_row_s.append({
    "mv_name": mv_name,
    "component_n": int(component_mu_abs.size),
    "median_mu_abs": float(np.median(component_mu_abs)),
  })
growth_neutral_df = pd.DataFrame(growth_neutral_row_s)
growth_neutral_value_s = growth_neutral_df["median_mu_abs"].to_numpy(dtype=float)
growth_neutral_test = sp.stats.wilcoxon(
  growth_neutral_value_s - 1.0, alternative="two-sided")
growth_neutral_iqr = np.quantile(growth_neutral_value_s, [0.25, 0.75])
print("figure:mode-more dynamical-mode neutral-stability check:")
print(growth_neutral_df)
print(
  f"median |mu| = {np.median(growth_neutral_value_s):.6f}; "
  f"IQR = [{growth_neutral_iqr[0]:.6f}, {growth_neutral_iqr[1]:.6f}]; "
  f"two-sided one-sample Wilcoxon W = {growth_neutral_test.statistic:.0f}, "
  f"p = {growth_neutral_test.pvalue:.6f}")

# Two example recordings, one BTO and one CCN, each contributing a spectrum
# row and a dynamics row, followed by the dataset-wide summaries. The dynamical PC
# trajectories (G, H rows) and the propagator spectra came out of Figure 1, which
# now carries only the panels that answer the section's question directly.
MODE_MORE_MV_S = [
  ("org", MODE_N_DEMO_MV, r"$\mathrm{BTO}$"),
  ("cab", MODE_MORE_MV_CAB, r"$\mathrm{CCN}$"),
]


def _mode_more_load(mv_name):
  """Everything the per-recording panels need, for one movie."""
  for _field in ["stk_pc", "stk_num_rmt_spct", "mode_sign", "mode_n", "stk_n", "frm_sep",
                 "mode_traj", "traj", "frame_dt", "prop_freq_floor"]:
    if mv_name not in mv_smry_s.get(_field, {}):
      raise KeyError(f"{_field} missing for {mv_name}; run per-movie analysis first")

  stk_pc = mv_smry_s["stk_pc"][mv_name]
  var = np.asarray(stk_pc[0], dtype=float)
  vec = np.asarray(stk_pc[1], dtype=float)
  sign = np.asarray(mv_smry_s["mode_sign"][mv_name], dtype=float)
  n_raw = float(mv_smry_s["mode_n"].get(mv_name, np.nan))
  mode_n = int(n_raw) if np.isfinite(n_raw) and (n_raw > 0) else 0
  stk_n = int(mv_smry_s["stk_n"][mv_name])
  frm_sep = int(mv_smry_s["frm_sep"][mv_name])
  dt = float(mv_smry_s["frame_dt"][mv_name])
  mode_traj = np.asarray(mv_smry_s["mode_traj"][mv_name], dtype=float)
  if mode_traj.ndim == 1:
    mode_traj = mode_traj[None, :]
  sig_n = max(0, min(mode_n, int(sign.size), int(mode_traj.shape[0]), int(vec.shape[1])))
  if sig_n <= 0:
    raise ValueError(f"No significant dynamical PCs available for fig:mode-more in {mv_name}")
  if vec.shape[0] % stk_n != 0:
    raise ValueError(f"Cannot reshape dynamical PC vectors for fig:mode-more in {mv_name}")
  return {
    "mv_name": mv_name,
    "var": var,
    "vec": vec,
    "rmt_spct": np.asarray(mv_smry_s["stk_num_rmt_spct"][mv_name], dtype=float),
    "sign": sign,
    "mode_n": mode_n,
    "stk_n": stk_n,
    "frm_sep": frm_sep,
    "dt": dt,
    "mode_traj": mode_traj,
    "traj_frm_n": int(np.asarray(mv_smry_s["traj"][mv_name], dtype=float).shape[1]),
    "freq_floor": float(mv_smry_s["prop_freq_floor"][mv_name]),
    "neur_n": int(vec.shape[0] // stk_n),
    "sig_n": sig_n,
  }


mode_more_d_s = [(tsu, _mode_more_load(mv), lbl) for tsu, mv, lbl in MODE_MORE_MV_S]
for _tsu, _d, _ in mode_more_d_s:
  print(f"figure:mode-more {_tsu} movie: {_d['mv_name']}, significant dynamical PCs: {_d['sig_n']}")

mode_more_max_sig_n = max(d["sig_n"] for _, d, _ in mode_more_d_s)
mode_more_width = 0.7 * max(3.2, min(6.2, 0.95 * mode_more_max_sig_n))

def _build_oob_var_frac_df():
  row_s = []
  for mv_name in sorted(mv_smry_s["mv_name"].keys()):
    if mv_smry_s["brst_type"].get(mv_name) != "multi":
      continue

    eig_val = np.asarray(mv_smry_s.get("stk_pc", {}).get(mv_name, [None])[0], dtype=float)
    mode_n_val = mv_smry_s.get("mode_n", {}).get(mv_name, np.nan)
    mode_n_i = int(mode_n_val) if np.isfinite(mode_n_val) and (mode_n_val > 0) else 0

    total_var = float(np.nansum(eig_val)) if eig_val.size > 0 else np.nan
    if (mode_n_i > 0) and np.isfinite(total_var) and (total_var > 0):
      out_bulk_var = float(np.nansum(eig_val[-mode_n_i:]))
      var_frac = out_bulk_var / total_var
    else:
      out_bulk_var = np.nan
      var_frac = np.nan

    row_s.append({
      "mv_name": mv_name,
      "tsu_type": TSU_TYPE_MAP.get(mv_smry_s["tsu_type"].get(mv_name), mv_smry_s["tsu_type"].get(mv_name)),
      "cond": COND_MAP.get(mv_smry_s["cond"].get(mv_name), mv_smry_s["cond"].get(mv_name)),
      "mode_n": mode_n_i,
      "out_bulk_var": out_bulk_var,
      "total_var": total_var,
      "value": var_frac,
    })

  out_df = pd.DataFrame(row_s)
  out_df = out_df.replace([np.inf, -np.inf], np.nan).dropna(subset=["tsu_type", "value"]).copy()
  out_df = out_df[out_df["tsu_type"].isin(ORDER) & out_df["cond"].isin(HUE_ORDER)].copy()
  out_df["tsu_type"] = pd.Categorical(out_df["tsu_type"], categories=ORDER, ordered=True)
  out_df["cond"] = pd.Categorical(out_df["cond"], categories=HUE_ORDER, ordered=True)
  return out_df.reset_index(drop=True)


oob_var_frac_df = _build_oob_var_frac_df()
if oob_var_frac_df.empty:
  raise ValueError("No multi-bursting movies with out-of-bulk eigenvalue variance fractions for S1 plot")

oob_var_frac_order = [g for g in ORDER if g in set(oob_var_frac_df["tsu_type"].astype(str))]
if len(oob_var_frac_order) < 2:
  raise ValueError("Need both BTO and CCN movies for S1 out-of-bulk variance fraction plot")


fig = plt.figure(figsize=(mode_more_width + 2.4, 8.0), constrained_layout=True)
# Six rows. Per recording: the spectrum on its own row, then the dynamical PC trajectories
# beside the within-component activity forms. Then the two propagator spectra, then the
# three dataset-wide summaries.
mode_more_gs = fig.add_gridspec(
  6, 1, height_ratios=[0.5, 1.2, 0.5, 1.2, 1.3 * (2.0 / 3.0) + 0.6, 1.6 * 0.75])

ax_spectrum_s, ax_mode_traj_s, ax_detail_grid_s, ax_detail_sort_s = [], [], [], []
for _row, (_tsu, _d, _lbl) in zip((0, 2), mode_more_d_s):
  ax_spectrum_s.append(fig.add_subplot(mode_more_gs[_row, 0]))
  _dyn_gs = mode_more_gs[_row + 1, 0].subgridspec(
    1, 3, width_ratios=[1.15, 0.16, 1.5], wspace=0.06)
  ax_mode_traj_s.append(fig.add_subplot(_dyn_gs[0, 0]))
  ax_detail_sort_s.append(fig.add_subplot(_dyn_gs[0, 1]))
  _row_n = max(_d["sig_n"], 1)
  _col_n = max(min(4, _d["neur_n"]), 1)
  _detail_gs = _dyn_gs[0, 2].subgridspec(_row_n, _col_n, hspace=0.01, wspace=0.02)
  ax_detail_grid_s.append([
    [fig.add_subplot(_detail_gs[r, c]) for c in range(_col_n)] for r in range(_row_n)])

_prop_gs = mode_more_gs[4, 0].subgridspec(1, 2, wspace=0.06)
ax_prop_s = [fig.add_subplot(_prop_gs[0, k]) for k in range(2)]

_summary_gs = mode_more_gs[5, 0].subgridspec(1, 3, wspace=0.06)
ax_oob = fig.add_subplot(_summary_gs[0, 0])
ax_dyn_var = fig.add_subplot(_summary_gs[0, 1])
ax_n_osc = fig.add_subplot(_summary_gs[0, 2])


# ---- Per-recording panels, each drawn once for the BTO and once for the CCN
# ---- example. Bodies are the Figure 1 and former-supplement blocks, parameterized.

def _mm_plot_spectrum(ax, d):
  """Dynamical PC variances against the surrogate null spectrum."""
  dat = d["var"][np.isfinite(d["var"])]
  null = d["rmt_spct"][np.isfinite(d["rmt_spct"])]
  edge = np.array([np.min(null), np.max(null)]) if null.size > 0 else np.array([])
  blue = np.array([31.0, 119.0, 180.0]) / 255.0
  msize = 16
  inside = ((dat >= edge[0]) & (dat <= edge[1])) if edge.size == 2 else np.zeros(dat.shape, bool)
  if np.any(~inside):
    ax.scatter(dat[~inside], np.full(np.sum(~inside), 1.0), color=blue, s=msize, alpha=ALPHA_MAIN)
  if np.any(inside):
    ax.scatter(dat[inside], np.full(np.sum(inside), 1.0), color=np.array([0.68] * 3), s=msize, alpha=0.40)
  ax.scatter(null, np.full(null.shape, -1.0), color="tab:orange", s=msize, alpha=0.40)
  if edge.size > 0:
    ax.vlines(edge, -1.3, 1.3, color="tab:orange", linewidth=LW_REFERENCE, alpha=ALPHA_MAIN)
  all_x = np.concatenate([dat.ravel(), null.ravel(), edge.ravel()])
  if all_x.size > 0:
    lo, hi = float(np.min(all_x)), float(np.max(all_x))
    pad = 0.05 * (hi - lo) if hi > lo else 0.05 * max(abs(hi), 1.0)
    ax.set_xlim(lo - pad, hi + pad)
  ax.set_ylim(-1.6, 1.6)
  ax.set_xlabel(r"$\mathrm{dynamical\ PC\ variance}$")
  ax.set_ylabel("")
  ax.set_yticks([-1.0, 1.0])
  ax.set_yticklabels([r"$\mathrm{null}$", r"$\mathrm{data}$"])


def _mm_plot_mode_traj(ax, d):
  """Significant dynamical PC trajectories, ordered by decreasing variance."""
  sig_n = d["sig_n"]
  time_s = np.arange(d["mode_traj"].shape[1], dtype=float) * d["dt"]
  # One colour for every dynamical PC and one shared denominator, as in the Figure 1 original:
  # the traces are stacked to be compared with each other, so relative amplitude has to
  # survive and colour must not imply a cross-recording component identity the panel does not carry.
  trace_s = []
  for src_idx in np.arange(sig_n - 1, -1, -1, dtype=int):
    sign_val = float(d["sign"][src_idx]) if src_idx < d["sign"].size else 1.0
    if (not np.isfinite(sign_val)) or (sign_val == 0):
      sign_val = 1.0
    trace_s.append(sign_val * np.asarray(d["mode_traj"][src_idx], dtype=float))
  denom = float(np.nanmax([np.nanmax(np.abs(tr)) for tr in trace_s]))
  if (not np.isfinite(denom)) or (denom <= 0):
    denom = 1.0
  for plot_idx, trace in enumerate(trace_s):
    offset = float(sig_n - 1 - plot_idx)
    ax.axhline(offset, color="0.78", linewidth=0.45 * LW_REFERENCE, zorder=0)
    ax.plot(time_s, offset + (0.42 / denom) * trace,
            color="C0", lw=LW_MAIN, alpha=ALPHA_MAIN, zorder=2)
  ax.set_xlim(float(time_s[0]), float(time_s[-1]))
  ax.set_ylim(-0.65, float(sig_n - 1) + 0.65)
  ax.set_yticks([])
  ax.set_xlabel(r"$\mathrm{time\ (s)}$")
  ax.set_ylabel(r"$\mathrm{dynamical\ PC}$" + "\n" + r"$\mathrm{trajectory}$")
  for _spine in ax.spines.values():
    _spine.set_visible(False)
  ax.spines["bottom"].set_visible(True)
  ax.spines["bottom"].set_linewidth(0.5 * LW_REFERENCE)


def _mm_plot_sort_arrow(ax):
  ax.set_axis_off()
  ax.annotate(
    "", xy=(-0.20, 0.12), xytext=(-0.20, 0.88), xycoords="axes fraction",
    arrowprops=dict(arrowstyle="->", color="0.35", lw=0.6 * LW_REFERENCE),
    annotation_clip=False,
  )
  ax.text(0.33, 0.50,
          r"$\mathrm{PCs\ sorted\ by}$" + "\n" + r"$\mathrm{decreasing\ variance}$",
          transform=ax.transAxes, ha="center", va="center", rotation=-90, fontsize=FIG_FONT_PT)


def _mm_plot_detail(ax_grid, d):
  """Activity form of example neurons within each significant dynamical PC."""
  sig_n = d["sig_n"]
  show_neur_n = len(ax_grid[0])
  lag_s = np.arange(d["stk_n"], dtype=float) * d["frm_sep"] * d["dt"]
  color_s = plt.get_cmap("tab20")(np.linspace(0, 1, max(show_neur_n, 1)))
  sig_col_s = np.arange(d["vec"].shape[1] - sig_n, d["vec"].shape[1], dtype=int)
  for plot_idx, src_idx in enumerate(np.arange(sig_n - 1, -1, -1, dtype=int)):
    col_idx = int(sig_col_s[src_idx])
    sign_val = float(d["sign"][src_idx]) if src_idx < d["sign"].size else 1.0
    if (not np.isfinite(sign_val)) or (sign_val == 0):
      sign_val = 1.0
    mode_mtx = (sign_val * np.sqrt(d["stk_n"] * d["neur_n"])
                * d["vec"][:, col_idx].reshape((d["stk_n"], d["neur_n"])))
    for neur_plot_idx in range(show_neur_n):
      ax = ax_grid[plot_idx][neur_plot_idx]
      trace = np.asarray(mode_mtx[:, neur_plot_idx], dtype=float)
      denom = float(np.nanmax(np.abs(trace))) if trace.size else np.nan
      if (not np.isfinite(denom)) or (denom <= 0):
        denom = 1.0
      ax.axhline(0.0, color="0.86", linewidth=0.25 * LW_REFERENCE, zorder=0)
      ax.plot(lag_s, 0.36 * trace / denom,
              color=color_s[neur_plot_idx % len(color_s)], lw=LW_SECONDARY, alpha=0.70)
      ax.set_xlim(float(lag_s[0]), float(lag_s[-1]))
      ax.set_ylim(-0.48, 0.48)
      ax.set_yticks([])
      if plot_idx == 0:
        ax.set_title(rf"$\mathrm{{neuron\ {neur_plot_idx + 1}}}$", pad=2.0)
      if plot_idx == sig_n - 1:
        # ticks at the true delay-window endpoint, not a rounded integer
        ax.set_xticks([0.0, float(lag_s[-1])])
        ax.set_xticklabels([r"$\mathrm{0}$", rf"$\mathrm{{{lag_s[-1]:.2f}}}$"])
        ax.tick_params(axis="x", labelsize=FIG_FONT_PT, length=2.0, pad=1.0)
      else:
        ax.set_xticks([])
      for _spine in ["left", "right", "top", "bottom"]:
        ax.spines[_spine].set_visible(False)
      if plot_idx == sig_n - 1:
        ax.spines["bottom"].set_visible(True)
    # Three dots off the right edge of each row, marking the neurons not shown.
    for ellipsis_x in [1.22, 1.36, 1.50]:
      ax_grid[plot_idx][-1].plot(
        [ellipsis_x], [0.5], marker="o", linestyle="", markersize=1.4,
        color="0.35", transform=ax_grid[plot_idx][-1].transAxes, clip_on=False)
  # Labels on the middle row and middle column, as in the original supplement panel.
  ax_grid[(sig_n - 1) // 2][0].set_ylabel(r"$\mathrm{activity\ form}$")
  ax_grid[-1][(show_neur_n - 1) // 2].set_xlabel(r"$\mathrm{delay\ (s)}$")


def _mm_plot_propagator(ax, d, title):
  """Dynamical-mode amplitude against frequency for the fitted propagator."""
  traj = d["mode_traj"][::-1]
  lo = (d["stk_n"] - 1) * d["frm_sep"]
  hi = d["traj_frm_n"]
  col_s = np.arange(lo, hi - 1)
  x1, x2 = traj[:, col_s], traj[:, col_s + 1]
  a_fwd = x2 @ np.linalg.pinv(x1)
  a_bwd = x1 @ np.linalg.pinv(x2)
  w, v = np.linalg.eig(a_fwd @ np.linalg.inv(a_bwd))
  prop = np.real(v @ np.diag(np.sqrt(w.astype(complex))) @ np.linalg.inv(v))
  mu, phi = np.linalg.eig(prop)
  coord = np.linalg.solve(phi, traj[:, lo:hi])
  amp = np.sqrt(np.mean(np.abs(coord) ** 2, axis=1))
  freq = np.abs(np.angle(mu)) / (2 * np.pi * d["dt"])
  floor = d["freq_floor"]
  keep_s = np.where(np.imag(mu) >= 0)[0]
  f_plot = np.array([freq[i] for i in keep_s])
  a_plot = np.array([amp[i] * (np.sqrt(2.0) if np.imag(mu[i]) > 0 else 1.0) for i in keep_s])
  sort_s = np.argsort(f_plot)
  f_plot, a_plot = f_plot[sort_s], a_plot[sort_s]
  res_s = f_plot >= floor
  ax.vlines(f_plot, 0.0, a_plot, color="0.65", lw=LW_MAIN, zorder=1)
  if np.any(res_s):
    ax.plot(f_plot[res_s], a_plot[res_s], linestyle="", marker="o",
            markersize=3.4, color="0.15", zorder=2)
  if np.any(~res_s):
    ax.plot(f_plot[~res_s], a_plot[~res_s], linestyle="", marker="o",
            markersize=3.4, markerfacecolor="none", color="0.55", zorder=2)
  ax.axvline(floor, color="0.5", linestyle="--", linewidth=LW_REFERENCE)
  ax.set_xlim(-0.012, float(np.max(f_plot)) * 1.18 + 0.012)
  ax.set_ylim(0.0, float(np.max(a_plot)) * 1.20)
  ax.set_xlabel(r"$\mathrm{frequency\ (Hz)}$")
  ax.set_ylabel(r"$\mathrm{mode\ amplitude}$")
  ax.set_title(title, pad=2.0)


for _k, (_tsu, _d, _lbl) in enumerate(mode_more_d_s):
  _mm_plot_spectrum(ax_spectrum_s[_k], _d)
  ax_spectrum_s[_k].set_title(_lbl, pad=2.0)
  _mm_plot_mode_traj(ax_mode_traj_s[_k], _d)
  _mm_plot_sort_arrow(ax_detail_sort_s[_k])
  _mm_plot_detail(ax_detail_grid_s[_k], _d)
  _mm_plot_propagator(ax_prop_s[_k], _d, _lbl)

box_kw = {
  "boxprops": {"facecolor": "none", "edgecolor": "0.45", "linewidth": LW_MAIN},
  "whiskerprops": {"color": "k", "linewidth": LW_MAIN},
  "capprops": {"color": "k", "linewidth": LW_MAIN},
  "medianprops": {"color": "k", "linewidth": LW_MAIN},
}

sns.boxplot(
  data=oob_var_frac_df,
  x="tsu_type",
  y="value",
  order=oob_var_frac_order,
  ax=ax_oob,
  **box_kw,
)
for tsu_type in oob_var_frac_order:
  for cond in HUE_ORDER:
    one_df = oob_var_frac_df[
      (oob_var_frac_df["tsu_type"].astype(str) == tsu_type)
      & (oob_var_frac_df["cond"].astype(str) == cond)
    ]
    if one_df.empty:
      continue
    sns.stripplot(
      data=one_df,
      x="tsu_type",
      y="value",
      order=oob_var_frac_order,
      jitter=False,
      alpha=ALPHA_MAIN,
      color=COND_COLOR.get(str(cond), "0.4"),
      marker=TSU_MARKER.get(str(tsu_type), "o"),
      size=S_STRIP,
      linewidth=0,
      ax=ax_oob,
    )
if ax_oob.get_legend() is not None:
  ax_oob.get_legend().remove()

oob_var_frac_pair_s = [("org", "cab")]
p_map = holm_pairwise_pvals(oob_var_frac_df, "tsu_type", "value", oob_var_frac_pair_s)
print_pairwise_annotation_table(
  oob_var_frac_df, "tsu_type", "value", oob_var_frac_pair_s, p_map,
  "figure:projection out-of-bulk fraction annotation Welch t-test",
)
annotate_box_stars(ax_oob, oob_var_frac_pair_s, p_map, order=oob_var_frac_order)

anova_tbl_oob_var_frac = print_tsu_cond_anova(
  oob_var_frac_df,
  "value",
  "ANOVA for multi_out_of_bulk_fraction_total_variance by tissue x condition",
)

print("\nSignificant dynamical PC variance fraction group means:")
print(
  oob_var_frac_df.groupby("tsu_type", observed=True)["value"]
  .agg(n="count", mean="mean").reindex(oob_var_frac_order).reset_index()
)
print("\nOut-of-bulk fraction of total variance by movie:")
print(oob_var_frac_df[["mv_name", "tsu_type", "cond", "mode_n", "out_bulk_var", "total_var", "value"]])

ax_oob.set_xlabel("")
ax_oob.set_ylabel(r"$\mathrm{fraction\ of\ total\ var.}$")
ax_oob.set_xticks(np.arange(len(oob_var_frac_order)))
ax_oob.set_xticklabels([
  {"org": r"$\mathrm{BTO}$", "cab": r"$\mathrm{CCN}$"}.get(str(g), rf"$\mathrm{{{g}}}$")
  for g in oob_var_frac_order
])

# ---- C, D: quantities moved out of Figure 1 ----------------------------------
# C is the dynamical variance ratio, formerly Figure 1H. D counts how many
# dynamically distinct modes the propagator resolves, merging modes
# that are integer multiples of a common fundamental, since those are harmonics
# of one non-sinusoidal rhythm and not separate processes.
supp_df = _build_multi_plot_df()


def _is_one_harmonic_family(freq_s, k_max=6, tol=0.05):
  freq_s = np.sort(np.asarray(freq_s, dtype=float))
  if freq_s.size <= 1:
    return True
  for fund in np.linspace(freq_s.min() / k_max, freq_s.min() * 1.0001, 4000):
    order_s = np.round(freq_s / fund)
    if order_s.min() < 1 or order_s.max() > k_max:
      continue
    if len(set(order_s)) < len(order_s):
      continue
    if np.max(np.abs(freq_s - order_s * fund) / freq_s) < tol:
      return True
  return False


def _distinct_component_n(mv):
  freq_s = np.asarray(mv_smry_s["prop_freq"][mv], dtype=float)
  freq_s = np.sort(freq_s[freq_s >= mv_smry_s["prop_freq_floor"][mv]])
  family_n = len(freq_s)
  for size in range(1, len(freq_s) + 1):
    for assign in itertools.product(range(size), repeat=len(freq_s)):
      if len(set(assign)) != size:
        continue
      if all(_is_one_harmonic_family(freq_s[np.asarray(assign) == grp])
             for grp in range(size)):
        family_n = size
        break
    else:
      continue
    break
  if len(freq_s) == 0:
    family_n = 0
  return family_n + int(mv_smry_s["prop_real_n"][mv])


supp_df["n_distinct_osc"] = [_distinct_component_n(mv) for mv in supp_df["mv_name"]]


def _supp_box_strip(ax, stat_name, ylabel, ref=None):
  stat_df = (
    supp_df[["tsu_type", "cond", stat_name]]
    .replace([np.inf, -np.inf], np.nan).dropna().copy())
  sns.boxplot(data=stat_df, x="tsu_type", y=stat_name, order=ORDER, ax=ax, **box_kw)
  for tsu in ORDER:
    for cond in ("spon", "stim"):
      sub = stat_df[(stat_df["tsu_type"] == tsu) & (stat_df["cond"] == cond)]
      if sub.empty:
        continue
      sns.stripplot(
        data=sub, x="tsu_type", y=stat_name, order=ORDER, jitter=False,
        alpha=ALPHA_MAIN, color=COND_COLOR[cond], marker=TSU_MARKER[tsu],
        size=S_STRIP, linewidth=0, ax=ax)
  if ax.get_legend() is not None:
    ax.get_legend().remove()
  ax.set_xlabel("")
  ax.set_ylabel(ylabel)
  if ref is not None:
    ax.hlines(ref, *ax.get_xlim(), colors="0.5", linestyles="--",
              linewidth=LW_REFERENCE)
  pair_s = [("org", "cab")]
  p_map = holm_pairwise_pvals(stat_df, "tsu_type", stat_name, pair_s)
  print_pairwise_annotation_table(
    stat_df, "tsu_type", stat_name, pair_s, p_map,
    f"figure:mode-more {stat_name} annotation Welch t-test")
  annotate_box_stars(ax, pair_s, p_map, order=ORDER)
  ax.set_xticks(np.arange(len(ORDER)))
  ax.set_xticklabels([r"$\mathrm{BTO}$", r"$\mathrm{CCN}$"])
  return stat_df


dyn_var_df = _supp_box_strip(ax_dyn_var, "avg_dyn_var",
                             r"$\mathrm{dynamical\ var.\ ratio}$", ref=0.0)
_print_onesample_vs_one(dyn_var_df, "avg_dyn_var",
                        r"$\mathrm{dynamical\ var.\ ratio}$",
                        popmean=0.0, alternative="greater")
n_osc_df = _supp_box_strip(ax_n_osc, "n_distinct_osc",
                           r"$\mathrm{distinct\ modes}$", ref=1.0)
_print_onesample_vs_one(n_osc_df, "n_distinct_osc",
                        r"$\mathrm{distinct\ modes}$",
                        popmean=1.0, alternative="greater")

for _ax in fig.axes:
  _ax.tick_params(labelsize=FIG_FONT_PT)
  _ax.xaxis.label.set_size(FIG_FONT_PT)
  _ax.yaxis.label.set_size(FIG_FONT_PT)
  for _txt in _ax.texts:
    _txt.set_fontsize(FIG_FONT_PT)

add_panel_labels(fig, [
  ("A", ax_spectrum_s[0], 1.0),
  ("B", ax_mode_traj_s[0], 1.0),
  ("C", ax_detail_grid_s[0][0][0], 1.0),
  ("D", ax_spectrum_s[1], 1.0),
  ("E", ax_mode_traj_s[1], 1.0),
  ("F", ax_detail_grid_s[1][0][0], 1.0),
  ("G", ax_prop_s[0], 1.0),
  ("H", ax_prop_s[1], 1.0),
  ("I", ax_oob, 1.0),
  ("J", ax_dyn_var, 1.0),
  ("K", ax_n_osc, 1.0),
])

(main_path / "analysis/result").mkdir(parents=True, exist_ok=True)
fig.savefig(main_path / "analysis/result/s_mode_more.pdf", format="pdf", bbox_inches="tight")


In [ ]:
# fig:cond - segmentation and layout

traj_cond_spon_mv = "cab-6-1-spon"
traj_cond_stim_mv = "cab-6-1-stim"


def _plot_cnmf_segment_panel(ax, mv_name, ylabel_text):
  payload = _load_figure_imshow(mv_name)
  ax.imshow(payload["mean_img"], cmap="gray", aspect="equal")
  spat = payload["spat"]
  if spat.size > 0:
    color_s = plt.get_cmap("tab20")(np.linspace(0, 1, max(spat.shape[0], 1)))
    for spat_idx, one_spat in enumerate(spat):
      vmax = float(np.nanmax(one_spat))
      if (not np.isfinite(vmax)) or (vmax <= 0):
        continue
      try:
        ax.contour(
          one_spat,
          levels=[0.4 * vmax],
          colors=[color_s[spat_idx % len(color_s)]],
          linewidths=0.5 * LW_CONTOUR,
          alpha=ALPHA_SECONDARY,
        )
      except ValueError:
        continue
  ax.set_ylabel(ylabel_text, fontsize=FIG_FONT_PT, rotation=90, labelpad=2)
  ax.set_xticks([])
  ax.set_yticks([])
  for spine in ax.spines.values():
    spine.set_visible(False)


for _mv_name in [traj_cond_spon_mv, traj_cond_stim_mv]:
  if _mv_name not in mv_smry_s.get("cnmf_traj", {}):
    raise KeyError(f"{_mv_name} missing from mv_smry_s['cnmf_traj']; run per-movie analysis first")
print(f"figure:seg-cond spontaneous movie: {traj_cond_spon_mv}")
print(f"figure:seg-cond excited movie: {traj_cond_stim_mv}")

fig = plt.figure(figsize=(6, 4.4), constrained_layout=True)
condition_outer_gs = fig.add_gridspec(2, 1, height_ratios=[1.2, 1.0], hspace=0.12)
condition_top_outer_gs = condition_outer_gs[0, 0].subgridspec(1, 2, width_ratios=[4.0, 1.0], wspace=0.06)
condition_top_gs = condition_top_outer_gs[0, 0].subgridspec(2, 2, width_ratios=[1.05, 3.95], wspace=0.06, hspace=0.08)
ax_seg_s = [
  fig.add_subplot(condition_top_gs[0, 0]),
  fig.add_subplot(condition_top_gs[1, 0]),
]
ax_traj_s = [
  fig.add_subplot(condition_top_gs[0, 1]),
  fig.add_subplot(condition_top_gs[1, 1]),
]
ax_burst_rate = fig.add_subplot(condition_top_outer_gs[0, 1])
condition_bottom_gs = condition_outer_gs[1, 0].subgridspec(1, 4, wspace=0.0)
ax_effective_mode = fig.add_subplot(condition_bottom_gs[0, 0])
ax_dyn_var = fig.add_subplot(condition_bottom_gs[0, 1])
ax_cnmf_similarity = fig.add_subplot(condition_bottom_gs[0, 2])
ax_hdbscan_score = fig.add_subplot(condition_bottom_gs[0, 3])

_plot_cnmf_segment_panel(ax_seg_s[0], traj_cond_spon_mv, r"$\mathrm{spontaneous}$")
_plot_cnmf_segment_panel(ax_seg_s[1], traj_cond_stim_mv, r"$\mathrm{excited}$")

for _ax in ax_seg_s:
  _ax.tick_params(labelsize=FIG_FONT_PT)

# fig:cond - trajectories


def _presentation_movie_meta(mv_name):
  if "mv_name" in mv_smry_s and mv_name in mv_smry_s["mv_name"]:
    tsu = mv_smry_s.get("tsu", {}).get(mv_name, None)
    cond = mv_smry_s.get("cond", {}).get(mv_name, None)
    tsu_type = mv_smry_s.get("tsu_type", {}).get(mv_name, None)
  else:
    tsu = cond = tsu_type = None
  if tsu is None or cond is None or tsu_type is None:
    parsed = _parse_movie_name(mv_name)
    tsu = parsed["tsu"] if tsu is None else tsu
    cond = parsed["cond"] if cond is None else cond
    tsu_type = parsed["tsu_type"] if tsu_type is None else tsu_type
  return {
    "tsu": tsu,
    "cond": COND_MAP.get(cond, cond),
    "tsu_type": TSU_TYPE_MAP.get(tsu_type, tsu_type),
  }


def _presentation_find_cond_pair(preferred_mv):
  mv_name_s = sorted(mv_smry_s.get("cnmf_traj", {}).keys())
  if len(mv_name_s) == 0:
    raise ValueError("No CNMF trajectories available for fig:traj-cond")

  preferred_tsu = _presentation_movie_meta(preferred_mv)["tsu"] if preferred_mv in mv_name_s else None
  tsu_order = []
  if preferred_tsu is not None:
    tsu_order.append(preferred_tsu)
  tsu_order.extend([
    _presentation_movie_meta(mv_name)["tsu"]
    for mv_name in mv_name_s
    if _presentation_movie_meta(mv_name)["tsu"] not in tsu_order
  ])

  for tsu in tsu_order:
    cond_mv = {}
    for mv_name in mv_name_s:
      meta = _presentation_movie_meta(mv_name)
      if meta["tsu"] != tsu:
        continue
      if meta["cond"] in ["spon", "stim"]:
        cond_mv.setdefault(meta["cond"], mv_name)
    if {"spon", "stim"}.issubset(cond_mv):
      return cond_mv["spon"], cond_mv["stim"]

  raise ValueError("Could not find a spontaneous/excited CNMF trajectory pair for fig:traj-cond")


def _plot_cnmf_traj_panel(ax, mv_name, ylabel_text, show_xlabel):
  cnmf_traj = np.asarray(mv_smry_s["cnmf_traj"][mv_name], dtype=float)
  if cnmf_traj.ndim != 2 or cnmf_traj.size == 0:
    raise ValueError(f"Invalid CNMF trajectory for {mv_name}")
  cnmf_max = float(np.nanmax(cnmf_traj)) if cnmf_traj.size else np.nan
  if (not np.isfinite(cnmf_max)) or (cnmf_max <= 0):
    cnmf_max = 1.0
  cnmf_traj_norm = cnmf_traj / cnmf_max
  neur_n, frm_n = cnmf_traj_norm.shape
  time_s = np.arange(frm_n, dtype=float) * 0.264
  color_map = plt.get_cmap("tab20")
  neur_color_s = color_map(np.linspace(0, 1, max(neur_n, 1)))

  for neur_idx in range(neur_n):
    ax.plot(
      time_s,
      cnmf_traj_norm[neur_idx],
      color=neur_color_s[neur_idx % len(neur_color_s)],
      lw=LW_SECONDARY,
      alpha=0.45,
      zorder=2,
    )

  pop_mean = np.nanmean(cnmf_traj_norm, axis=0)
  ax.plot(
    time_s,
    pop_mean,
    color="white",
    lw=2.4 * LW_MAIN,
    alpha=0.95,
    zorder=4,
  )
  ax.plot(
    time_s,
    pop_mean,
    color="black",
    lw=1.25 * LW_MAIN,
    alpha=0.95,
    zorder=5,
  )

  ax.set_xlim(0.0, float(time_s[-1]) if time_s.size > 0 else 1.0)
  ax.set_ylim(-0.4, 1.05)
  ax.set_xlabel(r"$\mathrm{time\ (s)}$" if show_xlabel else "")
  ax.tick_params(labelbottom=show_xlabel)
  ax.set_ylabel(ylabel_text)
  ax.set_yticks([0, 1])


traj_cond_spon_mv = "cab-6-1-spon"
traj_cond_stim_mv = "cab-6-1-stim"
for _mv_name in [traj_cond_spon_mv, traj_cond_stim_mv]:
  if _mv_name not in mv_smry_s.get("cnmf_traj", {}):
    raise KeyError(f"{_mv_name} missing from mv_smry_s['cnmf_traj']; run per-movie analysis first")
print(f"figure:traj-cond spontaneous movie: {traj_cond_spon_mv}")
print(f"figure:traj-cond excited movie: {traj_cond_stim_mv}")

_plot_cnmf_traj_panel(ax_traj_s[0], traj_cond_spon_mv, r"$\mathrm{activity\ (a.u.)}$", show_xlabel=False)
_plot_cnmf_traj_panel(ax_traj_s[1], traj_cond_stim_mv, r"$\mathrm{activity\ (a.u.)}$", show_xlabel=True)

for _ax in ax_traj_s:
  _ax.xaxis.label.set_size(FIG_FONT_PT)
  _ax.yaxis.label.set_size(FIG_FONT_PT)
  _ax.xaxis.labelpad = FIG_XLABELPAD
  _ax.yaxis.labelpad = FIG_YLABELPAD
  _ax.title.set_size(FIG_FONT_PT)
  _ax.tick_params(labelsize=FIG_FONT_PT)

# fig:cond - condition summaries and save

condition_summary_df = (
  _build_multi_plot_df([
    "brst_n",
    "mode_n",
    "mode_pr",
    "avg_dyn_var",
    "flt_kendall",
    "clstr_scr",
  ])[[
    "mv_name",
    "tsu_type",
    "cond",
    "brst_n",
    "mode_n",
    "mode_pr",
    "avg_dyn_var",
    "flt_kendall",
    "clstr_scr",
  ]]
  .replace([np.inf, -np.inf], np.nan)
  .dropna(subset=["tsu_type", "cond"])
  .copy()
)
condition_summary_df["effective_mode_n"] = (
  condition_summary_df["mode_n"].to_numpy(dtype=float)
  * condition_summary_df["mode_pr"].to_numpy(dtype=float)
)
# Panel C includes every spontaneous or excited recording, including recordings
# with zero or one burst. Muscimol and any other conditions are excluded explicitly.
burst_rate_row_s = []
for mv_name in sorted(mv_smry_s.get("brst_n", {})):
  raw_cond = mv_smry_s.get("cond", {}).get(mv_name)
  if raw_cond not in {"spon", "spon-am", "bic", "stim"}:
    continue
  if mv_name not in mv_smry_s.get("cnmf_traj", {}) or mv_name not in mv_smry_s.get("frame_dt", {}):
    continue
  duration_min = (
    np.asarray(mv_smry_s["cnmf_traj"][mv_name]).shape[1]
    * float(mv_smry_s["frame_dt"][mv_name]) / 60.0
  )
  burst_rate_row_s.append({
    "mv_name": mv_name,
    "tsu_type": TSU_TYPE_MAP.get(
      mv_smry_s.get("tsu_type", {}).get(mv_name),
      mv_smry_s.get("tsu_type", {}).get(mv_name),
    ),
    "cond": COND_MAP.get(raw_cond, raw_cond),
    "burst_rate": float(mv_smry_s["brst_n"][mv_name]) / duration_min,
  })
burst_rate_df = pd.DataFrame(burst_rate_row_s)
print("\nfigure:cond raw conditions included in burst-rate panel:")
print(pd.Series([
  mv_smry_s["cond"][mv_name] for mv_name in burst_rate_df["mv_name"]
]).value_counts().sort_index())
print("\nfigure:cond burst rate by condition (bursts/min):")
print(burst_rate_df.groupby("cond", observed=True)["burst_rate"].agg(["count", "mean", "std"]))

condition_tissue_order = [
  grp for grp in ORDER
  if grp in set(condition_summary_df["tsu_type"].astype(str))
]
condition_group_order = [
  grp for grp in HUE_ORDER
  if grp in set(condition_summary_df["cond"].astype(str))
]
if len(condition_tissue_order) == 0 or len(condition_group_order) < 2:
  raise ValueError("Need tissue labels and both conditions for fig:cond")

_condition_box_kw = box_kw if "box_kw" in globals() else {
  "boxprops": {"facecolor": "none", "edgecolor": "0.45", "linewidth": LW_MAIN},
  "whiskerprops": {"color": "k", "linewidth": LW_MAIN},
  "capprops": {"color": "k", "linewidth": LW_MAIN},
  "medianprops": {"color": "k", "linewidth": LW_MAIN},
}


def _plot_condition_summary(ax, data_df, stat_name, y_label, print_name, alternative="two-sided"):
  stat_df = (
    data_df[["mv_name", "tsu_type", "cond", stat_name]]
    .dropna(subset=["cond", stat_name])
    .rename(columns={stat_name: "value"})
    .copy()
  )
  if stat_df.empty:
    raise ValueError(f"No finite values for {print_name} in fig:cond")

  sns.boxplot(
    data=stat_df,
    x="cond",
    y="value",
    order=condition_group_order,
    ax=ax,
    **_condition_box_kw,
  )
  for cond_idx, cond in enumerate(condition_group_order):
    for tsu_type in condition_tissue_order:
      one_df = stat_df[
        (stat_df["cond"].astype(str) == str(cond))
        & (stat_df["tsu_type"].astype(str) == str(tsu_type))
      ]
      if one_df.empty:
        continue
      ax.scatter(
        np.full(len(one_df), float(cond_idx), dtype=float),
        one_df["value"].to_numpy(dtype=float),
        marker=TSU_MARKER.get(str(tsu_type), "o"),
        color=COND_COLOR.get(str(cond), "0.4"),
        s=S_SCATTER_HIGHLIGHT,
        alpha=ALPHA_MAIN,
        linewidths=0,
        zorder=3,
      )

  if ax.get_legend() is not None:
    ax.get_legend().remove()

  cond_pair_s = [
    ("spon", "stim")
  ] if set(HUE_ORDER).issubset(set(condition_group_order)) else []
  cond_p_map = {}
  if len(cond_pair_s) > 0:
    if alternative == "two-sided":
      cond_p_map = holm_pairwise_pvals(
        stat_df,
        "cond",
        "value",
        cond_pair_s,
      )
      print_pairwise_annotation_table(
        stat_df,
        "cond",
        "value",
        cond_pair_s,
        cond_p_map,
        f"figure:cond {print_name} annotation Welch t-test",
      )
    else:
      group_1, group_2 = cond_pair_s[0]
      value_1 = stat_df.loc[stat_df["cond"] == group_1, "value"].to_numpy(dtype=float)
      value_2 = stat_df.loc[stat_df["cond"] == group_2, "value"].to_numpy(dtype=float)
      value_1 = value_1[np.isfinite(value_1)]
      value_2 = value_2[np.isfinite(value_2)]
      t_stat = np.nan
      p_value = np.nan
      if value_1.size >= 2 and value_2.size >= 2:
        t_test = sp.stats.ttest_ind(
          value_1,
          value_2,
          equal_var=False,
          nan_policy="omit",
          alternative=alternative,
        )
        t_stat = float(t_test.statistic)
        p_value = float(t_test.pvalue)
      cond_p_map[(group_1, group_2)] = p_value
      print(f"\nfigure:cond {print_name} one-sided Welch t-test ({group_1} {alternative} {group_2}):")
      print(pd.DataFrame([{
        "comparison": f"{group_1} vs {group_2}",
        "n_A": int(value_1.size),
        "n_B": int(value_2.size),
        "T": t_stat,
        "p_unc": p_value,
      }]))
    annotate_box_stars(
      ax,
      cond_pair_s,
      cond_p_map,
      order=condition_group_order,
    )

  if alternative in ("less", "greater"):
    print(
      f"\nANOVA omitted for {print_name}: the hypothesis is directional "
      f"(excitation changes activity in a known direction), which a two-sided "
      f"F-test cannot represent. See the one-sided Welch test above."
    )
  else:
    print_tsu_cond_anova(
      stat_df,
      "value",
      f"ANOVA for figure:cond {print_name} by tissue x condition",
    )

  ax.set_xlabel("")
  ax.set_ylabel(y_label)
  ax.set_xticks(np.arange(len(condition_group_order)))
  ax.set_xticklabels([
    {
      "spon": r"$\mathrm{Spont.}$",
      "stim": r"$\mathrm{excited}$",
    }.get(str(group), rf"$\mathrm{{{group}}}$")
    for group in condition_group_order
  ])
  return stat_df


_plot_condition_summary(
  ax_burst_rate,
  burst_rate_df,
  "burst_rate",
  r"$\mathrm{bursts\ per\ min}$",
  "burst rate",
  alternative="less",
)
_plot_condition_summary(
  ax_effective_mode,
  condition_summary_df,
  "effective_mode_n",
  r"$\mathrm{effective\ dimensionality}$",
  "effective dimensionality",
)
_plot_condition_summary(
  ax_dyn_var,
  condition_summary_df,
  "avg_dyn_var",
  r"$\mathrm{dynamical\ var.\ ratio}$",
  "dynamical variance ratio",
)
_plot_condition_summary(
  ax_cnmf_similarity,
  condition_summary_df,
  "flt_kendall",
  r"$\mathrm{CNMF\ order\ similarity}$",
  "CNMF order similarity",
)
_plot_condition_summary(
  ax_hdbscan_score,
  condition_summary_df,
  "clstr_scr",
  r"$\mathrm{HDBSCAN\ DBCV}$",
  "HDBSCAN DBCV",
)

ax_effective_mode.set_yticks(np.arange(0, 8.1, 2))
ax_effective_mode.set_ylim(-0.1, 8.1)
ax_effective_mode.hlines(
  1.0,
  *ax_effective_mode.get_xlim(),
  colors="0.5",
  linestyles="--",
  linewidth=LW_REFERENCE,
)

ax_dyn_var.set_yticks(np.arange(0, 0.81, 0.2))
ax_dyn_var.set_ylim(-0.05, 0.8)
ax_dyn_var.hlines(
  0.0,
  *ax_dyn_var.get_xlim(),
  colors="0.5",
  linestyles="--",
  linewidth=LW_REFERENCE,
)

for ax in [
  ax_burst_rate,
  ax_effective_mode,
  ax_dyn_var,
  ax_cnmf_similarity,
  ax_hdbscan_score,
]:
  ax.xaxis.label.set_size(FIG_FONT_PT)
  ax.yaxis.label.set_size(FIG_FONT_PT)
  ax.xaxis.labelpad = FIG_XLABELPAD
  ax.yaxis.labelpad = FIG_YLABELPAD
  ax.tick_params(labelsize=FIG_FONT_PT)
  for txt in ax.texts:
    txt.set_fontsize(FIG_FONT_PT)

add_panel_labels(fig, [
  ("A", ax_seg_s),
  ("B", ax_traj_s),
  ("C", ax_burst_rate),
  ("D", ax_effective_mode),
  ("E", ax_dyn_var, -0.5),
  ("F", ax_cnmf_similarity, -0.5),
  ("G", ax_hdbscan_score, -0.5),
])

(main_path / "analysis/result").mkdir(parents=True, exist_ok=True)
fig.savefig(main_path / "analysis/result/s_cond.pdf", format="pdf", bbox_inches="tight")

plt.show()


In [ ]:
# fig:proj-order

# purpose-local frame; the shared plot_df is left for fig:adjust-mode
ord_plot_df = _build_multi_plot_df(["flt_kendall"])

def _build_latency_kendall_df():
  row_s = []
  if "mean_spk_time" not in mv_smry_s:
    return pd.DataFrame(columns=["mv_name", "tsu_type", "cond", "value"])

  for mv_name in sorted(mv_smry_s["mean_spk_time"].keys()):
    mean_v = np.asarray(mv_smry_s["mean_spk_time"].get(mv_name), dtype=float)
    flt_mean_v = mv_smry_s.get("flt_mean_spk_time", {}).get(mv_name)

    if flt_mean_v is None:
      flt_all = mv_smry_s.get("flt_all_spk_time", {}).get(mv_name)
      brst_neur = mv_smry_s.get("brst_neur", {}).get(mv_name)
      if (flt_all is None) or (brst_neur is None):
        continue
      flt_all = np.asarray(flt_all, dtype=float)
      brst_neur = np.asarray(brst_neur, dtype=bool)
      flt_mean_v = np.full(mean_v.shape, np.nan)
      if flt_all.ndim == 2 and brst_neur.shape[0] == mean_v.shape[0]:
        flt_mean_v[brst_neur] = np.nanmean(flt_all[:, brst_neur], axis=0)

    flt_mean_v = np.asarray(flt_mean_v, dtype=float)
    if mean_v.shape != flt_mean_v.shape:
      continue

    finite_m = np.isfinite(mean_v) & np.isfinite(flt_mean_v)
    if np.sum(finite_m) < 3:
      continue

    tau, p_val = sp.stats.kendalltau(mean_v[finite_m], flt_mean_v[finite_m])
    row_s.append({
      "mv_name": mv_name,
      "tsu_type": TSU_TYPE_MAP.get(mv_smry_s.get("tsu_type", {}).get(mv_name), mv_smry_s.get("tsu_type", {}).get(mv_name)),
      "cond": COND_MAP.get(mv_smry_s.get("cond", {}).get(mv_name), mv_smry_s.get("cond", {}).get(mv_name)),
      "value": float(tau),
      "p_val": float(p_val),
      "n": int(np.sum(finite_m)),
    })

  out_df = pd.DataFrame(row_s)
  if out_df.empty:
    return pd.DataFrame(columns=["mv_name", "tsu_type", "cond", "value", "p_val", "n"])
  out_df = out_df.replace([np.inf, -np.inf], np.nan).dropna(subset=["tsu_type", "cond", "value"]).copy()
  out_df = out_df[out_df["tsu_type"].isin(ORDER) & out_df["cond"].isin(HUE_ORDER)].copy()
  out_df["tsu_type"] = pd.Categorical(out_df["tsu_type"], categories=ORDER, ordered=True)
  out_df["cond"] = pd.Categorical(out_df["cond"], categories=HUE_ORDER, ordered=True)
  return out_df.reset_index(drop=True)


lat_kendall_df = _build_latency_kendall_df()
if lat_kendall_df.empty:
  raise ValueError("No movies with paired mean_spk_time and filtered mean spike time for latency Kendall plot")


fig = plt.figure(figsize=(6, 1.80))

# Order-preservation summaries retained after the CNMF example moves to Figure 2A.
ax_ord = fig.add_axes([0.085, 0.130, 0.395, 0.860])
ax_stab = fig.add_axes([0.570, 0.130, 0.400, 0.860])
box_kw = {
  "boxprops": {"facecolor": "none", "edgecolor": "0.45", "linewidth": LW_MAIN},
  "whiskerprops": {"color": "k", "linewidth": LW_MAIN},
  "capprops": {"color": "k", "linewidth": LW_MAIN},
  "medianprops": {"color": "k", "linewidth": LW_MAIN},
}

# Paired CNMF versus denoised order stability.
pair_df = (
  ord_plot_df[["mv_name", "tsu_type", "cond", "flt_kendall", "kendall"]]
  .replace([np.inf, -np.inf], np.nan)
  .dropna()
  .drop_duplicates(subset=["mv_name"])
  .copy()
)

pair_order = ["CNMF", "denoised"]
pair_long_df = pd.concat([
  pair_df.rename(columns={"flt_kendall": "value"}).assign(metric="CNMF")[
    ["mv_name", "tsu_type", "cond", "metric", "value"]
  ],
  pair_df.rename(columns={"kendall": "value"}).assign(metric="denoised")[
    ["mv_name", "tsu_type", "cond", "metric", "value"]
  ],
], ignore_index=True)

sns.boxplot(
  data=pair_long_df,
  x="metric", y="value",
  order=pair_order,
  ax=ax_stab,
  **box_kw,
)
# draw paired connectors under markers
x_sm = float(pair_order.index("CNMF"))
x_pr = float(pair_order.index("denoised"))
for y_sm, y_pr in zip(
  pair_df["flt_kendall"].to_numpy(dtype=float),
  pair_df["kendall"].to_numpy(dtype=float),
):
  ax_stab.plot(
    [x_sm, x_pr], [y_sm, y_pr],
    color="0.6", lw=LW_SECONDARY, alpha=0.6, zorder=0.5,
  )
for tsu_type in ORDER:
  for cond in HUE_ORDER:
    one_df = pair_long_df[(pair_long_df["tsu_type"] == tsu_type) & (pair_long_df["cond"] == cond)]
    if one_df.empty:
      continue
    sns.stripplot(
      data=one_df,
      x="metric", y="value",
      order=pair_order,
      jitter=False,
      alpha=ALPHA_MAIN,
      color=COND_COLOR.get(str(cond), "0.4"),
      marker=TSU_MARKER.get(str(tsu_type), "o"),
      size=S_STRIP,
      linewidth=0,
      ax=ax_stab,
    )
if ax_stab.get_legend() is not None:
  ax_stab.get_legend().remove()

pair_t = np.nan
pair_p = np.nan
CNMF_v = pair_df["flt_kendall"].to_numpy(dtype=float)
denoised_v = pair_df["kendall"].to_numpy(dtype=float)
if len(pair_df) >= 2:
  tt = sp.stats.ttest_rel(CNMF_v, denoised_v, nan_policy="omit")
  pair_t = float(tt.statistic)
  pair_p = float(tt.pvalue)

pair_tbl = pd.DataFrame([
  {
    "comparison": "CNMF vs denoised",
    "T": pair_t,
    "p_unc": pair_p,
  }
])
print("\norder_stability CNMF vs denoised paired t-test:")
print(pair_tbl[["comparison", "T", "p_unc"]])

annotate_box_stars(
  ax_stab,
  [("CNMF", "denoised")],
  {("CNMF", "denoised"): pair_p},
  order=pair_order,
)
ax_stab.set_xlabel("")
ax_stab.set_ylabel(r"$\mathrm{order\ stability}$")
ax_stab.set_yticks(np.arange(0.0, 1.01, 0.2))
ax_stab.set_ylim(-0.06, 1.3)
ax_stab.set_xticks(np.arange(len(pair_order)))
ax_stab.set_xticklabels([r"$\mathrm{CNMF}$", r"$\mathrm{denoised}$"])

sns.boxplot(
  data=lat_kendall_df,
  x="tsu_type",
  y="value",
  order=ORDER,
  ax=ax_ord,
  **box_kw,
)
for tsu_type in ORDER:
  for cond in HUE_ORDER:
    one_df = lat_kendall_df[
      (lat_kendall_df["tsu_type"].astype(str) == tsu_type)
      & (lat_kendall_df["cond"].astype(str) == cond)
    ]
    if one_df.empty:
      continue
    sns.stripplot(
      data=one_df,
      x="tsu_type",
      y="value",
      order=ORDER,
      jitter=False,
      alpha=ALPHA_MAIN,
      color=COND_COLOR.get(str(cond), "0.4"),
      marker=TSU_MARKER.get(str(tsu_type), "o"),
      size=S_STRIP,
      linewidth=0,
      ax=ax_ord,
    )
if ax_ord.get_legend() is not None:
  ax_ord.get_legend().remove()

lat_kendall_pair_s = [("org", "cab")]
p_map = holm_pairwise_pvals(lat_kendall_df, "tsu_type", "value", lat_kendall_pair_s)
print_pairwise_annotation_table(
  lat_kendall_df, "tsu_type", "value", lat_kendall_pair_s, p_map,
  "figure:projection latency-order Kendall annotation Welch t-test",
)
annotate_box_stars(ax_ord, lat_kendall_pair_s, p_map, order=ORDER)

ax_ord.hlines(0.0, *ax_ord.get_xlim(), colors="0.5", linestyles="--", linewidth=LW_REFERENCE)
ax_ord.set_xlabel("")
ax_ord.set_ylabel(r"$\mathrm{CNMF\ vs\ denoised\ similarity}$")
ax_ord.set_xticks(np.arange(len(ORDER)))
ax_ord.set_xticklabels([r"$\mathrm{BTO}$", r"$\mathrm{CCN}$"])
ax_ord.set_ylim(-0.05, 1.05)

print("\nKendall tau between mean_spk_time and flt_mean_spk_time:")
print(lat_kendall_df[["mv_name", "tsu_type", "cond", "n", "value", "p_val"]])


for _ax in fig.axes:
  _ax.tick_params(labelsize=FIG_FONT_PT)
  _ax.xaxis.label.set_size(FIG_FONT_PT)
  _ax.yaxis.label.set_size(FIG_FONT_PT)
  for _txt in _ax.texts:
    _txt.set_fontsize(FIG_FONT_PT)

add_panel_labels(fig, [
  ("A", ax_ord, 0.0, 0.25),
  ("B", ax_stab, 0.0, 0.55),
])

(main_path / "analysis/result").mkdir(parents=True, exist_ok=True)
fig.savefig(main_path / "analysis/result/s_proj_order.pdf", format="pdf", bbox_inches="tight")

plt.show()


In [ ]:
# fig:consistent-dyn-var

consistent_bi_modal_df = (
  _build_multi_plot_df(["stb_avg_dyn_var"])[["mv_name", "tsu_type", "cond", "stb_avg_dyn_var"]]
  .replace([np.inf, -np.inf], np.nan)
  .dropna(subset=["tsu_type", "cond", "stb_avg_dyn_var"])
  .rename(columns={"stb_avg_dyn_var": "value"})
  .copy()
)

if consistent_bi_modal_df.empty:
  raise ValueError("No multi-bursting movies with consistent dynamical variance ratio for figure:consistent-dyn-var")

value_arr = consistent_bi_modal_df["value"].to_numpy(dtype=float)
value_arr = value_arr[np.isfinite(value_arr)]
if value_arr.size == 0:
  raise ValueError("No finite consistent dynamical variance ratio values for figure:consistent-dyn-var")

fig, ax = plt.subplots(1, 1, figsize=(2.4, 2.0), constrained_layout=True)

counts, bins, _ = ax.hist(
  value_arr,
  bins="auto",
  color="0.82",
  edgecolor="0.35",
  linewidth=LW_MAIN,
)
y_max = max(float(np.nanmax(counts)) if len(counts) else 1.0, 1.0)
rng = np.random.default_rng(0)

for tsu_type in ORDER:
  for cond in HUE_ORDER:
    one_df = consistent_bi_modal_df[
      (consistent_bi_modal_df["tsu_type"].astype(str) == tsu_type)
      & (consistent_bi_modal_df["cond"].astype(str) == cond)
    ]
    if one_df.empty:
      continue
    y_jitter = rng.uniform(0.06, 0.16, size=len(one_df)) * y_max
    ax.scatter(
      one_df["value"],
      y_jitter,
      alpha=ALPHA_MAIN,
      color=COND_COLOR.get(str(cond), "0.4"),
      marker=TSU_MARKER.get(str(tsu_type), "o"),
      s=S_STRIP ** 2,
      linewidths=0,
      zorder=3,
    )

def _run_combined_gmm_bic_test(value_s, random_state=0):
  x = np.asarray(value_s, dtype=float)
  x = x[np.isfinite(x)]
  out = {
    "group": "combined",
    "n": int(x.size),
    "gmm_bic_1": np.nan,
    "gmm_bic_2": np.nan,
    "gmm_delta_bic_1_minus_2": np.nan,
    "gmm_preferred": "",
  }

  if x.size < 3:
    out["gmm_preferred"] = "skipped: n<3"
    return out

  try:
    from sklearn.mixture import GaussianMixture
    x_col = x.reshape(-1, 1)
    gmm_1 = GaussianMixture(n_components=1, covariance_type="full", n_init=20, random_state=random_state).fit(x_col)
    gmm_2 = GaussianMixture(n_components=2, covariance_type="full", n_init=20, random_state=random_state).fit(x_col)
    bic_1 = float(gmm_1.bic(x_col))
    bic_2 = float(gmm_2.bic(x_col))
    out["gmm_bic_1"] = bic_1
    out["gmm_bic_2"] = bic_2
    out["gmm_delta_bic_1_minus_2"] = bic_1 - bic_2
    out["gmm_preferred"] = "2 components" if bic_2 < bic_1 else "1 component"
  except Exception as exc:
    out["gmm_preferred"] = f"skipped: {type(exc).__name__}"

  return out

bimodal_test_df = pd.DataFrame([_run_combined_gmm_bic_test(value_arr)])
print("\nGaussian mixture BIC comparison for figure:consistent-dyn-var consistent dynamical variance ratio:")
print(bimodal_test_df[["group", "n", "gmm_bic_1", "gmm_bic_2", "gmm_delta_bic_1_minus_2", "gmm_preferred"]])

ax.set_xlabel(r"$\mathrm{consistent\ dynamical\ var.\ ratio}$")
ax.set_ylabel(r"$\mathrm{count}$")
ax.set_ylim(0, y_max * 1.12)

for _ax in fig.axes:
  _ax.xaxis.label.set_size(FIG_FONT_PT)
  _ax.yaxis.label.set_size(FIG_FONT_PT)
  _ax.xaxis.labelpad = FIG_XLABELPAD
  _ax.yaxis.labelpad = FIG_YLABELPAD
  _ax.title.set_size(FIG_FONT_PT)
  _ax.tick_params(labelsize=FIG_FONT_PT)
  for _txt in _ax.texts:
    _txt.set_fontsize(FIG_FONT_PT)


(main_path / "analysis/result").mkdir(parents=True, exist_ok=True)
fig.savefig(main_path / "analysis/result/s_consistent_dyn_var.pdf", format="pdf", bbox_inches="tight")

plt.show()


In [ ]:
# fig:adjust-mode

# the *_sig_mode_frac columns are derived, not returned by _build_multi_plot_df,
# so rebuild them here rather than relying on which cell last bound plot_df
plot_df = _build_multi_plot_df()
for _part_key, _frac_key in [
  ("oth_sig_mode_n", "oth_sig_mode_frac"),
  ("itrm_sig_mode_n", "itrm_sig_mode_frac"),
  ("stb_sig_mode_n", "stb_sig_mode_frac"),
]:
  plot_df[_frac_key] = np.divide(
    plot_df[_part_key].to_numpy(dtype=float),
    plot_df["mode_n"].to_numpy(dtype=float),
    out=np.full(len(plot_df), np.nan, dtype=float),
    where=(plot_df["mode_n"].to_numpy(dtype=float) > 0),
  )

mode_count_palette = SOURCE_PALETTE


def _loading_breadth_rows(mv_name):
  mode_n_raw = float(mv_smry_s.get("mode_n", {}).get(mv_name, np.nan))
  if (not np.isfinite(mode_n_raw)) or (mode_n_raw < 1):
    return []
  mode_n = int(mode_n_raw)
  if mv_name not in mv_smry_s.get("stk_pc", {}):
    return []
  stk_pc_vec = np.asarray(mv_smry_s["stk_pc"][mv_name][1], dtype=float)
  stk_n = int(mv_smry_s.get("stk_n", {}).get(mv_name, 0))
  traj = np.asarray(mv_smry_s.get("traj", {}).get(mv_name, []), dtype=float)
  if traj.ndim != 2:
    return []
  neur_n = int(traj.shape[0])
  if (stk_n <= 0) or (neur_n <= 0) or (stk_n * neur_n != stk_pc_vec.shape[0]):
    return []

  mode_col_s = np.arange(stk_pc_vec.shape[1] - mode_n, stk_pc_vec.shape[1], dtype=int)
  stk_mode = stk_pc_vec[:, mode_col_s].reshape(stk_n, neur_n, mode_n)
  neur_mode_pwr = np.mean(stk_mode ** 2, axis=0) * float(neur_n * stk_n)
  mean_pwr = np.mean(neur_mode_pwr, axis=1)
  mean_sq_pwr = np.mean(neur_mode_pwr ** 2, axis=1)
  loading_breadth = np.divide(
    mean_pwr ** 2,
    mean_sq_pwr,
    out=np.full(neur_n, np.nan, dtype=float),
    where=np.isfinite(mean_sq_pwr) & (mean_sq_pwr > 0),
  )

  row_s = []
  for source_name, mask_key in [
    ("unrecruited", "oth_neur"),
    ("partial", "itrm_neur"),
    ("recruited", "stb_neur"),
  ]:
    mask = np.asarray(mv_smry_s.get(mask_key, {}).get(mv_name, []), dtype=bool)
    if (mask.size != neur_n) or (not np.any(mask)):
      continue
    value_s = loading_breadth[mask]
    value_s = value_s[np.isfinite(value_s)]
    if value_s.size == 0:
      continue
    row_s.append({
      "mv_name": mv_name,
      "tsu_type": TSU_TYPE_MAP.get(mv_smry_s["tsu_type"].get(mv_name), mv_smry_s["tsu_type"].get(mv_name)),
      "cond": COND_MAP.get(mv_smry_s["cond"].get(mv_name), mv_smry_s["cond"].get(mv_name)),
      "source": source_name,
      "value": float(np.mean(value_s)),
      "neuron_n": int(value_s.size),
      "mode_n": mode_n,
    })
  return row_s


loading_breadth_row_s = []
for _mv_name in sorted(mv_smry_s["mv_name"].keys()):
  if mv_smry_s["brst_type"].get(_mv_name) == "multi":
    loading_breadth_row_s.extend(_loading_breadth_rows(_mv_name))
loading_breadth_df = pd.DataFrame(loading_breadth_row_s)
if loading_breadth_df.empty:
  raise ValueError("No valid recordings for figure:adjust-mode loading-breadth panel")
loading_breadth_df["source"] = pd.Categorical(
  loading_breadth_df["source"], categories=source_order_part, ordered=True)

demo_mv_name = ADJUST_MODE_DEMO_MV
if demo_mv_name not in mv_smry_s.get("mv_name", {}):
  raise KeyError(f"{demo_mv_name} missing from mv_smry_s; run per-movie analysis first")

itrm_adj_spct = np.asarray(mv_smry_s["itrm_adj_spct"][demo_mv_name], dtype=float)
stb_adj_spct = np.asarray(mv_smry_s["stb_adj_spct"][demo_mv_name], dtype=float)
oth_adj_spct = np.asarray(mv_smry_s["oth_adj_spct"][demo_mv_name], dtype=float)
part_edge_hi = float(mv_smry_s["part_edge_hi"].get(demo_mv_name, np.nan))
demo_mode_n = int(np.asarray(itrm_adj_spct).shape[0])
orig_spct = np.asarray(mv_smry_s["stk_pc"][demo_mv_name][0], dtype=float)[-demo_mode_n:]

demo_stk_pc_vec = np.asarray(mv_smry_s["stk_pc"][demo_mv_name][1], dtype=float)
demo_stk_n = int(mv_smry_s["stk_n"][demo_mv_name])
demo_neur_n = int(np.asarray(mv_smry_s["traj"][demo_mv_name], dtype=float).shape[0])
demo_itrm_neur = np.asarray(mv_smry_s["itrm_neur"][demo_mv_name], dtype=bool)
demo_stb_neur = np.asarray(mv_smry_s["stb_neur"][demo_mv_name], dtype=bool)
demo_oth_neur = np.asarray(mv_smry_s["oth_neur"][demo_mv_name], dtype=bool)
demo_part_n_s = {
  "unrecruited": int(np.sum(demo_oth_neur)),
  "partial": int(np.sum(demo_itrm_neur)),
  "recruited": int(np.sum(demo_stb_neur)),
}
if any(part_n == 0 for part_n in demo_part_n_s.values()):
  raise ValueError(f"figure:adjust-mode demo movie {demo_mv_name} is missing partition(s): {demo_part_n_s}")

top_mode_col_n = max(1, int(demo_mode_n))
fig = plt.figure(figsize=(6, 5.2), constrained_layout=True)
outer_gs = fig.add_gridspec(3, 1, height_ratios=[1, 1, 1])
mode_gs = outer_gs[0, 0].subgridspec(
  1, top_mode_col_n + 1, width_ratios=np.r_[np.ones(top_mode_col_n), 1.20])
ax_mode_s = []
for mode_plot_idx in range(top_mode_col_n):
  sharey_ax = ax_mode_s[0] if ax_mode_s else None
  ax_mode_s.append(fig.add_subplot(mode_gs[0, mode_plot_idx], sharey=sharey_ax))
ax_loading_breadth = fig.add_subplot(mode_gs[0, top_mode_col_n])

row2_gs = outer_gs[1, 0].subgridspec(1, 2, width_ratios=[2, 1])
row3_gs = outer_gs[2, 0].subgridspec(1, 2, width_ratios=[1, 1])
ax_spct = fig.add_subplot(row2_gs[0, 0])
ax_sig_mode = fig.add_subplot(row2_gs[0, 1])
ax_no_burst_mode = fig.add_subplot(row3_gs[0, 0])
ax_no_burst_dyn = fig.add_subplot(row3_gs[0, 1])

# Mode-wise neuron representation (left to right: largest modes first).
if demo_mode_n > 0 and (demo_stk_n * demo_neur_n == demo_stk_pc_vec.shape[0]):
  mode_col_s = np.arange(demo_stk_pc_vec.shape[1] - demo_mode_n, demo_stk_pc_vec.shape[1], dtype=int)
  stk_mode_demo = demo_stk_pc_vec[:, mode_col_s].reshape(demo_stk_n, demo_neur_n, demo_mode_n)
  neur_mode_pwr = np.mean(stk_mode_demo ** 2, axis=0) * float(demo_neur_n * demo_stk_n)

  show_mode_n = min(len(ax_mode_s), demo_mode_n)
  mode_idx_s = [demo_mode_n - 1 - i for i in range(show_mode_n)]
  label_plot_idx = (len(ax_mode_s) - 1) // 2

  for plot_idx, ax_mode in enumerate(ax_mode_s):
    ax_mode.set_title(rf"$\mathrm{{mode\ {plot_idx + 1}}}$")
    ax_mode.set_xticks([0, 25, 50])
    ax_mode.set_xlabel(r"$\mathrm{sorted\ neuron}$" if plot_idx == label_plot_idx else "")
    ax_mode.set_ylabel((r"$\mathrm{normalized\ squared}$" + "\n" + r"$\mathrm{loading}$") if plot_idx == 0 else "")
    if plot_idx != 0:
      ax_mode.tick_params(labelleft=False)

    if plot_idx >= show_mode_n:
      ax_mode.text(0.5, 0.5, r"$\mathrm{no\ mode}$", ha="center", va="center", transform=ax_mode.transAxes)
      continue

    mode_idx0 = mode_idx_s[plot_idx]
    ord_idx = np.argsort(neur_mode_pwr[:, mode_idx0])
    pwr_ord = neur_mode_pwr[ord_idx, mode_idx0]
    x_ord = np.arange(demo_neur_n)

    oth_ord = demo_oth_neur[ord_idx]
    stb_ord = demo_stb_neur[ord_idx]
    itrm_ord = demo_itrm_neur[ord_idx]

    ax_mode.scatter(x_ord[oth_ord], pwr_ord[oth_ord], s=0.6 * S_SCATTER_SMALL, color=PART_COLOR["unrecruited"], alpha=ALPHA_MAIN, linewidths=0, zorder=1)
    ax_mode.scatter(x_ord[itrm_ord], pwr_ord[itrm_ord], s=0.6 * S_SCATTER_SMALL, color=PART_COLOR["partial"], alpha=ALPHA_MAIN, linewidths=0, zorder=2)
    ax_mode.scatter(x_ord[stb_ord], pwr_ord[stb_ord], s=0.6 * S_SCATTER_SMALL, color=PART_COLOR["recruited"], alpha=ALPHA_MAIN, linewidths=0, zorder=3)

    mean_line_s = [
      (np.ones(demo_neur_n, dtype=bool), "k"),
      (demo_oth_neur, PART_COLOR["unrecruited"]),
      (demo_itrm_neur, PART_COLOR["partial"]),
      (demo_stb_neur, PART_COLOR["recruited"]),
    ]
    for mask, line_color in mean_line_s:
      mask = np.asarray(mask, dtype=bool)
      if mask.size != demo_neur_n or np.sum(mask) == 0:
        continue
      one_mean = float(np.mean(neur_mode_pwr[mask, mode_idx0]))
      if np.isfinite(one_mean):
        ax_mode.axhline(one_mean, color=line_color, ls="--", lw=LW_REFERENCE, alpha=ALPHA_MAIN)

# Per-neuron loading breadth across significant modes, averaged within each
# participation type so every point is one recording. The normalized
# participation ratio is 1 for equal loading across modes and approaches
# 1 / mode_n as loading concentrates in one mode.
sns.boxplot(
  data=loading_breadth_df, x="source", y="value",
  order=source_order_part, hue="source", palette=SOURCE_PALETTE,
  dodge=False, legend=False, boxprops={"alpha": ALPHA_BOX},
  ax=ax_loading_breadth,
)
breadth_pivot = loading_breadth_df.pivot(
  index="mv_name", columns="source", values="value").dropna(
  subset=source_order_part)
for _, one_row in breadth_pivot.iterrows():
  ax_loading_breadth.plot(
    np.arange(len(source_order_part)),
    [float(one_row[source]) for source in source_order_part],
    color="0.75", lw=LW_SECONDARY, alpha=ALPHA_BACKGROUND, zorder=0.5,
  )
sns.stripplot(
  data=loading_breadth_df, x="source", y="value",
  order=source_order_part, hue="source", palette=SOURCE_PALETTE,
  dodge=False, jitter=False, alpha=ALPHA_MAIN, size=S_STRIP, linewidth=0,
  ax=ax_loading_breadth,
)
if ax_loading_breadth.get_legend() is not None:
  ax_loading_breadth.get_legend().remove()

breadth_pair_s = [
  (source_order_part[i], source_order_part[j])
  for i in range(len(source_order_part))
  for j in range(i + 1, len(source_order_part))
]
breadth_test_row_s = []
breadth_p_unc_s = []
for source_1, source_2 in breadth_pair_s:
  if breadth_pivot.shape[0] >= 2:
    one_test = sp.stats.ttest_rel(
      breadth_pivot[source_1].to_numpy(dtype=float),
      breadth_pivot[source_2].to_numpy(dtype=float),
      nan_policy="omit",
    )
    t_stat = float(one_test.statistic)
    p_unc = float(one_test.pvalue)
  else:
    t_stat = p_unc = np.nan
  breadth_p_unc_s.append(p_unc)
  breadth_test_row_s.append({
    "comparison": f"{source_1} vs {source_2}",
    "n": int(breadth_pivot.shape[0]),
    "T": t_stat,
    "p_unc": p_unc,
  })
breadth_p_unc_a = np.asarray(breadth_p_unc_s, dtype=float)
breadth_p_holm_a = np.full_like(breadth_p_unc_a, np.nan)
breadth_valid = np.isfinite(breadth_p_unc_a)
if np.any(breadth_valid):
  breadth_p_holm_a[breadth_valid] = multipletests(
    breadth_p_unc_a[breadth_valid], method="holm")[1]
breadth_p_map = {}
for row_idx, ((source_1, source_2), p_holm) in enumerate(
  zip(breadth_pair_s, breadth_p_holm_a)):
  breadth_test_row_s[row_idx]["p_holm"] = float(p_holm)
  breadth_p_map[(source_1, source_2)] = float(p_holm)
print("\nfigure:adjust-mode loading breadth group means (paired recordings):")
print(pd.DataFrame({
  "source": source_order_part,
  "n": [int(breadth_pivot.shape[0])] * len(source_order_part),
  "mean": [float(breadth_pivot[source].mean()) for source in source_order_part],
}))
print("\nfigure:adjust-mode loading breadth paired t-tests:")
print(pd.DataFrame(breadth_test_row_s)[
  ["comparison", "n", "T", "p_unc", "p_holm"]])
if breadth_pivot.shape[0] >= 2:
  breadth_friedman = sp.stats.friedmanchisquare(*[
    breadth_pivot[source].to_numpy(dtype=float) for source in source_order_part])
  print("\nfigure:adjust-mode loading breadth Friedman test:")
  print(pd.DataFrame([{
    "n": int(breadth_pivot.shape[0]),
    "chi2": float(breadth_friedman.statistic),
    "p_unc": float(breadth_friedman.pvalue),
  }]))

ax_loading_breadth.set_ylim(0.0, 1.0)
_annotate_partition_brackets(
  ax_loading_breadth, breadth_pair_s, breadth_p_map,
  source_order_part, level_s=[0, 1, 0],
)
ax_loading_breadth.set(
  xlabel="",
  ylabel=r"$\mathrm{loading\ breadth}$" + "\n" + r"$\mathrm{across\ modes}$",
)
ax_loading_breadth.set_yticks([0.0, 0.5, 1.0])
ax_loading_breadth.set_xticks(np.arange(len(source_order_part)))
ax_loading_breadth.set_xticklabels(
  [r"$\mathrm{non\mbox{-}part.}$", r"$\mathrm{intermittent}$", r"$\mathrm{consistent}$"],
  rotation=25, ha="right",
)

# Original + partition-adjusted spectra in Figure-1 style.
ax_spct.scatter(orig_spct, np.full_like(orig_spct, 3.0), s=S_SCATTER, color="k", alpha=ALPHA_MAIN, label=r"$\mathrm{original\ spectrum}$")
ax_spct.scatter(oth_adj_spct, np.full_like(oth_adj_spct, 2.0), s=S_SCATTER, color=PART_COLOR["unrecruited"], alpha=ALPHA_MAIN, label=r"$\mathrm{non-participating\ adjusted\ spectrum}$")
ax_spct.scatter(itrm_adj_spct, np.full_like(itrm_adj_spct, 1.0), s=S_SCATTER, color=PART_COLOR["partial"], alpha=ALPHA_MAIN, label=r"$\mathrm{intermittent\ adjusted\ spectrum}$")
ax_spct.scatter(stb_adj_spct, np.full_like(stb_adj_spct, 0.0), s=S_SCATTER, color=PART_COLOR["recruited"], alpha=ALPHA_MAIN, label=r"$\mathrm{consistent\ adjusted\ spectrum}$")
for mode_idx0 in range(demo_mode_n):
  x_mode = np.array([
    stb_adj_spct[mode_idx0],
    itrm_adj_spct[mode_idx0],
    oth_adj_spct[mode_idx0],
    orig_spct[mode_idx0],
  ], dtype=float)
  if np.all(np.isfinite(x_mode)):
    ax_spct.plot(x_mode, [0, 1, 2, 3], color="0.65", lw=LW_SECONDARY, alpha=ALPHA_BACKGROUND, zorder=0)
if np.isfinite(part_edge_hi):
  ax_spct.vlines(part_edge_hi, -0.35, 3.35, color="tab:orange", lw=LW_REFERENCE, ls="--", label=r"$\mathrm{numeric\ edge\ (high)}$")
all_spct_x = np.concatenate([
  np.asarray(orig_spct, dtype=float).ravel(),
  np.asarray(oth_adj_spct, dtype=float).ravel(),
  np.asarray(stb_adj_spct, dtype=float).ravel(),
  np.asarray(itrm_adj_spct, dtype=float).ravel(),
])
all_spct_x = all_spct_x[np.isfinite(all_spct_x)]
if all_spct_x.size > 0:
  x_lo, x_hi = float(np.min(all_spct_x)), float(np.max(all_spct_x))
  x_pad = 0.05 * (x_hi - x_lo) if x_hi > x_lo else 0.05 * max(abs(x_hi), 1.0)
  ax_spct.set_xlim(x_lo - x_pad, x_hi + x_pad)
ax_spct.set(
  xlabel=r"$\mathrm{adjusted\ mode\ variance}$",
  yticks=[0, 1, 2, 3],
  yticklabels=[r"$\mathrm{consistent}$", r"$\mathrm{intermittent}$", r"$\mathrm{non\mbox{-}part.}$", r"$\mathrm{original}$"],
)
y_pad = 0.4
ax_spct.set_ylim(0.0 - y_pad, 3.0 + y_pad)

# Adjusted-significance fraction by partition.
source_df = _build_part_source_df([
  ("unrecruited", "oth_sig_mode_frac"),
  ("partial", "itrm_sig_mode_frac"),
  ("recruited", "stb_sig_mode_frac"),
])
sns.boxplot(
  data=source_df,
  x="source",
  y="value",
  hue="source",
  order=source_order_part,
  palette=SOURCE_PALETTE,
  dodge=False,
  legend=False,
  boxprops={"alpha": ALPHA_BOX},
  ax=ax_sig_mode,
)
sns.stripplot(
  data=source_df,
  x="source",
  y="value",
  order=source_order_part,
  hue="source",
  palette=SOURCE_PALETTE,
  dodge=False,
  jitter=False,
  alpha=ALPHA_MAIN,
  size=S_STRIP,
  linewidth=0,
  ax=ax_sig_mode,
)
if ax_sig_mode.get_legend() is not None:
  ax_sig_mode.get_legend().remove()

pair_s = [
  (source_order_part[i], source_order_part[j])
  for i in range(len(source_order_part))
  for j in range(i + 1, len(source_order_part))
]
p_map = holm_pairwise_pvals(source_df, "source", "value", pair_s)
annotate_box_stars(ax_sig_mode, pair_s, p_map, order=source_order_part)

_anova_and_posthoc(source_df, "partition_sig_mode_frac")
_onesample_vs_zero_holm(source_df, "partition_sig_mode_frac", popmean=0.0)
_spread_nonparticipating_intermittent_vs_consistent(source_df, "partition_sig_mode_frac")

ax_sig_mode.set(
  xlabel="",
  ylabel=r"$\mathrm{adjusted\mbox{-}significance}$" + "\n" + r"$\mathrm{fraction}$",
)
ax_sig_mode.set_yticks([0, 0.5, 1.0])
ax_sig_mode.set_xticks(np.arange(len(source_order_part)))
ax_sig_mode.set_xticklabels(
  [r"$\mathrm{non\mbox{-}part.}$", r"$\mathrm{intermittent}$", r"$\mathrm{consistent}$"],
  rotation=25, ha="right",
)
ax_sig_mode.tick_params(labelsize=8)

mode_count_df = _build_no_burst_mode_count_source_df()
_plot_no_burst_no_burst_nonparticipating_panel(
  ax_no_burst_mode,
  mode_count_df,
  r"$\mathrm{adjusted\mbox{-}significance}$" + "\n" + r"$\mathrm{count}$",
  "no_burst_unrecruited_mode_n",
)

dyn_var_df = _build_no_burst_dyn_var_source_df()
_plot_no_burst_no_burst_nonparticipating_panel(
  ax_no_burst_dyn,
  dyn_var_df,
  r"$\mathrm{dynamical\ var.\ ratio}$",
  "no_burst_unrecruited_avg_dyn_var",
)

for _ax in fig.axes:
  _ax.xaxis.label.set_size(FIG_FONT_PT)
  _ax.yaxis.label.set_size(FIG_FONT_PT)
  _ax.xaxis.labelpad = FIG_XLABELPAD
  _ax.yaxis.labelpad = FIG_YLABELPAD
  _ax.title.set_size(FIG_FONT_PT)
  _ax.tick_params(labelsize=FIG_FONT_PT)
  for _txt in _ax.texts:
    _txt.set_fontsize(FIG_FONT_PT)
  _leg = _ax.get_legend()
  if _leg is not None:
    _leg.get_title().set_fontsize(FIG_FONT_PT)
    for _txt in _leg.get_texts():
      _txt.set_fontsize(FIG_FONT_PT)

add_panel_labels(fig, [
  ("A", ax_mode_s),
  ("B", ax_loading_breadth),
  ("C", ax_spct, -1.0, 0.5),
  ("D", ax_sig_mode),
  ("E", ax_no_burst_mode),
  ("F", ax_no_burst_dyn),
])

(main_path / "analysis/result").mkdir(parents=True, exist_ok=True)
fig.savefig(main_path / "analysis/result/s_adjust_mode.pdf", format="pdf", bbox_inches="tight")

plt.show()


In [ ]:
# fig:cluster-count

cluster_count_df = (
  _build_multi_plot_df(["clstr_n"])[["mv_name", "tsu_type", "cond", "clstr_n"]]
  .replace([np.inf, -np.inf], np.nan)
  .dropna(subset=["tsu_type", "cond", "clstr_n"])
  .copy()
)
cluster_count_df["clstr_n"] = cluster_count_df["clstr_n"].astype(float)
if cluster_count_df.empty:
  raise ValueError("No finite cluster-count rows for fig:cluster-count")

cluster_count_order = [grp for grp in ORDER if grp in set(cluster_count_df["tsu_type"].astype(str))]
if len(cluster_count_order) < 2:
  raise ValueError("Need both BTO and CCN movies for fig:cluster-count")

_cluster_count_box_kw = box_kw_nonhue if "box_kw_nonhue" in globals() else {
  "boxprops": {"facecolor": "none", "edgecolor": "0.45", "linewidth": LW_MAIN},
  "whiskerprops": {"color": "k", "linewidth": LW_MAIN},
  "capprops": {"color": "k", "linewidth": LW_MAIN},
  "medianprops": {"color": "k", "linewidth": LW_MAIN},
}

fig, ax_s = plt.subplots(1, 3, figsize=(5.4, 1.85), constrained_layout=True)
ax_bto_hist, ax_ccn_hist, ax_summary = ax_s
rng = np.random.default_rng(0)

all_count_v = cluster_count_df["clstr_n"].to_numpy(dtype=float)
all_count_v = all_count_v[np.isfinite(all_count_v)]
if all_count_v.size == 0:
  raise ValueError("No finite cluster-count values for fig:cluster-count")
bin_lo = np.floor(float(np.nanmin(all_count_v))) - 0.5
bin_hi = np.ceil(float(np.nanmax(all_count_v))) + 1.5
cluster_count_bins = np.arange(bin_lo, bin_hi, 1.0)
if cluster_count_bins.size < 2:
  cluster_count_bins = np.array([bin_lo, bin_lo + 1.0])


def _plot_cluster_count_hist(ax, tsu_type, title_text):
  one_tsu_df = cluster_count_df[cluster_count_df["tsu_type"].astype(str) == str(tsu_type)].copy()
  value_arr = one_tsu_df["clstr_n"].to_numpy(dtype=float)
  value_arr = value_arr[np.isfinite(value_arr)]
  if value_arr.size == 0:
    raise ValueError(f"No finite cluster-count values for {title_text}")
  counts, _, _ = ax.hist(
    value_arr,
    bins=cluster_count_bins,
    color="0.82",
    edgecolor="0.35",
    linewidth=LW_MAIN,
  )
  y_max = max(float(np.nanmax(counts)) if len(counts) else 1.0, 1.0)
  for cond in HUE_ORDER:
    cond_df = one_tsu_df[one_tsu_df["cond"].astype(str) == str(cond)]
    if cond_df.empty:
      continue
    y_jitter = rng.uniform(0.06, 0.16, size=len(cond_df)) * y_max
    ax.scatter(
      cond_df["clstr_n"].to_numpy(dtype=float),
      y_jitter,
      alpha=ALPHA_MAIN,
      color=COND_COLOR.get(str(cond), "0.4"),
      marker=TSU_MARKER.get(str(tsu_type), "o"),
      s=S_STRIP ** 2,
      linewidths=0,
      zorder=3,
    )
  ax.set_title(title_text, fontsize=FIG_FONT_PT)
  ax.set_xlabel(r"$\mathrm{cluster\ count}$")
  ax.set_ylabel(r"$\mathrm{count}$")
  ax.set_ylim(0, y_max * 1.12)
  ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))
  ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))


_plot_cluster_count_hist(ax_bto_hist, "org", r"$\mathrm{BTO}$")
_plot_cluster_count_hist(ax_ccn_hist, "cab", r"$\mathrm{CCN}$")

sns.boxplot(
  data=cluster_count_df,
  x="tsu_type",
  y="clstr_n",
  order=cluster_count_order,
  ax=ax_summary,
  **_cluster_count_box_kw,
)
for tsu_type in cluster_count_order:
  for cond in HUE_ORDER:
    one_df = cluster_count_df[
      (cluster_count_df["tsu_type"].astype(str) == str(tsu_type))
      & (cluster_count_df["cond"].astype(str) == str(cond))
    ]
    if one_df.empty:
      continue
    sns.stripplot(
      data=one_df,
      x="tsu_type",
      y="clstr_n",
      order=cluster_count_order,
      jitter=False,
      alpha=ALPHA_MAIN,
      color=COND_COLOR[str(cond)],
      marker=TSU_MARKER.get(str(tsu_type), "o"),
      size=S_STRIP,
      linewidth=0,
      ax=ax_summary,
    )

cluster_count_pair_s = [("org", "cab")]
cluster_count_p_map = holm_pairwise_pvals(cluster_count_df, "tsu_type", "clstr_n", cluster_count_pair_s)
annotate_box_stars(ax_summary, cluster_count_pair_s, cluster_count_p_map, order=cluster_count_order)
print_pairwise_annotation_table(
  cluster_count_df,
  "tsu_type",
  "clstr_n",
  cluster_count_pair_s,
  cluster_count_p_map,
  "figure:cluster-count cluster count annotation Welch t-test",
)

legend_handles = [
  Line2D([], [], linestyle="", marker="o", markersize=LEGEND_MARKER_SMALL, color=COND_COLOR["spon"], label=r"$\mathrm{spont.}$"),
  Line2D([], [], linestyle="", marker="o", markersize=LEGEND_MARKER_SMALL, color=COND_COLOR["stim"], label=r"$\mathrm{excited}$"),
]
ax_summary.legend(handles=legend_handles, loc="lower right", frameon=False)
ax_summary.set_xlabel("")
ax_summary.set_ylabel(r"$\mathrm{cluster\ count}$")
ax_summary.set_xticks(np.arange(len(cluster_count_order)))
ax_summary.set_xticklabels([
  r"$\mathrm{BTO}$" if grp == "org" else r"$\mathrm{CCN}$"
  for grp in cluster_count_order
])
ax_summary.yaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))

for _ax in fig.axes:
  _ax.xaxis.label.set_size(FIG_FONT_PT)
  _ax.yaxis.label.set_size(FIG_FONT_PT)
  _ax.xaxis.labelpad = FIG_XLABELPAD
  _ax.yaxis.labelpad = FIG_YLABELPAD
  _ax.title.set_size(FIG_FONT_PT)
  _ax.tick_params(labelsize=FIG_FONT_PT)

add_panel_labels(fig, [("A", ax_bto_hist, 0, 0.0), ("B", ax_ccn_hist, 0, 0.0), ("C", ax_summary, 0, 0.0)])

(main_path / "analysis/result").mkdir(parents=True, exist_ok=True)
fig.savefig(main_path / "analysis/result/s_cluster_count.pdf", format="pdf", bbox_inches="tight")

plt.show()

# Table II (tab:cond-anova) row: number of activity types.
# Reported in the manuscript but previously not backed by a printed ANOVA (V16).
print_tsu_cond_anova(
  cluster_count_df,
  "clstr_n",
  "ANOVA for table:cond-anova number of activity types by tissue x condition",
)


In [ ]:
# fig:cluster-centroid

centroid_row_s = []
for mv_name in sorted(mv_smry_s["mv_name"].keys()):
  if mv_smry_s["brst_type"].get(mv_name) != "multi":
    continue

  centroid_contribution = np.asarray(mv_smry_s.get("clstr_centroid_contribution", {}).get(mv_name, []), dtype=float)
  if centroid_contribution.ndim != 2 or centroid_contribution.size == 0:
    continue

  mode_n_val = int(centroid_contribution.shape[1])
  eig_val = np.asarray(mv_smry_s.get("stk_pc", {}).get(mv_name, [np.array([])])[0], dtype=float)
  mode_var = eig_val[-mode_n_val:] if eig_val.size >= mode_n_val else np.full(mode_n_val, np.nan)
  dyn_var = np.asarray(mv_smry_s.get("dyn_var", {}).get(mv_name, np.full(mode_n_val, np.nan)), dtype=float)

  if dyn_var.size != mode_n_val:
    dyn_var = np.full(mode_n_val, np.nan)

  for axis_idx in range(centroid_contribution.shape[0]):
    for mode_idx in range(mode_n_val):
      centroid_row_s.append({
        "mv_name": mv_name,
        "tsu_type": TSU_TYPE_MAP.get(mv_smry_s["tsu_type"].get(mv_name), mv_smry_s["tsu_type"].get(mv_name)),
        "cond": COND_MAP.get(mv_smry_s["cond"].get(mv_name), mv_smry_s["cond"].get(mv_name)),
        "axis_idx": axis_idx,
        "mode_idx": mode_idx,
        "contribution": float(centroid_contribution[axis_idx, mode_idx]),
        "mode_var": float(mode_var[mode_idx]) if mode_idx < mode_var.size else np.nan,
        "dyn_var": float(dyn_var[mode_idx]) if mode_idx < dyn_var.size else np.nan,
      })

centroid_fig_df = pd.DataFrame(centroid_row_s)
if centroid_fig_df.empty:
  raise ValueError("No valid centroid contribution rows for figure:cluster-centroid")

cluster_demo_mv = CLUSTER_DEMO_MV
if cluster_demo_mv not in mv_smry_s.get("mv_name", {}):
  raise KeyError(f"{cluster_demo_mv} missing from mv_smry_s; run per-movie analysis first")
cluster_demo_clstr_lbl = np.asarray(mv_smry_s["clstr_lbl"][cluster_demo_mv], dtype=int)
form_vec = np.asarray(mv_smry_s["stk_pc"][cluster_demo_mv][1], dtype=float)
form_mode_n = int(mv_smry_s["mode_n"][cluster_demo_mv])
form_stk_n = int(mv_smry_s["stk_n"][cluster_demo_mv])
form_neur_n = form_vec.shape[0] // form_stk_n
form_sign = np.asarray(mv_smry_s["mode_sign"][cluster_demo_mv], dtype=float)
form_dt = float(mv_smry_s["frame_dt"][cluster_demo_mv])
form_sep = int(mv_smry_s["frm_sep"][cluster_demo_mv])
form_lag_s = np.arange(form_stk_n, dtype=float) * form_sep * form_dt
form_mode = (
  form_vec[:, -form_mode_n:].reshape(form_stk_n, form_neur_n, form_mode_n)
  * form_sign[None, None, :])
form_prof = form_mode / np.maximum(
  np.sqrt((form_mode ** 2).mean(axis=0, keepdims=True)), 1e-12)
form_lbl_s = sorted(set(int(v) for v in cluster_demo_clstr_lbl if v >= 0))
form_color = _cluster_color_map(cluster_demo_clstr_lbl)

fig = plt.figure(figsize=(6, 4.0), constrained_layout=True)
centroid_outer_gs = fig.add_gridspec(2, 1, height_ratios=[0.85, 1.0])
form_gs = centroid_outer_gs[0, 0].subgridspec(1, form_mode_n, wspace=0.12)
ax_form_s = [fig.add_subplot(form_gs[0, k]) for k in range(form_mode_n)]
for k, ax_form in enumerate(ax_form_s):
  mode_idx = form_mode_n - 1 - k
  ax_form.axhline(0.0, color="0.86", lw=0.25 * LW_REFERENCE, zorder=0)
  for one_lbl in form_lbl_s:
    ax_form.plot(
      form_lag_s, form_prof[:, cluster_demo_clstr_lbl == one_lbl, mode_idx].mean(1),
      color=form_color.get(one_lbl, "0.4"), lw=LW_MAIN, zorder=2)
  ax_form.set_ylim(-1.6, 1.6)
  ax_form.set_xlim(float(form_lag_s[0]), float(form_lag_s[-1]))
  ax_form.set_title(rf"$\mathrm{{mode\ {k + 1}}}$", pad=2.0)
  if k == 0:
    ax_form.set_ylabel(r"$\mathrm{activity\ form}$")
  else:
    ax_form.set_yticks([])
  ax_form.set_xticks([0, float(form_lag_s[-1])])
  ax_form.set_xticklabels([r"$\mathrm{0}$", rf"$\mathrm{{{form_lag_s[-1]:.2f}}}$"])
ax_form_s[form_mode_n // 2].set_xlabel(r"$\mathrm{delay\ (s)}$")

centroid_bottom_gs = centroid_outer_gs[1, 0].subgridspec(1, 2)
ax_s = [
  fig.add_subplot(centroid_bottom_gs[0, 0]),
  fig.add_subplot(centroid_bottom_gs[0, 1]),
]
centroid_panel_s = [
  ("mode_var", r"$\mathrm{dynamical\ PC\ variance}$"),
  ("dyn_var", r"$\mathrm{dynamical\ variance\ ratio}$"),
]

corr_row_s = []
for ax, (x_key, x_label) in zip(ax_s, centroid_panel_s):
  panel_df = centroid_fig_df[["mv_name", "tsu_type", "cond", "axis_idx", x_key, "contribution"]].replace([np.inf, -np.inf], np.nan).dropna(subset=[x_key, "contribution"]).copy()
  if panel_df.empty:
    raise ValueError(f"No finite rows for figure:cluster-centroid {x_key}")

  for tsu_type in ORDER:
    for cond in HUE_ORDER:
      one_df = panel_df[(panel_df["tsu_type"].astype(str) == tsu_type) & (panel_df["cond"].astype(str) == cond)]
      if one_df.empty:
        continue
      ax.scatter(
        one_df[x_key].to_numpy(dtype=float),
        one_df["contribution"].to_numpy(dtype=float),
        marker=TSU_MARKER.get(str(tsu_type), "o"),
        color=COND_COLOR.get(str(cond), "0.4"),
        s=S_SCATTER_HIGHLIGHT,
        alpha=ALPHA_MAIN,
        linewidths=0,
        zorder=3,
      )

  x = panel_df[x_key].to_numpy(dtype=float)
  y = panel_df["contribution"].to_numpy(dtype=float)
  if len(panel_df) >= 3 and (not np.isclose(np.nanstd(x), 0)) and (not np.isclose(np.nanstd(y), 0)):
    r_val, p_val = sp.stats.pearsonr(x, y)
    r_val = float(r_val)
    p_val = float(p_val)
  else:
    r_val, p_val = np.nan, np.nan
  corr_row_s.append({"x": x_key, "n": int(len(panel_df)), "r": r_val, "p": p_val})

  ax.set_xlabel(x_label)
  ax.set_ylabel(r"$\mathrm{centroid\ axis\ contribution}$")
  ax.set_ylim(-0.05, 1.2)
  ax.set_yticks(np.arange(0, 1.01, 0.2))
  ax.text(
    0.04, 0.96,
    format_corr_r_stars(r_val, p_val) if np.isfinite(r_val) and np.isfinite(p_val) else r"$r=\mathrm{nan}$",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=FIG_FONT_PT,
    bbox={"facecolor": "none", "edgecolor": "none", "alpha": 0.0},
  )

print("\\nfigure:cluster-centroid pooled mode contribution correlations:")
print(pd.DataFrame(corr_row_s)[["x", "n", "r", "p"]])

for _ax in fig.axes:
  _ax.xaxis.label.set_size(FIG_FONT_PT)
  _ax.yaxis.label.set_size(FIG_FONT_PT)
  _ax.xaxis.labelpad = FIG_XLABELPAD
  _ax.yaxis.labelpad = FIG_YLABELPAD
  _ax.tick_params(labelsize=FIG_FONT_PT)
  for _txt in _ax.texts:
    _txt.set_fontsize(FIG_FONT_PT)

add_panel_labels(fig, [
  ("A", ax_form_s, 0.5, 0.0),
  ("B", ax_s[0]),
  ("C", ax_s[1]),
])

(main_path / "analysis/result").mkdir(parents=True, exist_ok=True)
fig.savefig(main_path / "analysis/result/s_cluster_centroid.pdf", format="pdf", bbox_inches="tight")

plt.show()



# Appendix table


In [ ]:
# Appendix Table: within- and between-preparation variation
# The preparation assignments were collected manually from the recording names.
# source_record identifies the corresponding local burst-detection record used to
# confirm that every listed recording belongs to the multi-burst analysis set.
replicate_record_s = [
  ("org-11-1-bic",     "org-11-1", "BTO", "excited"),
  ("org-11-2-bic",     "org-11-2", "BTO", "excited"),
  ("org-11-3-bic",     "org-11-3", "BTO", "excited"),
  ("org-13-3-bic",     "org-13-3", "BTO", "excited"),
  ("org-13-3-spon",    "org-13-3", "BTO", "spontaneous"),
  ("org-13-4-spon",    "org-13-4", "BTO", "spontaneous"),
  ("org-13-6-bic",     "org-13-6", "BTO", "excited"),
  ("org-13-6-spon",    "org-13-6", "BTO", "spontaneous"),
  ("org-13-6-spon-am", "org-13-6", "BTO", "spontaneous"),
  ("cab-2-1-spon",     "cab-2-1",  "CCN", "spontaneous"),
  ("cab-2-1-stim",     "cab-2-1",  "CCN", "excited"),
  ("cab-4-1-spon",     "cab-4-1",  "CCN", "spontaneous"),
  ("cab-4-1-stim",     "cab-4-1",  "CCN", "excited"),
  ("cab-6-1-spon",     "cab-6-1",  "CCN", "spontaneous"),
  ("cab-6-1-stim",     "cab-6-1",  "CCN", "excited"),
  ("cab-7-1-spon",     "cab-7-1",  "CCN", "spontaneous"),
  ("cab-7-1-stim",     "cab-7-1",  "CCN", "excited"),
]

replicate_summary_path = main_path / "analysis/result/mv_smry_s.pkl"
with open(replicate_summary_path, "rb") as f:
  replicate_mv_smry_s = pickle.load(f)

replicate_df = pd.DataFrame(
  replicate_record_s,
  columns=["mv_name", "preparation", "cytoarchitecture", "condition"])
replicate_df["source_record"] = replicate_df["mv_name"].map(
  lambda name: main_path / f"analysis/result/multi/01-wins-{name}.pdf")

missing_source_s = [str(path) for path in replicate_df["source_record"] if not path.exists()]
if missing_source_s:
  raise FileNotFoundError("Missing local source record(s): " + ", ".join(missing_source_s))
if set(replicate_df["mv_name"]) != {
    name for name, kind in replicate_mv_smry_s["brst_type"].items() if kind == "multi"}:
  raise ValueError("Manual replicate entries do not match the multi-burst entries in mv_smry_s.pkl")

replicate_df["effective dimensionality"] = [
  replicate_mv_smry_s["mode_n"][name] * replicate_mv_smry_s["mode_pr"][name]
  for name in replicate_df["mv_name"]
]
replicate_df["dynamical variance ratio"] = [
  replicate_mv_smry_s["avg_dyn_var"][name] for name in replicate_df["mv_name"]
]
replicate_df["activity-form DBCV"] = [
  replicate_mv_smry_s["clstr_scr"][name] for name in replicate_df["mv_name"]
]

def _replicate_table_row(one_df, value_col):
  value_by_preparation = [
    group[value_col].dropna().to_numpy(dtype=float)
    for _, group in one_df.groupby("preparation")
  ]
  repeated_variance_s = [np.var(value_s, ddof=1) for value_s in value_by_preparation if value_s.size >= 2]
  within_sd = np.sqrt(np.mean(repeated_variance_s))
  preparation_mean_s = np.asarray([np.mean(value_s) for value_s in value_by_preparation])
  f_stat, p_value = sp.stats.f_oneway(*value_by_preparation)
  return within_sd, np.std(preparation_mean_s, ddof=1), f_stat, p_value

quantity_s = ["effective dimensionality", "dynamical variance ratio", "activity-form DBCV"]
replicate_table_s = []
for quantity in quantity_s:
  for cytoarchitecture in ["BTO", "CCN"]:
    one_df = replicate_df[replicate_df["cytoarchitecture"] == cytoarchitecture]
    within_sd, between_sd, f_stat, p_value = _replicate_table_row(one_df, quantity)
    replicate_table_s.append({
      "quantity": quantity,
      "cytoarchitecture": cytoarchitecture,
      "within-preparation SD": within_sd,
      "between-preparation SD": between_sd,
      "F": f_stat,
      "p": p_value,
    })
replicate_table_df = pd.DataFrame(replicate_table_s)

ccn_pair_df = replicate_df[replicate_df["cytoarchitecture"] == "CCN"]
ccn_corr_s = []
for quantity in quantity_s:
  pair_df = ccn_pair_df.pivot(index="preparation", columns="condition", values=quantity)
  corr_r, corr_p = sp.stats.pearsonr(pair_df["spontaneous"], pair_df["excited"])
  ccn_corr_s.append({"quantity": quantity, "r": corr_r, "p": corr_p})
ccn_corr_df = pd.DataFrame(ccn_corr_s)

print("Appendix replicate-variation table (source: mv_smry_s.pkl):")
print(replicate_table_df)
print("\nCCN spontaneous-excited correlations:")
print(ccn_corr_df)
print("\nManual recording/preparation entries and local source records:")
print(replicate_df[["mv_name", "preparation", "cytoarchitecture", "condition", "source_record"]])


In [ ]:
# Appendix Table II: condition x cytoarchitecture ANOVA
# This cell assembles the analyses used above into the manuscript table.

condition_table_recording_df = _build_multi_plot_df([
  "mode_n", "mode_pr", "avg_dyn_var", "prop_fund_freq",
  "brst_neur_frac", "flt_kendall", "clstr_scr", "clstr_n",
  "x_clstr_scr", "x_t_cor",
])
condition_table_recording_df["effective_dimensionality"] = (
  condition_table_recording_df["mode_n"].to_numpy(dtype=float)
  * condition_table_recording_df["mode_pr"].to_numpy(dtype=float)
)

def _condition_table_row(quantity, data, dv):
  anova_df = _factorial_anova_df(
    data, dv, {"tsu_type": ORDER, "cond": HUE_ORDER})
  missing_s = _missing_factorial_cells(
    anova_df, {"tsu_type": ORDER, "cond": HUE_ORDER})
  if missing_s:
    raise ValueError(
      f"{quantity}: missing factorial cell(s): " + "; ".join(missing_s))
  anova_tbl = anova_with_holm(
    anova_df, dv=dv, between=["tsu_type", "cond"])
  p_by_term = anova_tbl.set_index("Source")["p_unc"]
  return {
    "quantity": quantity,
    "condition": float(p_by_term["cond"]),
    "cytoarchitecture": float(p_by_term["tsu_type"]),
    "interaction": float(p_by_term["tsu_type * cond"]),
  }

# Every row uses one value per recording except within-type gain, for which
# each activity type is an observation, as described in the manuscript.
condition_table_spec_s = [
  ("effective dimensionality", condition_table_recording_df, "effective_dimensionality"),
  ("significant dynamical PC variance fraction", oob_var_frac_df, "value"),
  ("dynamical variance ratio", condition_table_recording_df, "avg_dyn_var"),
  ("fundamental frequency", condition_table_recording_df, "prop_fund_freq"),
  ("sequence participation fraction", condition_table_recording_df, "brst_neur_frac"),
  ("order stability", condition_table_recording_df, "flt_kendall"),
  ("activity-form DBCV", condition_table_recording_df, "clstr_scr"),
  ("number of activity types", cluster_count_df, "clstr_n"),
  ("physical DBCV", x_clstr_scr_tsu_df, "x_clstr_scr"),
  ("within-type gain in distance--latency correlation", x_delta_group_df, "x_t_cor_delta"),
  ("global distance--latency correlation", condition_table_recording_df, "x_t_cor"),
]
condition_anova_table_df = pd.DataFrame([
  _condition_table_row(quantity, data, dv)
  for quantity, data, dv in condition_table_spec_s
])

print("Appendix condition x cytoarchitecture ANOVA table:")
print(condition_anova_table_df.to_string(index=False))


# Unused supplementary figures


In [ ]:
# fig:frac-part-ei

frac_part_ei_df = (
  _build_multi_plot_df(["brst_neur_frac"])[["mv_name", "tsu", "tsu_type", "cond", "brst_neur_frac"]]
  .replace([np.inf, -np.inf], np.nan)
  .dropna(subset=["tsu", "tsu_type", "cond", "brst_neur_frac"])
  .copy()
)

frac_part_ei_df = frac_part_ei_df[frac_part_ei_df["cond"].isin(HUE_ORDER)].copy()
if frac_part_ei_df.empty:
  raise ValueError("No multi-bursting movies with fraction of sequence participation for figure:frac-part-ei")

fig, ax = plt.subplots(1, 1, figsize=(2.4, 2.0), constrained_layout=True)
box_kw = {
  "boxprops": {"facecolor": "none", "edgecolor": "0.45", "linewidth": LW_MAIN},
  "whiskerprops": {"color": "k", "linewidth": LW_MAIN},
  "capprops": {"color": "k", "linewidth": LW_MAIN},
  "medianprops": {"color": "k", "linewidth": LW_MAIN},
}

cab_frac_df = frac_part_ei_df[frac_part_ei_df["tsu_type"].astype(str) == "cab"].copy()
cab_pair_df = (
  cab_frac_df.pivot_table(index="tsu", columns="cond", values="brst_neur_frac", aggfunc="mean")
  .replace([np.inf, -np.inf], np.nan)
  .dropna(subset=HUE_ORDER)
  .reset_index()
)
if cab_pair_df.empty:
  raise ValueError("Need paired spontaneous and excited CCN movies for figure:frac-part-ei")

cab_long_df = cab_pair_df.melt(id_vars="tsu", value_vars=HUE_ORDER, var_name="cond", value_name="brst_neur_frac")
cab_long_df["cond"] = pd.Categorical(cab_long_df["cond"], categories=HUE_ORDER, ordered=True)
sns.boxplot(data=cab_long_df, x="cond", y="brst_neur_frac", order=HUE_ORDER, ax=ax, **box_kw)
for _, row in cab_pair_df.iterrows():
  y_pair = row[HUE_ORDER].to_numpy(dtype=float)
  ax.plot(np.arange(len(HUE_ORDER), dtype=float), y_pair, color="0.65", lw=LW_SECONDARY, alpha=ALPHA_BACKGROUND, zorder=1)
for cond_idx, cond in enumerate(HUE_ORDER):
  one_v = cab_pair_df[cond].to_numpy(dtype=float)
  ax.scatter(
    np.full(len(one_v), cond_idx, dtype=float),
    one_v,
    marker=TSU_MARKER.get("cab", "o"),
    color=COND_COLOR.get(str(cond), "0.4"),
    s=S_SCATTER_HIGHLIGHT,
    alpha=ALPHA_MAIN,
    linewidths=0,
    zorder=3,
  )
if ax.get_legend() is not None:
  ax.get_legend().remove()

cab_t = np.nan
cab_p = np.nan
if len(cab_pair_df) >= 2:
  tt = sp.stats.ttest_rel(cab_pair_df["spon"].to_numpy(dtype=float), cab_pair_df["stim"].to_numpy(dtype=float), nan_policy="omit")
  cab_t = float(tt.statistic)
  cab_p = float(tt.pvalue)
cab_pair_tbl = pd.DataFrame([{"comparison": "spontaneous vs excited | CCN paired", "n_pair": int(len(cab_pair_df)), "T": cab_t, "p_unc": cab_p}])
print("\nPaired t-test for figure:frac-part-ei CCN fraction of sequence participation by condition:")
print(cab_pair_tbl[["comparison", "n_pair", "T", "p_unc"]])

cab_abs_delta = np.abs(
  cab_pair_df["stim"].to_numpy(dtype=float)
  - cab_pair_df["spon"].to_numpy(dtype=float)
)
cab_abs_delta = cab_abs_delta[np.isfinite(cab_abs_delta)]
cab_abs_t = np.nan
cab_abs_p = np.nan
if cab_abs_delta.size >= 2:
  tt_abs = sp.stats.ttest_1samp(cab_abs_delta, popmean=0.0, alternative="greater", nan_policy="omit")
  cab_abs_t = float(tt_abs.statistic)
  cab_abs_p = float(tt_abs.pvalue)
cab_abs_tbl = pd.DataFrame([{
  "test": "abs(excited - spontaneous) > 0 | CCN paired",
  "n_pair": int(cab_abs_delta.size),
  "mean_abs_delta": float(np.nanmean(cab_abs_delta)) if cab_abs_delta.size > 0 else np.nan,
  "T": cab_abs_t,
  "p_unc": cab_abs_p,
}])
print("\nAbsolute paired change test for figure:frac-part-ei CCN fraction of sequence participation:")
print(cab_abs_tbl[["test", "n_pair", "mean_abs_delta", "T", "p_unc"]])
annotate_box_stars(ax, [("spon", "stim")], {("spon", "stim"): cab_p}, order=HUE_ORDER)

ax.set_xlabel("")
ax.set_ylabel(r"$\mathrm{frac.\ of\ seq.\ participation}$")
ax.set_xticks(np.arange(len(HUE_ORDER)))
ax.set_xticklabels([r"$\mathrm{spontaneous}$", r"$\mathrm{excited}$"])
ax.set_ylim(-0.06, 1.2)

for _ax in fig.axes:
  _ax.xaxis.label.set_size(FIG_FONT_PT)
  _ax.yaxis.label.set_size(FIG_FONT_PT)
  _ax.xaxis.labelpad = FIG_XLABELPAD
  _ax.yaxis.labelpad = FIG_YLABELPAD
  _ax.title.set_size(FIG_FONT_PT)
  _ax.tick_params(labelsize=FIG_FONT_PT)
  for _txt in _ax.texts:
    _txt.set_fontsize(FIG_FONT_PT)

add_panel_labels(fig, [("A", ax, -2.5)])

(main_path / "analysis/result").mkdir(parents=True, exist_ok=True)
# fig.savefig(main_path / "analysis/result/s_frac_part_ei.pdf", format="pdf", bbox_inches="tight")

plt.show()




In [ ]:
# fig:physical-more

x_t_cor_tsu_df = (
  _build_multi_plot_df(["x_t_cor"])[["mv_name", "tsu_type", "cond", "x_t_cor"]]
  .replace([np.inf, -np.inf], np.nan)
  .dropna(subset=["tsu_type", "x_t_cor"])
  .drop_duplicates(subset=["mv_name"])
  .copy()
)
if x_t_cor_tsu_df.empty:
  raise ValueError("No multi-bursting movies with x_t_cor for figure:physical-more real-space global correlation plot")

x_t_cor_tsu_order = [g for g in ORDER if g in set(x_t_cor_tsu_df["tsu_type"].astype(str))]
if len(x_t_cor_tsu_order) < 2:
  raise ValueError("Need both BTO and CCN movies for figure:physical-more real-space global correlation plot")

fig, ax_a = plt.subplots(1, 1, figsize=(2.4, 2.2), constrained_layout=True)
box_kw = {
  "boxprops": {"facecolor": "none", "edgecolor": "0.45", "linewidth": LW_MAIN},
  "whiskerprops": {"color": "k", "linewidth": LW_MAIN},
  "capprops": {"color": "k", "linewidth": LW_MAIN},
  "medianprops": {"color": "k", "linewidth": LW_MAIN},
}

sns.boxplot(
  data=x_t_cor_tsu_df,
  x="tsu_type",
  y="x_t_cor",
  order=x_t_cor_tsu_order,
  ax=ax_a,
  **box_kw,
)
for tsu_type in x_t_cor_tsu_order:
  for cond in HUE_ORDER:
    one_df = x_t_cor_tsu_df[
      (x_t_cor_tsu_df["tsu_type"].astype(str) == tsu_type)
      & (x_t_cor_tsu_df["cond"].astype(str) == cond)
    ]
    if one_df.empty:
      continue
    sns.stripplot(
      data=one_df,
      x="tsu_type",
      y="x_t_cor",
      order=x_t_cor_tsu_order,
      jitter=False,
      alpha=ALPHA_MAIN,
      color=COND_COLOR.get(str(cond), "0.4"),
      marker=TSU_MARKER.get(str(tsu_type), "o"),
      size=S_STRIP,
      linewidth=0,
      ax=ax_a,
    )
if ax_a.get_legend() is not None:
  ax_a.get_legend().remove()

x_t_cor_pair_s = [("org", "cab")]
p_map = holm_pairwise_pvals(x_t_cor_tsu_df, "tsu_type", "x_t_cor", x_t_cor_pair_s)
print_pairwise_annotation_table(
  x_t_cor_tsu_df, "tsu_type", "x_t_cor", x_t_cor_pair_s, p_map,
  "figure:physical-more real-space global correlation annotation Welch t-test",
)
annotate_box_stars(ax_a, x_t_cor_pair_s, p_map, order=x_t_cor_tsu_order)

anova_tbl_x_t_cor_tsu = print_tsu_cond_anova(
  x_t_cor_tsu_df,
  "x_t_cor",
  "ANOVA for figure:physical-more real-space global correlation by tissue x condition",
)
print("\nReal-space global correlation by movie:")
print(x_t_cor_tsu_df[["mv_name", "tsu_type", "cond", "x_t_cor"]])

ax_a.set_xlabel("")
ax_a.set_ylabel(r"$\mathrm{distance\!-latency\ cor.}$")
ax_a.set_xticks(np.arange(len(x_t_cor_tsu_order)))
ax_a.set_xticklabels([
  {"org": r"$\mathrm{BTO}$", "cab": r"$\mathrm{CCN}$"}.get(str(g), rf"$\mathrm{{{g}}}$")
  for g in x_t_cor_tsu_order
])
ax_a.tick_params(labelsize=FIG_FONT_PT)
ax_a.xaxis.label.set_size(FIG_FONT_PT)
ax_a.yaxis.label.set_size(FIG_FONT_PT)
for _txt in ax_a.texts:
  _txt.set_fontsize(FIG_FONT_PT)

for _ax in fig.axes:
  _ax.xaxis.label.set_size(FIG_FONT_PT)
  _ax.yaxis.label.set_size(FIG_FONT_PT)
  _ax.xaxis.labelpad = FIG_XLABELPAD
  _ax.yaxis.labelpad = FIG_YLABELPAD
  _ax.tick_params(labelsize=FIG_FONT_PT)
  for _txt in _ax.texts:
    _txt.set_fontsize(FIG_FONT_PT)

add_panel_labels(fig, [("A", ax_a)])

(main_path / "analysis/result").mkdir(parents=True, exist_ok=True)
# fig.savefig(main_path / "analysis/result/s_physical_more.pdf", format="pdf", bbox_inches="tight")

plt.show()


